# Offline Wikipedia RAG · Portable, resumable comparison edition

This notebook **modifies your latest “Final With Proceed Reports” notebook**. It keeps the required
**google/flan-t5-small** generator, the frozen **all-MiniLM-L6-v2** embedder, **220-token chunks / 32-token overlap / 188-token stride**,
and the existing `desktop-dense-v1` corpus identity. The original chunk/vector code is retained verbatim so compatible saved shards remain reusable.

Your four laptop shards live on **your** disk; they are not embedded in this notebook. The notebook checks them at runtime.
On that laptop it automatically uses `~/OfflineWikipediaRAG_Desktop`. On Kaggle it uses
`/kaggle/working/OfflineWikipediaRAG_Desktop`. To move saved work, export a bundle and attach it on the other platform.
Opening the same `.ipynb` does not move its database/model files or restore Python variables.

**First use:** Python 3.11/3.12 Jupyter → Run All → choose a workflow and click Proceed. Subsequent modules execute in order,
with their own status/output. Rerun setup and Module 1 after a kernel restart. Automatic mode resumes an unfinished selected shard;
otherwise it compares existing indexed shards. It does not silently select shard 5 after finishing shard 4.

**Validation only** loads saved models, vectors, reports and Q&A. It never downloads Wikipedia, embeds passages, rebuilds indexes or trains.
**Process selected shard** adds one full shard. **Compare existing** builds/verifies derived Chroma/BM25 indexes and measures alternatives.
**Final test** is explicit and freezes one release before opening reserved labels. Export workflows create sharing/resume archives.

In Kaggle, enable Internet when packages/assets must first be downloaded, select an available accelerator manually, and save/export outputs before the session ends.
Unattended Kaggle batch versions use the saved shard and Auto mode instead of waiting for a widget click. CPU and NVIDIA CUDA are supported;
TPU/XPU training is not implemented. Kaggle Q&A runs in notebook widgets; the laptop opens a separate local browser page.

All results start empty. No speedup, accuracy, zero-hallucination guarantee, or real-hardware validation is prefilled.


## Design comparison and experiment protocol

| Choice | Baseline / alternative | Reason and evaluation |
|---|---|---|
| Generator | Flan-T5-small only; pretrained / previous / latest tuned weights | Required by the brief. Tuning must beat a baseline rather than be assumed better. |
| Corpus / chunks | Existing Wikipedia snapshot, full selected shards, 220/32 | Preserve existing vectors; corpus processing is never sampled. |
| Database / index | LanceDB IVF-PQ versus Chroma cosine HNSW | Compare identical full stored vectors. Quantized search trades recall for index size; a graph has different memory/build costs. |
| Lexical retrieval | SQLite FTS5 BM25 | A resumable sidecar avoids modifying sealed Lance partitions; useful for exact names/terms. |
| Semantic retrieval | Frozen MiniLM cosine neighbours | Semantic baseline; query vectors and distance policy stay consistent. |
| Hybrid | BM25 + dense, reciprocal-rank fusion | Avoid treating BM25, cosine and fusion scores as interchangeable probabilities. |
| Optional reranker | Local cross-encoder | Additional pairwise inference; off by default, benchmark it explicitly using the selector checkbox. |
| Fusion | Concatenate versus generate-per-passage then merge | Same prompt budget and original evidence; generated candidates remain unverified. |
| Supervision | SQuAD-v2 QA matched to indexed articles | Wikipedia alone is not labelled QA. Unsupported aliases/context matches are excluded. |
| Split / replay | Original article-disjoint train/dev/reserved-test; train-only replay | Preserve lineage and limit leakage/forgetting. Historical dev questions have already been viewed. |
| Sharing | Versioned artifact bundle + notebook | Code alone cannot reproduce an offline model with its saved corpus. |

The experiment first benchmarks **retrieval on up to 40 development questions** across covered shards (exact dense reference for up to 8).
Then it selects a retrieval strategy on development article hit/MRR and compares **distinct saved readers × k=1/3/5/10 × two fusion methods**
on up to **24 development questions**. These are explicit evaluation-cost limits, not corpus limits. This staged design is not a full
database × retrieval × reader Cartesian-product experiment. CPU profiling and cross-encoder reranking are optional additional costs.

RRF uses `sum(1/(60+rank))`; BM25 uses FTS5 term relevance. Cosine distance is `1 − cosine_similarity`, not confidence.
EM measures normalized exact reference overlap; token F1 balances shared-token precision and recall. A lexical unsupported-sentence proxy
does not establish entailment. **Factual hallucination rate stays unavailable until human-adjudicated claims/labels exist.**

Primary references: [Chroma supplied vectors](https://docs.trychroma.com/docs/collections/add-data),
[Chroma HNSW configuration](https://docs.trychroma.com/docs/collections/configure),
[Lance indexes](https://lance.org/format/index/vector/), [LanceDB Python](https://lancedb.github.io/lancedb/python/python/),
[FTS5 BM25](https://www.sqlite.org/fts5.html#the_bm25_function),
[cross-encoders](https://sbert.net/docs/cross_encoder/usage/usage.html), [Kaggle notebooks](https://www.kaggle.com/docs/notebooks).


## New · Interactive backend and generation controls

Main Q&A now offers **Saved deployment, LanceDB only, Chroma only, BM25 only, or Compare all three**.
All three comparison answers use the **same saved final generator**, deployed shard scope, top-k, fusion and generation settings.
They run **sequentially** with a single shared question embedding to limit laptop memory use.
Controls include retrieval k, answer method, output token limit, deterministic/sampling mode, temperature, top-p,
seed, optional reference answer, and source/calculation visibility. The prompt budget stays 512 tokens.
Temperature/top-p are inactive in deterministic mode. Missing/incomplete indexes are disabled, never built by the Q&A page.

Each backend has a separate answer and measured metric row. Repeated complete three-way comparisons produce a paired
interactive mean matrix grouped by exact settings, release and hardware. These are exploratory measurements, **not held-out
accuracy**. Reference EM/F1 require a reference; factual hallucination remains unmeasured. Original saved evaluation matrices
and deployment settings stay unchanged. JSON/CSV answer exports include the effective controls. Kaggle widgets expose the same choices.

### Continue an existing Chroma conversion safely

**Let the old conversion finish, or interrupt its running cell cleanly before opening this revision. Do not run two writers.**
Use the **same project/cache folders and workflow selection**. This revision preserves the original embedding, chunking,
LanceDB, Chroma-copy and BM25 functions, model/training identity, package requirements, and migration batch of **512**.
Completed compatible stages are reused. A partial Chroma conversion scans saved source batches, skips receipt-confirmed batches,
and continues the remainder. A batch committed just before interruption but lacking a receipt may be replayed safely by upsert.
Source checksums/counts still run and can take time; they are verification, not re-embedding or retraining.

To open the new controls using a saved release, choose **Validation only**. To finish an interrupted derived-index conversion,
choose **Compare existing indexed shards only**; this also runs/reuses the comparison stages afterward.
Choose **Process selected shard** when adding a new shard. Kaggle requires the actual saved project artifacts restored from a
resume bundle to continue work from a laptop; the notebook file alone does not move your data.

**Checks for this revision:** synthetic four-shard data with real LanceDB/Chroma/BM25 operations and small saved random models;
interrupted Chroma-copy recovery, unchanged source/CFG/model identities, selected-backend routing, paired comparisons,
sampling seed restoration, reference scoring, real local HTTP, Kaggle-style widget callbacks, and JavaScript execution checks.
Native Windows, actual Kaggle, full Wikipedia scale, real pretrained-model answer quality, and a visual browser render were not run here.


## Setup · Environment and unchanged corpus configuration

The next cell detects Kaggle versus a laptop and chooses writable paths. It preserves the laptop's existing project root,
so saved shards are discovered rather than recreated. `device='auto'` chooses CUDA when available and CPU otherwise.
Embedding/article batches remain **256**; training stays **batch 2 × accumulation 8**, with **2 epochs per new stage**.
The old `dev_questions=12` remains for training-checkpoint compatibility; the new benchmark limits are separate.

Defaults require no code edits. The workflow selector controls validation, processing, final testing and export.
Keep only the intended input bundle beside a local notebook or attached to Kaggle. On an empty project it will be verified/restored automatically.


In [1]:
def portable_environment():
    """Choose writable paths automatically; keep the existing desktop root to reuse its shards."""
    kaggle=Path('/kaggle/working').is_dir() and Path('/kaggle/input').is_dir()
    root=Path('/kaggle/working/OfflineWikipediaRAG_Desktop') if kaggle else Path.home()/'OfflineWikipediaRAG_Desktop'
    override=os.environ.get('WIKI_RAG_ROOT')
    if override:root=Path(override).expanduser()
    root=root.resolve();root.mkdir(parents=True,exist_ok=True)
    return {'kaggle':kaggle,'root':root,'cache':root.parent/'OfflineWikipediaRAG_Desktop_cache',
        'batch':kaggle and os.environ.get('KAGGLE_KERNEL_RUN_TYPE','').casefold() in {'batch','script'}}


**Apply the detected environment.** Set the existing project/cache roots and unchanged model/chunk settings. The separate comparison limits control evaluation cost without limiting corpus processing.


In [2]:
from pathlib import Path
import os,json,time
ENVIRONMENT=portable_environment()
PROJECT_ROOT=ENVIRONMENT['root'];CACHE_ROOT=ENVIRONMENT['cache']
CPU_THREADS=max(1,min(8,os.cpu_count() or 1))
DESKTOP_CONFIG={
    'device': 'auto', 'offline': False, 'seed': 42,
    'chunk_tokens': 220, 'chunk_overlap': 32, 'article_batch': 256,
    'encode_batch': 256, 'input_cap': 512, 'max_new_tokens': 128,
    'max_question_chars': 800, 'train_epochs': 2.0, 'train_lr': 3e-05,
    'train_batch': 2, 'train_accum': 8, 'train_max_steps': -1,
    'checkpoint_steps': 25, 'checkpoint_seconds': 300, 'min_new_train': 4,
    'replay_fraction': 0.2, 'replay_max': 128, 'qa_context_overlap': 0.5,
    'dev_questions': 12, 'k_values': [1, 3, 5, 10], 'max_cosine_distance': 0.65,
    'require_answer_occurrence': True, 'nprobes': 32, 'output_budget_bytes': 50000000000,
    'free_reserve_bytes': 2000000000, 'time_budget_hours': 0, 'verify_vector_hashes': False,
    'web_port': 7860, 'open_browser': True, 'manual_samples_per_shard': 3,
    'migration_batch': 512, 'compare_chroma': True, 'enable_reranker': False,
    'run_cpu_profile': False, 'benchmark_dev_questions': 40, 'generation_dev_questions': 24,
    'retrieval_candidates': 24, 'retrieval_select_k': 5, 'query_repeats': 3,
    'exact_reference_queries': 8, 'final_test_questions': 100,
}
DESKTOP_CONFIG.update(project_root=str(PROJECT_ROOT),cache_root=str(CACHE_ROOT),cpu_threads=CPU_THREADS)
if ENVIRONMENT['kaggle']:
    DESKTOP_CONFIG.update(output_budget_bytes=18_000_000_000,free_reserve_bytes=1_000_000_000)
for name in ['OMP_NUM_THREADS','MKL_NUM_THREADS','RAYON_NUM_THREADS']:os.environ[name]=str(CPU_THREADS)
os.environ.update(USE_TORCH_XLA='0',PYTHONIOENCODING='utf-8',TOKENIZERS_PARALLELISM='false',
    HF_HOME=str(CACHE_ROOT/'huggingface'),ANONYMIZED_TELEMETRY='False')
os.environ['CUDA_VISIBLE_DEVICES']=os.environ.get('CUDA_VISIBLE_DEVICES','0').split(',')[0] # Keep one device's batch geometry.
print('Platform:','Kaggle' if ENVIRONMENT['kaggle'] else 'Laptop / local Jupyter')
print('Project:',PROJECT_ROOT,'| Cache:',CACHE_ROOT)


Platform: Laptop / local Jupyter
Project: C:\Users\Anirban\OfflineWikipediaRAG_Desktop | Cache: C:\Users\Anirban\OfflineWikipediaRAG_Desktop_cache


## Setup · Dependencies

This cell installs only missing/incompatible requirements into the active kernel environment. An existing compatible CUDA Torch
installation is retained; a laptop without Torch receives CPU wheels. It avoids blindly replacing Kaggle's GPU stack.
If an already imported binary package changes, restart the kernel and Run All. Internet is required for missing packages;
fully offline dependency installation requires a separately prepared wheel cache/environment.


In [3]:
import sys,subprocess,importlib.metadata as metadata
AUTO_INSTALL = True
REQUIREMENTS = ['torch>=2.6,<3','transformers==4.57.6','sentence-transformers>=3.3,<6',
    'huggingface-hub>=0.34,<1','datasets>=3,<4','accelerate>=1.2,<2','lancedb==0.40.0',
    'pyarrow>=18,<27','numpy>=1.26,<3','pandas>=2.1,<4','matplotlib>=3.8,<4',
    'sentencepiece>=0.2,<1','safetensors>=0.4,<1','ipywidgets>=8.1,<9','packaging>=23,<27','jupyter-ui-poll==1.1.0','chromadb==1.5.9','psutil>=5.9,<8']
try:
    from packaging.requirements import Requirement
except ImportError:
    if not AUTO_INSTALL:raise
    subprocess.check_call([sys.executable,'-m','pip','install','packaging>=23,<27','jupyter-ui-poll==1.1.0','chromadb==1.5.9','psutil>=5.9,<8'])
    from packaging.requirements import Requirement
missing=[]
for spec in REQUIREMENTS:
    req=Requirement(spec)
    try:version=metadata.version(req.name)
    except metadata.PackageNotFoundError:version=None
    if version is None or version not in req.specifier:missing.append(spec)
if missing:
    if not AUTO_INSTALL:raise RuntimeError('Missing/incompatible packages: '+str(missing))
    loaded=any(Requirement(spec).name.replace('-','_') in sys.modules for spec in missing)
    torch_specs=[s for s in missing if Requirement(s).name=='torch']
    if torch_specs and ENVIRONMENT['kaggle']:
        raise RuntimeError('Kaggle Torch is missing/incompatible. Choose a current supported Kaggle image rather than replacing its CUDA stack with CPU wheels.')
    if torch_specs:
        subprocess.check_call([sys.executable,'-m','pip','install',*torch_specs,'--index-url','https://download.pytorch.org/whl/cpu'])
    others=[s for s in missing if Requirement(s).name!='torch']
    if others:subprocess.check_call([sys.executable,'-m','pip','install',*others,'torch=='+metadata.version('torch')])
    if loaded:raise RuntimeError('Dependencies changed after imports. Restart the kernel, then Run All again.')
print('Dependencies verified.')


Dependencies verified.


**Runtime imports.** Load data, vector, model and reporting libraries after dependencies are checked.
The same import block is also used by exported child workers. Matplotlib uses a noninteractive backend so reports can be saved on Kaggle.


In [4]:
from pathlib import Path
import os,sys,json,hashlib,collections,math,re,string,random,time,shutil,gc,types,inspect,uuid,subprocess
import html,urllib.parse,urllib.request,unicodedata,threading,secrets,traceback,base64,webbrowser
import importlib.metadata as metadata
from datetime import datetime,timezone,timedelta
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import lancedb,torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
from huggingface_hub import HfApi,hf_hub_download,snapshot_download
from tqdm.auto import tqdm
class BudgetPause(RuntimeError):pass

import copy

from IPython.display import display,Markdown,HTML,FileLink
from IPython import get_ipython
import ipywidgets as widgets


W1031 11:14:05.810000 12024 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1031 11:14:05.891000 12024 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


**Worker registry.** Record library function names and shared imports. Each heavy stage exports only functions already defined
above it, then runs that stage in a fresh process. This isolates database clients, makes memory measurements comparable,
and allows the later browser backend to load the exact same inference implementation.


In [5]:
PORTABLE_EXPORT_NAMES=['digest', 'file_sha', 'read_json', 'dataset_remote_file', 'validate_dataset_inventory', 'render_prompt', 'normalize_answer', 'em_f1', 'matrix_plot', 'web_sources_html', 'atomic_json', 'configure_desktop', 'acquire_local_lock', 'release_local_lock', 'file_inventory', 'check_inventory', 'guard_resources', 'shard_record', 'save_shard_record', 'resolve_desktop_lock', 'prepare_desktop_model', 'download_desktop_shard', 'load_desktop_embedder', 'vector_schema', 'desktop_table', 'desktop_chunks', 'commit_vectors', 'committed_article_cursor', 'embed_desktop_shard', 'index_desktop_shard', 'native_desktop_stage', 'open_desktop_collection', 'retrieve_desktop', 'prepare_labelled_qa', 'article_key', 'qa_article_split', 'build_shard_qa', 'match_qa_row', 'all_matched_qa', 'alias_in_text', 'load_generator', 'pack_desktop_evidence', 'training_rows', 'verify_saved_reader', 'base_reader', 'latest_training_reader', 'promote_training_lineage', 'incremental_training_spec', 'checked_incremental_checkpoint', 'incremental_callbacks', 'execute_incremental_training', 'plan_incremental_training', 'train_desktop_shard', 'generate_desktop', 'lexical_grounding', 'finish_grounded_output', 'answer_desktop', 'generate_merge_desktop', 'reader_candidates', 'corpus_snapshot', 'read_evaluation_cache', 'evaluation_spec', 'development_questions', 'score_desktop_prediction', 'run_development_grid', 'export_desktop_matrices', 'select_development_reader', 'run_desktop_assessment', 'save_deployment', 'load_saved_deployment', 'saved_pipeline_offline_check', 'desktop_status', 'completed_desktop_output', 'desktop_html', 'desktop_javascript', 'qa_status_payload', 'qa_report_payload', 'live_desktop_answer', 'desktop_http_handler', 'serve_desktop_qa', 'existing_qa_connection', 'stop_existing_qa', 'launch_desktop_qa', 'desktop_cli', 'offline_socket_guard_script', 'report_json_safe', 'append_report_jsonl', 'read_report_jsonl', 'record_desktop_error', 'reference_score_details', 'answer_calculation_details', 'report_validation_perplexity', 'collect_training_report', 'collect_desktop_logs', 'prediction_error_row', 'manual_shard_questions', 'manual_sample_answers', 'write_report_csv', 'plot_report_training', 'plot_report_errors', 'report_table_html', 'standalone_desktop_report', 'build_desktop_reports', 'available_desktop_reports', 'resolve_desktop_report', 'qa_samples_payload', 'persist_desktop_answer', 'qa_application_version', 'desktop_api_get', 'desktop_api_post', 'qa_report_css', 'qa_markup_template', 'portable_environment', 'portable_configure', 'portable_scope', 'portable_part_key', 'portable_batches', 'portable_vector_rows', 'portable_chroma_client', 'portable_chroma_collection', 'portable_verify_chroma', 'portable_migrate_part', 'portable_migrate_all', 'portable_lexical_path', 'portable_build_bm25', 'portable_fetch_passages', 'portable_bm25', 'portable_rrf', 'portable_dense', 'portable_retrieve', 'portable_answer', 'portable_prepare_reranker', 'portable_load_reranker', 'portable_progress', 'portable_export_runtime', 'portable_spawn', 'portable_worker', 'portable_hardware', 'portable_question_bank', 'portable_query_reference', 'portable_rss_sampler', 'portable_retrieval_worker', 'portable_benchmark_spec', 'portable_retrieval_suite', 'portable_readers', 'portable_generation_worker', 'portable_assess', 'portable_save_release', 'portable_release_pointer', 'portable_validate_release', 'portable_load_release', 'portable_offline_check_body', 'portable_offline_verify', 'portable_unsupported_probes', 'portable_train_selected', 'portable_cpu_profile', 'portable_retrieval_matrices', 'portable_index_costs', 'portable_final_test_worker', 'portable_bundle_files', 'portable_export_bundle', 'portable_import_bundle', 'portable_restore_inputs', 'portable_write_instructions', 'portable_state_fingerprint', 'portable_status', 'portable_ui_payload', 'portable_markup', 'portable_ui_script', 'portable_cli', 'qa_backend_status', 'qa_control_options', 'qa_parse_controls', 'qa_generate_controlled', 'qa_answer_controlled', 'qa_request_references', 'qa_controlled_result', 'qa_comparison_row', 'qa_comparison_history', 'qa_inline_render', 'qa_inline_controls', 'qa_inline_callback', 'qa_controls_markup', 'qa_controls_css', 'qa_controls_script']
PORTABLE_MODULE_HEADER="from pathlib import Path\nimport os,sys,json,hashlib,collections,math,re,string,random,time,shutil,gc,types,inspect,uuid,subprocess\nimport html,urllib.parse,urllib.request,unicodedata,threading,secrets,traceback,base64,webbrowser\nimport importlib.metadata as metadata\nfrom datetime import datetime,timezone,timedelta\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport lancedb,torch\nfrom sentence_transformers import SentenceTransformer\nfrom transformers import AutoTokenizer,AutoModelForSeq2SeqLM\nfrom huggingface_hub import HfApi,hf_hub_download,snapshot_download\nfrom tqdm.auto import tqdm\nclass BudgetPause(RuntimeError):pass\n\nimport copy\n"


## Module 1 · Restore, select and inspect saved progress

The helpers below verify identity/checksums, retain the original chunking and answer policy, and coordinate module locks.
An input bundle is verified before extraction; conflicting existing files are refused. Each archive member has its own checksum.
Your original `parts/` data and saved training lineage remain the source of truth.

The final cells of this module restore an attached bundle when appropriate, show the shard/workflow selector, and wait for Proceed.
After restarting a kernel, run from the top so imports, functions and dropdown variables exist again.


**Helper: `digest`**

Provide a shared deterministic operation.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [6]:
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(",", ":"), allow_nan=False).encode("utf-8")).hexdigest()


**Helper: `file_sha`**

Provide a shared deterministic operation.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [7]:
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


**Helper: `read_json`**

Provide a shared deterministic operation.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [8]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


**Helper: `dataset_remote_file`**

Provide a shared deterministic operation.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [9]:
def dataset_remote_file(item):
    lfs = getattr(item, "lfs", None)
    sha = lfs.get("sha256") if isinstance(lfs, dict) else getattr(lfs, "sha256", None)
    size = getattr(item, "size", None)
    if size is None or (item.rfilename.endswith(".parquet") and not sha):
        raise ValueError("Hub file metadata lacks size/SHA256; do not declare an unverifiable dataset complete.")
    return {"path": item.rfilename, "size": int(size), "sha256": sha,
            "git_blob": getattr(item, "blob_id", None)}


**Helper: `validate_dataset_inventory`**

Provide a shared deterministic operation.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [10]:
def validate_dataset_inventory(files):
    parts = [entry for entry in files if entry["path"].endswith(".parquet")]
    pattern = re.compile(re.escape(DATASET_CONFIG) + r"/train-(\d+)-of-(\d+)\.parquet$")
    matches = [pattern.fullmatch(entry["path"]) for entry in parts]
    if not matches or not all(matches):
        raise ValueError("Unexpected English dataset shard layout; inspect the pinned repository before downloading.")
    totals = {int(match.group(2)) for match in matches}
    if len(totals) != 1 or {int(match.group(1)) for match in matches} != set(range(next(iter(totals)))):
        raise ValueError("The remote dataset inventory is missing or duplicating shards.")
    if len(parts) != next(iter(totals)) or len({entry["path"] for entry in files}) != len(files):
        raise ValueError("Dataset inventory contains duplicates.")
    for entry in files:
        path = Path(entry["path"])
        if path.is_absolute() or ".." in path.parts:
            raise ValueError("Unsafe path in remote inventory.")


**Helper: `render_prompt`**

Use the same grounding instructions and whole-prompt budget for both packing policies; never include reference answers.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [11]:
def render_prompt(question, selected, question_only=False):
    """Use the same grounding instructions and whole-prompt budget for both packing policies; never include reference answers."""
    if question_only:
        return f'Answer briefly. If you do not know, say {ABSTAIN}\nQuestion: {question}\nAnswer:'
    blocks = [f"[S{i}] {span['title']}: {span['text']}" for i, span in enumerate(selected, 1)]
    context = '\n'.join(blocks) if blocks else '(no evidence supplied)'
    return ('Answer briefly using only the evidence. Do not output source labels alone. '
        f'If the evidence does not answer the question, say exactly: {ABSTAIN}\n'
        f'Evidence:\n{context}\nQuestion: {question}\nAnswer:')


**Helper: `normalize_answer`**

Use a declared SQuAD-style normalization for EM/token F1. This measures reference overlap, not factual entailment.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [12]:
def normalize_answer(text):
    """Use a declared SQuAD-style normalization for EM/token F1. This measures reference overlap, not factual entailment."""
    text = str(text).casefold()
    text = ''.join(character for character in text if character not in string.punctuation)
    return ' '.join(re.sub(r'\b(a|an|the)\b', ' ', text).split())


**Helper: `atomic_json`**

Publish complete checkpoint metadata atomically, allowing brief Windows sharing contention.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [13]:
def atomic_json(path,value):
    """Publish complete checkpoint metadata atomically, allowing brief Windows sharing contention."""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temporary=path.with_name(path.name+'.'+uuid.uuid4().hex+'.tmp')
    temporary.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    try:
        for attempt in range(8):
            try:os.replace(temporary,path);return
            except PermissionError:
                if os.name!='nt' or attempt==7:raise
                time.sleep(.02*(attempt+1))
    finally:
        if temporary.exists():temporary.unlink()


**Helper: `configure_desktop`**

Apply explicit local configuration before model execution; Intel Graphics is not treated as CUDA.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [14]:
def configure_desktop(config):
    """Apply explicit local configuration before model execution; Intel Graphics is not treated as CUDA."""
    global CFG,ROOT,DEVICE,STARTED,ABSTAIN,DATASET_CONFIG
    CFG=dict(config);ROOT=Path(CFG['project_root']).expanduser().resolve();ROOT.mkdir(parents=True,exist_ok=True)
    globals()['RESOURCE_CACHE']={'at':0.,'used':0}
    STARTED=time.perf_counter();ABSTAIN='Insufficient evidence in the indexed dataset.';DATASET_CONFIG='20231101.en'
    for name in ['OMP_NUM_THREADS','MKL_NUM_THREADS']:os.environ[name]=str(CFG['cpu_threads'])
    os.environ['PYTHONIOENCODING']='utf-8';os.environ['USE_TORCH_XLA']='0'
    os.environ['TOKENIZERS_PARALLELISM']='false';os.environ['HF_HOME']=str(Path(CFG['cache_root'])/'huggingface')
    os.environ['HF_HUB_OFFLINE']=os.environ['HF_DATASETS_OFFLINE']='1' if CFG['offline'] else '0'
    Path(CFG['cache_root']).mkdir(parents=True,exist_ok=True)
    if CFG['device']=='cuda' and not torch.cuda.is_available():raise RuntimeError('CUDA was requested but no NVIDIA CUDA device is available.')
    DEVICE=CFG['device'];torch.set_num_threads(CFG['cpu_threads'])
    random.seed(CFG['seed']);np.random.seed(CFG['seed']);torch.manual_seed(CFG['seed'])


**Helper: `acquire_local_lock`**

Keep one writer/server per project with Windows byte locks or Unix flock; no stale-PID guessing.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [15]:
def acquire_local_lock(path):
    """Keep one writer/server per project with Windows byte locks or Unix flock; no stale-PID guessing."""
    import errno
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);handle=open(path,'a+b');handle.seek(0)
    try:
        if os.name=='nt':
            import msvcrt
            msvcrt.locking(handle.fileno(),msvcrt.LK_NBLCK,1)
        else:
            import fcntl
            fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except OSError as error:
        handle.close()
        if error.errno in {errno.EACCES,errno.EAGAIN,errno.EDEADLK}:raise BlockingIOError('Another process owns this project lock.') from error
        raise
    return handle


**Helper: `release_local_lock`**

Unlock precisely the region acquired above; process exit also releases OS-held locks.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [16]:
def release_local_lock(handle):
    """Unlock precisely the region acquired above; process exit also releases OS-held locks."""
    try:
        handle.seek(0)
        if os.name=='nt':
            import msvcrt
            msvcrt.locking(handle.fileno(),msvcrt.LK_UNLCK,1)
        else:
            import fcntl
            fcntl.flock(handle,fcntl.LOCK_UN)
    finally:handle.close()


**Helper: `file_inventory`**

Record portable relative paths, sizes and hashes after a stage commits successfully.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [17]:
def file_inventory(folder,exclude=()):
    """Record portable relative paths, sizes and hashes after a stage commits successfully."""
    folder=Path(folder)
    return {p.relative_to(folder).as_posix():{'size':p.stat().st_size,'sha256':file_sha(p)}
        for p in sorted(folder.rglob('*')) if p.is_file() and '.cache' not in p.parts
        and p.name not in set(exclude)}


**Helper: `check_inventory`**

Never reuse missing/truncated artifacts; optionally rehash large vectors on every rerun.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [18]:
def check_inventory(folder,files,deep=False):
    """Never reuse missing/truncated artifacts; optionally rehash large vectors on every rerun."""
    folder=Path(folder).resolve()
    if not files:raise ValueError('A completed artifact has an empty file inventory.')
    for relative,entry in files.items():
        p=(folder/relative).resolve()
        if not p.is_relative_to(folder) or not p.is_file() or p.stat().st_size!=entry['size']:
            raise ValueError('Artifact missing, truncated or unsafe: '+relative)
        if deep and file_sha(p)!=entry['sha256']:raise ValueError('Artifact checksum mismatch: '+relative)


**Helper: `guard_resources`**

Pause rather than claim completion if disk reserve, declared budget or optional runtime budget is reached.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [19]:
def guard_resources(extra=0):
    """Pause with measured usage/capacity details while keeping all existing compute receipts intact."""
    free=shutil.disk_usage(ROOT).free;reserve=CFG['free_reserve_bytes'];cap=CFG['output_budget_bytes']
    if free<reserve+extra:
        raise BudgetPause(f'Physical disk reserve reached: free {free/1e9:.2f} GB; keep free {reserve/1e9:.2f} GB; '
            f'next-operation allowance {extra/1e9:.2f} GB. Free space or move the complete project before resuming.')
    if CFG['time_budget_hours'] and time.perf_counter()-STARTED>CFG['time_budget_hours']*3600:
        raise BudgetPause('Configured time budget reached. Rerun the same workflow to resume from saved receipts.')
    if cap:
        cache=globals().setdefault('RESOURCE_CACHE',{'at':0.,'used':0})
        if time.perf_counter()-cache['at']>30 or extra>100_000_000:
            cache.update(at=time.perf_counter(),used=sum(p.stat().st_size for p in ROOT.rglob('*') if p.is_file()))
        used=cache['used']
        if used+extra>cap:
            raise BudgetPause(f'Configured project limit reached: measured project {used/1e9:.2f} GB + '
                f'next-operation allowance {extra/1e9:.2f} GB > limit {cap/1e9:.2f} GB; '
                f'physical free {free/1e9:.2f} GB, reserve {reserve/1e9:.2f} GB. '
                'Use the setup disk controls to measure usage and save a suitable limit, then rerun Module 1. '
                'For interrupted Chroma conversion choose Compare existing indexed shards only. '
                'Committed embeddings and conversion receipts are retained.')


**Helper: `shard_record`**

Read one persistent per-shard receipt; all 41 statuses are reconstructed from saved records.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [20]:
def shard_record(number):
    """Read one persistent per-shard receipt; all 41 statuses are reconstructed from saved records."""
    p=ROOT/'parts'/f'shard_{number-1:03d}'/'desktop_manifest.json'
    return read_json(p) if p.exists() else {'shard':number,'download':'pending','embedding':'pending','indexing':'pending','training':'pending','evaluation':'pending'}


**Helper: `save_shard_record`**

Update status only after its operation succeeds; incomplete stages remain resumable.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [21]:
def save_shard_record(number,**updates):
    """Update status only after its operation succeeds; incomplete stages remain resumable."""
    record={**shard_record(number),**updates,'updated_at':datetime.now(timezone.utc).isoformat()}
    atomic_json(ROOT/'parts'/f'shard_{number-1:03d}'/'desktop_manifest.json',record)
    return record


**Helper: `resolve_desktop_lock`**

Reuse pinned dataset/models/QA revisions; query remote metadata once without downloading all Wikipedia files.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [22]:
def resolve_desktop_lock():
    """Reuse pinned dataset/models/QA revisions; query remote metadata once without downloading all Wikipedia files."""
    global LOCK
    p=ROOT/'desktop_project_lock.json'
    spec={'schema':'desktop-dense-v1','dataset':'wikimedia/wikipedia','config':DATASET_CONFIG,
        'revision':'b04c8d1ceb2f5cd4588862100d08de323dccfbaa','chunk_tokens':CFG['chunk_tokens'],
        'overlap':CFG['chunk_overlap'],'models':{'generator':'google/flan-t5-small','embedder':'sentence-transformers/all-MiniLM-L6-v2'},
        'chunk_code':digest(inspect.getsource(desktop_chunks)+inspect.getsource(vector_schema)+inspect.getsource(commit_vectors))}
    if p.exists():
        LOCK=read_json(p)
        if LOCK['spec']!=spec or LOCK['identity']!=digest({k:v for k,v in LOCK.items() if k!='identity'}):
            raise ValueError('Project identity/configuration changed or is corrupt. Use the original settings or a separate project folder.')
        return LOCK
    if CFG['offline']:raise FileNotFoundError('First setup needs Internet; no pinned project lock exists.')
    info=HfApi().dataset_info(spec['dataset'],revision=spec['revision'],files_metadata=True)
    if info.sha!=spec['revision']:raise ValueError('Dataset revision mismatch.')
    files=[dataset_remote_file(x) for x in info.siblings if x.rfilename.startswith(DATASET_CONFIG+'/') and x.rfilename.endswith('.parquet')]
    files.sort(key=lambda x:x['path']);validate_dataset_inventory(files)
    if len(files)!=41:raise ValueError('Expected 41 pinned Wikipedia shards; review the dataset inventory.')
    models={k:{'repo':v,'revision':HfApi().model_info(v).sha} for k,v in spec['models'].items()}
    qa={'repo':'rajpurkar/squad_v2','revision':HfApi().dataset_info('rajpurkar/squad_v2').sha}
    LOCK={'spec':spec,'files':files,'models':models,'qa':qa};LOCK['identity']=digest(LOCK)
    atomic_json(p,LOCK);return LOCK


**Helper: `vector_schema`**

Persist source identifiers and the committed source cursor in the same atomic vector transaction.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [23]:
def vector_schema():
    """Persist source identifiers and the committed source cursor in the same atomic vector transaction."""
    return pa.schema([('passage_id',pa.string()),('article_id',pa.string()),('title',pa.string()),('url',pa.string()),
        ('text',pa.string()),('source_row',pa.int64()),('source_shard',pa.int32()),('char_start',pa.int64()),
        ('char_end',pa.int64()),('cursor_end',pa.int64()),('vector',pa.list_(pa.float32(),384))])


**Helper: `desktop_table`**

Use one LanceDB partition per source shard; queries merge results across every completed partition.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [24]:
def desktop_table(number,create=False):
    """Use one LanceDB partition per source shard; queries merge results across every completed partition."""
    folder=ROOT/'parts'/f'shard_{number-1:03d}'/'lance';folder.mkdir(parents=True,exist_ok=True)
    db=lancedb.connect(str(folder))
    if create:return db.create_table('passages',schema=vector_schema(),exist_ok=True)
    return db.open_table('passages')


**Helper: `desktop_chunks`**

Chunk the full source article by tokenizer offsets, without article-prefix or passage-count caps.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [25]:
def desktop_chunks(item,row,number,embedder):
    """Chunk the full source article by tokenizer offsets, without article-prefix or passage-count caps."""
    text=str(item.get('text') or '')
    if not text.strip():return []
    encoded=embedder.tokenizer(text,add_special_tokens=False,truncation=False,return_offsets_mapping=True,verbose=False)
    offsets=encoded['offset_mapping'];width=CFG['chunk_tokens'];stride=width-CFG['chunk_overlap']
    if stride<=0:raise ValueError('Chunk overlap must be smaller than the chunk size.')
    result=[]
    for start in range(0,len(offsets),stride):
        end=min(start+width,len(offsets));a,b=offsets[start][0],offsets[end-1][1]
        if b<=a:continue
        passage=text[a:b]
        if len(embedder.tokenizer.encode(passage,add_special_tokens=True))>embedder.max_seq_length:
            raise ValueError('A prepared passage exceeds the embedding model input limit.')
        result.append({'passage_id':f'{number}:{row}:{start}','article_id':str(item['id']),'title':str(item['title']),
            'url':str(item['url']),'text':passage,'source_row':row,'source_shard':number,'char_start':a,'char_end':b})
        if end==len(offsets):break
    return result


**Helper: `commit_vectors`**

Append vectors and their exact article cursor together; interrupted uncommitted work can be safely replayed.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [26]:
def commit_vectors(table,payload,end,embedder):
    """Append vectors and their exact article cursor together; interrupted uncommitted work can be safely replayed."""
    if not payload:return
    vectors=embedder.encode([r['text'] for r in payload],batch_size=CFG['encode_batch'],
        normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
    if vectors.shape!=(len(payload),384) or not np.isfinite(vectors).all():raise ValueError('Invalid embeddings.')
    records=[{**r,'cursor_end':end,'vector':v.astype(np.float32)} for r,v in zip(payload,vectors)]
    table.add(pa.Table.from_pylist(records,schema=vector_schema()))


**Helper: `open_desktop_collection`**

Only completed verified partitions enter retrieval; missing partitions named in a frozen release are errors.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [27]:
def open_desktop_collection(numbers=None):
    """Only completed verified partitions enter retrieval; missing partitions named in a frozen release are errors."""
    parts=[];expected=set(numbers) if numbers is not None else None
    for number in range(1,len(LOCK['files'])+1):
        if expected is not None and number not in expected:continue
        record=shard_record(number)
        if record.get('indexing')!='complete':
            if expected is not None:raise FileNotFoundError(f'Deployed shard {number} is missing or unfinished.')
            continue
        if record['project']!=LOCK['identity']:raise ValueError('Incompatible vector partition identity.')
        folder=ROOT/'parts'/f'shard_{number-1:03d}'
        check_inventory(folder,record['files'],CFG['verify_vector_hashes']);table=desktop_table(number)
        if table.count_rows()!=record['passages']:raise ValueError('Vector row count changed.')
        parts.append({'number':number,'table':table,'record':record})
    if not parts:raise ValueError('No complete indexed shard is available.')
    return parts


**Helper: `pack_desktop_evidence`**

Apply the same whole-prompt token budget to training and inference without using reference answers.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [28]:
def pack_desktop_evidence(question,hits,tokenizer):
    """Apply the same whole-prompt token budget to training and inference without using reference answers."""
    chosen=[]
    if len(tokenizer.encode(render_prompt(question,[])))>CFG['input_cap']:raise ValueError('Question exceeds the prompt budget.')
    for hit in hits:
        trial=chosen+[hit]
        if len(tokenizer.encode(render_prompt(question,trial)))<=CFG['input_cap']:chosen=trial;continue
        low,high=0,len(hit['text'])
        while low<high:
            middle=(low+high+1)//2;partial={**hit,'text':hit['text'][:middle]}
            if len(tokenizer.encode(render_prompt(question,chosen+[partial])))<=CFG['input_cap']:low=middle
            else:high=middle-1
        if low:chosen.append({**hit,'text':hit['text'][:low]})
        break
    return chosen,render_prompt(question,chosen)


**Helper: `verify_saved_reader`**

Accept only complete saved model bundles with their own checksum inventory.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [29]:
def verify_saved_reader(path):
    """Accept only complete saved model bundles with their own checksum inventory."""
    path=Path(path)
    manifest=path/'reader_manifest.json'
    if not manifest.exists():raise FileNotFoundError('Saved generator has no verified completion manifest.')
    record=read_json(manifest);check_inventory(path,record['files'],True)
    if record['project']!=LOCK['identity']:raise ValueError('Saved generator belongs to another corpus/model identity.')
    return {'id':record['id'],'path':str(path),'manifest':record}


**Helper: `lexical_grounding`**

Expose a lexical support proxy; substring occurrence cannot prove factual correctness or entailment.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [30]:
def lexical_grounding(answer,evidence):
    """Expose a lexical support proxy; substring occurrence cannot prove factual correctness or entailment."""
    if not answer or normalize_answer(answer)==normalize_answer(ABSTAIN):return {'unsupported_sentence_proxy':None,'answer_occurs_in_evidence':False}
    source=normalize_answer(' '.join(s['text'] for s in evidence));sentences=re.split(r'(?<=[.!?])\s+',answer)
    def overlap(sentence):
        words=set(normalize_answer(sentence).split());return len(words&set(source.split()))/max(1,len(words))
    return {'unsupported_sentence_proxy':sum(overlap(s)<.8 for s in sentences)/max(1,len(sentences)),
        'answer_occurs_in_evidence':bool(normalize_answer(answer)) and normalize_answer(answer) in source}


**Helper: `finish_grounded_output`**

Apply a conservative lexical answer-occurrence gate to deployed RAG, with its limitations disclosed.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [31]:
def finish_grounded_output(output,evidence,closed_book=False):
    """Apply a conservative lexical answer-occurrence gate to deployed RAG, with its limitations disclosed."""
    proxy=lexical_grounding(output['answer'],evidence);output.update(proxy)
    output['raw_model_answer']=output['answer'];output['hallucination_index']=None
    if not closed_book and CFG['require_answer_occurrence'] and not proxy['answer_occurs_in_evidence']:
        output['answer']=ABSTAIN;output['abstention_reason']='Answer not found verbatim after normalization in supplied evidence (heuristic).'
    output['abstained']=normalize_answer(output['answer'])==normalize_answer(ABSTAIN)
    output['evidence']=evidence;return output


**Helper: `answer_desktop`**

Retrieve across the frozen shard scope, then concatenate or generate candidates per passage and merge.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [32]:
def answer_desktop(question,k,method,runtime,hits=None):
    """Retrieve across the frozen shard scope, then concatenate or generate candidates per passage and merge."""
    started=time.perf_counter();question=str(question).strip()
    if not question or len(question)>CFG['max_question_chars']:raise ValueError('Question is empty or exceeds the character limit.')
    t=time.perf_counter();hits=retrieve_desktop(question,k,runtime.embedder,runtime.parts) if hits is None else hits[:k]
    retrieval_ms=(time.perf_counter()-t)*1000;tokenizer,model=runtime.tokenizer,runtime.model
    eligible=[h for h in hits if h.get('_distance',0)<=CFG['max_cosine_distance']]
    if not eligible:return {'answer':ABSTAIN,'abstained':True,'evidence':[],'retrieved':hits,'input_tokens':0,'output_tokens':0,
        'generation_ms':0.,'retrieval_ms':retrieval_ms,'total_ms':(time.perf_counter()-started)*1000,'model_calls':0,
        'hallucination_index':None,'unsupported_sentence_proxy':None,'abstention_reason':'No passage passed the heuristic retrieval-distance gate.'}
    evidence,prompt=pack_desktop_evidence(question,eligible,tokenizer);intermediate=[]
    if method=='concat':output=generate_desktop(prompt,tokenizer,model);calls=1
    elif method=='generate_merge':
        output,calls,intermediate=generate_merge_desktop(question,evidence,runtime)
        evidence=output.pop('merge_evidence')
    else:raise ValueError('Unknown fusion method.')
    output=finish_grounded_output(output,evidence)
    output.update(retrieved=hits,retrieval_ms=retrieval_ms,total_ms=(time.perf_counter()-started)*1000,
        model_calls=calls,intermediate=intermediate)
    return output


**Helper: `generate_merge_desktop`**

Generated candidates are unverified; merge prompts retain original source passages within the same budget.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [33]:
def generate_merge_desktop(question,evidence,runtime):
    """Generated candidates are unverified; merge prompts retain original source passages within the same budget."""
    partial=[]
    for passage in evidence:
        _,prompt=pack_desktop_evidence(question,[passage],runtime.tokenizer)
        partial.append(generate_desktop(prompt,runtime.tokenizer,runtime.model))
    candidates='\n'.join(f'{i+1}. {p["answer"]}' for i,p in enumerate(partial))
    instruction='\nUnverified candidate answers (check against evidence):\n'+candidates
    selected=list(evidence)
    while True:
        prompt=render_prompt(question,selected)+instruction
        if len(runtime.tokenizer.encode(prompt))<=CFG['input_cap']:break
        if len(instruction)>80:instruction=instruction[:max(80,len(instruction)//2)]
        elif selected:selected=selected[:-1]
        else:raise ValueError('Merge instructions exceed the input budget.')
    result=generate_desktop(prompt,runtime.tokenizer,runtime.model)
    result['input_tokens']+=sum(p['input_tokens'] for p in partial)
    result['output_tokens']+=sum(p['output_tokens'] for p in partial)
    result['generation_ms']+=sum(p['generation_ms'] for p in partial)
    # Return the actual original passages in the merge prompt, not dropped passages.
    result['merge_evidence']=selected
    return result,len(partial)+1,[p['answer'] for p in partial]


**Helper: `corpus_snapshot`**

Bind evaluation/deployment to immutable vector contents, separately from mutable training/status fields.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [34]:
def corpus_snapshot(parts):
    """Bind evaluation/deployment to immutable vector contents, separately from mutable training/status fields."""
    return [{'shard':p['number'],'passages':p['record']['passages'],
        'fingerprint':digest({'project':p['record']['project'],'files':p['record']['files'],
            'passages':p['record']['passages'],'index_type':p['record']['index_type']})} for p in parts]


**Helper: `desktop_status`**

Separate completed workflow stages, searchable vector coverage and the deployed model's exact scope.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [35]:
def desktop_status():
    """Separate completed workflow stages, searchable vector coverage and the deployed model's exact scope."""
    rows=[]
    for number in range(1,len(LOCK['files'])+1):
        r=shard_record(number)
        rows.append({'shard':number,'download':r['download'],'embedding':r['embedding'],'indexing':r['indexing'],
            'training':r['training'],'evaluation':r['evaluation'],'passages':r.get('passages',0),
            'training_note':r.get('training_reason',''),'model_version':r.get('model_version','')})
    return rows


**Helper: `existing_qa_connection`**

Check the actual loopback server and release ID before reusing a saved connection record.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [36]:
def existing_qa_connection():
    """Check the actual loopback server and release ID before reusing a saved connection record."""
    p=ROOT/'qa_connection.json'
    if not p.exists():return None
    saved=read_json(p);parsed=urllib.parse.urlsplit(saved['url'])
    if parsed.scheme!='http' or parsed.hostname!='127.0.0.1' or not parsed.port:raise ValueError('Invalid saved local server URL.')
    try:
        with urllib.request.urlopen(saved['url']+'/health',timeout=2) as response:health=json.load(response)
        if health['release']==saved['release'] and health['project']==LOCK['identity']:return saved
    except (OSError,ValueError,KeyError):pass
    return None


**Helper: `stop_existing_qa`**

Ask only this project's token-identified local server to exit before any new writes; never kill arbitrary PIDs.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [37]:
def stop_existing_qa():
    """Ask only this project's token-identified local server to exit before any new writes; never kill arbitrary PIDs."""
    saved=existing_qa_connection()
    if saved:
        request=urllib.request.Request(saved['url']+'/api/shutdown',
            data=json.dumps({'token':saved['token']}).encode(),headers={'Content-Type':'application/json'},method='POST')
        with urllib.request.urlopen(request,timeout=10) as response:response.read()
    deadline=time.perf_counter()+20
    while True:
        try:handle=acquire_local_lock(ROOT/'desktop_qa.lock')
        except BlockingIOError:
            if time.perf_counter()>deadline:raise RuntimeError('The previous Q&A process has not exited; close it before starting new shard processing.')
            time.sleep(.2);continue
        release_local_lock(handle);return


**Helper: `report_json_safe`**

Keep exported JSON strict: non-finite measurements are unavailable, never fabricated zeros.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [38]:
def report_json_safe(value):
    """Keep exported JSON strict: non-finite measurements are unavailable, never fabricated zeros."""
    if isinstance(value,dict):return {str(k):report_json_safe(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)):return [report_json_safe(v) for v in value]
    if isinstance(value,(float,np.floating)):return float(value) if math.isfinite(value) else None
    if isinstance(value,np.integer):return int(value)
    if isinstance(value,Path):return str(value)
    return value


**Helper: `append_report_jsonl`**

Persist diagnostics after each event; only a torn final line may be recovered when reading.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [39]:
def append_report_jsonl(path,row):
    """Persist diagnostics after each event; only a torn final line may be recovered when reading."""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with open(path,'a',encoding='utf-8') as handle:
        handle.write(json.dumps(report_json_safe(row),ensure_ascii=False,allow_nan=False)+'\n')
        handle.flush();os.fsync(handle.fileno())


**Helper: `read_report_jsonl`**

Read optional telemetry without treating an interrupted final record as a completed event.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [40]:
def read_report_jsonl(path):
    """Read optional telemetry without treating an interrupted final record as a completed event."""
    path=Path(path)
    if not path.exists():return []
    lines=path.read_text(encoding='utf-8').splitlines();rows=[]
    for number,line in enumerate(lines):
        if not line.strip():continue
        try:rows.append(json.loads(line))
        except json.JSONDecodeError:
            if number!=len(lines)-1:raise ValueError('Corrupt diagnostic JSONL: '+str(path))
    return rows


**Helper: `record_desktop_error`**

Save the entire exception chain for new failures; historical one-line receipts cannot restore old tracebacks.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [41]:
def record_desktop_error(stage,error,number=None):
    """Save the entire exception chain for new failures; historical one-line receipts cannot restore old tracebacks."""
    item={'at':datetime.now(timezone.utc).isoformat(),'stage':str(stage),'shard':number,
        'type':type(error).__name__,'message':str(error),
        'traceback':''.join(traceback.format_exception(type(error),error,error.__traceback__))}
    append_report_jsonl(ROOT/'error_logs'/'events.jsonl',item)
    with open(ROOT/'error_logs'/'full_errors.log','a',encoding='utf-8') as handle:
        handle.write('\n['+item['at']+'] '+str(stage)+' · shard '+str(number)+'\n'+item['traceback'])


**Helper: `portable_configure`**

Keep corpus semantics fixed while selecting CPU/CUDA; no unsupported TPU/Intel-GPU substitution.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [42]:
def portable_configure(config):
    """Keep corpus semantics fixed while selecting CPU/CUDA; no unsupported TPU/Intel-GPU substitution."""
    config=dict(config)
    if config.get('device')=='auto':config['device']='cuda' if torch.cuda.is_available() else 'cpu'
    if config['chunk_tokens']!=220 or config['chunk_overlap']!=32:
        raise ValueError('This edition preserves 220-token chunks and 32-token overlap. Restore these settings.')
    configure_desktop(config)
    temporary=Path(CFG['cache_root'])/'tmp';temporary.mkdir(parents=True,exist_ok=True)
    os.environ['TMPDIR']=str(temporary)
    if CFG['device'] not in {'cpu','cuda'}:raise ValueError('Supported backends are CPU and NVIDIA CUDA.')
    os.environ['ANONYMIZED_TELEMETRY']='False'
    if DEVICE=='cuda' and torch.cuda.device_count()>1:
        raise RuntimeError('Use one visible CUDA device for portable optimizer geometry. Restart the kernel and Run All; setup selects device 0 before CUDA initializes.')
    print('Execution:',DEVICE,'| Project:',ROOT,'| Threads:',CFG['cpu_threads'])
    return CFG


**Helper: `portable_scope`**

Use the existing sealed LanceDB partitions as the source of truth, independent of derived indexes.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [43]:
def portable_scope():
    """Use the existing sealed LanceDB partitions as the source of truth, independent of derived indexes."""
    return open_desktop_collection()


**Helper: `portable_part_key`**

Bind a derived index to one immutable source partition, including all original vector-file checksums.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [44]:
def portable_part_key(part):
    """Bind a derived index to one immutable source partition, including all original vector-file checksums."""
    return digest(['portable-source-v1',LOCK['identity'],corpus_snapshot([part])[0]])[:24]


**Helper: `portable_import_bundle`**

Verify all archive members before extraction; refuse conflicting existing files and path traversal.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [45]:
def portable_import_bundle(path,destination):
    """Verify all archive members before extraction; refuse conflicting existing files and path traversal."""
    import zipfile
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        manifest=json.loads(archive.read('bundle_manifest.json'))
        if manifest.get('format')!='wiki-rag-bundle-v1':raise ValueError('Unknown sharing-bundle format.')
        expected={'project/'+name for name in manifest['files']}|{'bundle_manifest.json'}
        if set(archive.namelist())!=expected or len(archive.namelist())!=len(expected):raise ValueError('Unexpected/duplicate archive members.')
        pending=[];required=0
        for name,record in manifest['files'].items():
            relative=Path(name)
            if relative.is_absolute() or '..' in relative.parts or '\\' in name or ':' in name:raise ValueError('Unsafe archive path.')
            target=(destination/relative).resolve()
            if not target.is_relative_to(destination):raise ValueError('Archive path escapes the destination.')
            info=archive.getinfo('project/'+name)
            if info.file_size!=record['size']:raise ValueError('Archive size/manifest mismatch.')
            with archive.open(info) as handle:
                checksum=hashlib.sha256()
                for block in iter(lambda:handle.read(1024*1024),b''):checksum.update(block)
            if checksum.hexdigest()!=record['sha256']:raise ValueError('Archive checksum mismatch: '+name)
            if target.exists():
                if not target.is_file() or file_sha(target)!=record['sha256']:raise ValueError('Existing file conflicts with the bundle: '+str(target))
            else:pending.append((name,target,record));required+=record['size']
        if shutil.disk_usage(destination).free<required+500_000_000:raise BudgetPause('Insufficient space for bundle extraction.')
        for index,(name,target,record) in enumerate(pending):
            target.parent.mkdir(parents=True,exist_ok=True);temporary=target.with_name(target.name+'.import.tmp')
            with archive.open('project/'+name) as source,open(temporary,'wb') as output:shutil.copyfileobj(source,output)
            if file_sha(temporary)!=record['sha256']:raise ValueError('Extracted checksum mismatch.')
            os.replace(temporary,target)
    atomic_json(destination/'bundle_import_receipt.json',{'archive':Path(path).name,'kind':manifest['kind'],
        'release_id':manifest['release_id'],'verified_files':len(manifest['files'])})
    return manifest


**Helper: `portable_restore_inputs`**

Restore one attached bundle automatically into an empty project; existing desktop work always takes precedence.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [46]:
def portable_restore_inputs(environment):
    """Restore one attached bundle automatically into an empty project; existing desktop work always takes precedence."""
    root=environment['root']
    if (root/'desktop_project_lock.json').exists():return {'status':'existing project retained'}
    candidates=[]
    if environment['kaggle']:
        candidates=sorted(Path('/kaggle/input').glob('**/wiki_rag_*.zip'))
    else:
        candidates=sorted(Path.cwd().glob('wiki_rag_*.zip'))+sorted(root.parent.glob('wiki_rag_*.zip'))
    candidates=list(dict.fromkeys(p.resolve() for p in candidates))
    if len(candidates)>1:raise ValueError('Multiple bundles found. Keep/attach only the intended bundle for automatic restoration.')
    if candidates:return portable_import_bundle(candidates[0],root)
    return {'status':'new project; no input bundle found'}


**Helper: `portable_release_pointer`**

Prefer the new release while retaining a read-only fallback to a previously saved desktop release.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [47]:
def portable_release_pointer():
    """Prefer the new release while retaining a read-only fallback to a previously saved desktop release."""
    path=ROOT/'portable_active_release.json'
    if not path.exists():path=ROOT/'active_release.json'
    if not path.exists():raise FileNotFoundError('No saved final release exists. Run the process/benchmark modules first.')
    return read_json(path)


**Helper: `portable_validate_release`**

Validate checksums and relative paths before loading models; refuse silently missing or changed source partitions.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [48]:
def portable_validate_release(release):
    """Validate checksums and relative paths before loading models; refuse silently missing or changed source partitions."""
    if release['project']!=LOCK['identity']:raise ValueError('Saved release belongs to a different project.')
    for field in ['generator','embedder','report']:
        path=Path(release[field])
        if path.is_absolute() or '..' in path.parts:raise ValueError('Release contains an unsafe/nonportable path.')
    target=ROOT/release['generator'];marker=target/'saved_model.json'
    if file_sha(marker)!=release['model_manifest_sha256']:raise ValueError('Saved generator manifest changed.')
    check_inventory(target,read_json(marker)['files'],True)
    embedder=ROOT/release['embedder'];check_inventory(embedder,read_json(embedder/'desktop_model.json')['files'],True)
    folder=ROOT/release['report']
    if file_sha(folder/'assessment.json')!=release['report_sha256']:raise ValueError('Saved assessment changed.')
    report=read_json(folder/'assessment.json');check_inventory(folder,report['files'],True)
    parts=open_desktop_collection([r['shard'] for r in release['corpus']])
    if corpus_snapshot(parts)!=release['corpus']:raise ValueError('Saved vector snapshot changed.')
    if release.get('reranker'):
        if file_sha(ROOT/release['reranker']/'source.json')!=release['reranker_manifest_sha256']:
            raise ValueError('Saved reranker revision/manifest changed.')
    return parts,report


**Helper: `portable_state_fingerprint`**

Detect another writer between module cells without retaining an OS lock while the notebook is idle.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [49]:
def portable_state_fingerprint():
    """Detect another writer between module cells without retaining an OS lock while the notebook is idle."""
    paths=[ROOT/'training_head.json',ROOT/'active_release.json',ROOT/'portable_active_release.json']
    paths+=sorted((ROOT/'parts').glob('shard_*/desktop_manifest.json'))
    return digest({p.relative_to(ROOT).as_posix():read_json(p) for p in paths if p.exists()})


**Helper: `portable_initialize`**

Restore the project identity first; automatic mode compares existing shards without adding a fifth shard.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [50]:
def portable_initialize(choice):
    """Restore the project identity first; automatic mode compares existing shards without adding a fifth shard."""
    global PORTABLE_RUN,DESKTOP_RUN
    config=dict(DESKTOP_CONFIG);config['enable_reranker']=choice['reranker'];config['run_cpu_profile']=choice['cpu_profile']
    portable_configure(config);handle=acquire_local_lock(ROOT/'desktop_writer.lock')
    try:
        resolve_desktop_lock();stop_existing_qa()
        available=[n for n in range(1,42) if shard_record(n).get('indexing')=='complete']
        mode=choice['mode']
        imported=ROOT/'bundle_import_receipt.json'
        validation_bundle=imported.exists() and read_json(imported).get('kind')=='validation' and not (ROOT/'training_head.json').exists()
        if mode=='auto':
            selected=shard_record(int(choice['number']))
            unfinished=selected.get('embedding')=='in progress' or (selected.get('embedding')=='complete'
                and (selected.get('indexing')!='complete' or selected.get('training') not in {'complete','skipped'}))
            mode='validate' if validation_bundle else ('process' if unfinished or not available else 'compare')
        if mode=='process' and validation_bundle:
            raise ValueError('A validation bundle omits optimizer/model lineage. Restore a resume bundle to continue shard training safely.')
        if mode=='compare' and not available:raise ValueError('No indexed shards exist yet. Choose Process selected shard.')
        if mode in {'validate','final_test','export_validation','export_resume'}:portable_validate_release(portable_release_pointer())
        if mode=='final_test' and any(not (ROOT/'qa'/f'shard_{n-1:03d}'/'receipt.json').exists() for n in available):
            raise ValueError('Final-test labels are absent. Restore a resume bundle; validation bundles intentionally omit reserved QA.')
        PORTABLE_RUN={'number':int(choice['number']),'mode':mode,'done':set(),'results':{},'parts':[],
            'configuration':digest(CFG),'fingerprint':portable_state_fingerprint(),'release':None}
        DESKTOP_RUN=PORTABLE_RUN;atomic_json(ROOT/'portable_config.json',CFG)
        print('Workflow:',mode,'| Selected shard:',choice['number'],'| Existing indexed shards:',available)
        print('No existing passage embeddings will be recomputed.')
        return desktop_status()
    finally:release_local_lock(handle)


**Helper: `portable_run_module`**

Execute one visible module with prerequisite checks, error logs, disk guards and a short-lived writer lock.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [51]:
def portable_run_module(name,operation,requires=(),modes=None):
    """Execute one visible module with prerequisite checks, error logs, disk guards and a short-lived writer lock."""
    run=globals().get('PORTABLE_RUN')
    if not run:raise RuntimeError('Run setup/selection and Module 1 first; notebook outputs do not restore Python variables.')
    if digest(CFG)!=run['configuration']:raise RuntimeError('Configuration changed. Rerun Module 1 before continuing.')
    missing=set(requires)-run['done']
    if missing:raise RuntimeError('Run prerequisite modules first: '+', '.join(sorted(missing)))
    handle=acquire_local_lock(ROOT/'desktop_writer.lock')
    try:
        if portable_state_fingerprint()!=run['fingerprint']:raise RuntimeError('Another run changed saved project state. Rerun Module 1.')
        if name in run['done'] and name!='interface':return run['results'][name]
        began=time.perf_counter();print(name+' · starting',flush=True)
        try:
            if modes is not None and run['mode'] not in modes:result={'status':'skipped for workflow '+run['mode']}
            else:result=operation()
        except BaseException as error:
            record_desktop_error(name,error,run['number']);raise
        finally:run['fingerprint']=portable_state_fingerprint()
        run['results'][name]=result;run['done'].add(name)
        print(f'{name} · complete · {time.perf_counter()-began:.1f} s',flush=True);return result
    finally:release_local_lock(handle)


**Helper: `portable_progress`**

Expose actual committed/query counts to the parent notebook; never invent a completion percentage.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [52]:
def portable_progress(phase,done=None,total=None):
    """Expose actual committed/query counts to the parent notebook; never invent a completion percentage."""
    atomic_json(ROOT/'portable_progress.json',{'phase':phase,'done':done,'total':total,
        'at':datetime.now(timezone.utc).isoformat()})


**Helper: `portable_export_runtime`**

Export currently defined library functions so heavy tasks and the browser can run in clean child processes.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [53]:
def portable_export_runtime():
    """Export currently defined library functions so heavy tasks and the browser can run in clean child processes."""
    names=[n for n in PORTABLE_EXPORT_NAMES if n in globals() and inspect.isfunction(globals()[n])]
    preamble=PORTABLE_MODULE_HEADER+'\nPORTABLE_EXPORT_NAMES='+repr(PORTABLE_EXPORT_NAMES)+'\nPORTABLE_MODULE_HEADER='+repr(PORTABLE_MODULE_HEADER)+'\n'
    text=preamble+'\n\n'+'\n\n'.join(inspect.getsource(globals()[n]) for n in names)
    if 'portable_cli' in names:text+="\nif __name__=='__main__':portable_cli()\n"
    compile(text,'portable_rag.py','exec')
    target=ROOT/'portable_rag.py';temporary=target.with_suffix('.tmp')
    temporary.write_text(text,encoding='utf-8');os.replace(temporary,target)
    atomic_json(ROOT/'portable_config.json',CFG)
    return target


**Helper: `portable_spawn`**

Isolate database workers, show heartbeat/real counters, retain full logs, and stop children on notebook interruption.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [54]:
def portable_spawn(task,payload=None):
    """Isolate database workers, show heartbeat/real counters, retain full logs, and stop children on notebook interruption."""
    portable_export_runtime();job=ROOT/'portable_jobs'/uuid.uuid4().hex;job.mkdir(parents=True)
    atomic_json(job/'request.json',{'task':task,'payload':payload or {}})
    script='import portable_rag as p; p.portable_worker('+repr(str(job))+')'
    env=os.environ.copy();env.update(PYTHONIOENCODING='utf-8',ANONYMIZED_TELEMETRY='False')
    portable_progress('Starting '+task,0,None)
    with open(job/'worker.log','w',encoding='utf-8') as log:
        process=subprocess.Popen([sys.executable,'-u','-c',script],cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT)
        try:
            with tqdm(total=None,desc=task+' · worker elapsed',unit='s') as bar:
                while process.poll() is None:
                    time.sleep(1);bar.update(1);path=ROOT/'portable_progress.json'
                    if path.exists():
                        try:
                            state=read_json(path);bar.set_postfix_str(f'{state["phase"]}: {state.get("done")} / {state.get("total")}')
                        except (OSError,ValueError):pass
                    guard_resources()
        except BaseException:
            process.terminate()
            try:process.wait(timeout=20)
            except subprocess.TimeoutExpired:process.kill();process.wait(timeout=10)
            raise
    if process.returncode or not (job/'result.json').exists():
        message=(job/'worker.log').read_text(encoding='utf-8',errors='replace')[-5000:]
        raise RuntimeError('Worker '+task+' failed; committed checkpoints retained. Log: '+str(job/'worker.log')+'\n'+message)
    return read_json(job/'result.json')


**Helper: `portable_worker`**

Dispatch a single child task using relative project paths and the parent's saved configuration.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [55]:
def portable_worker(job):
    """Dispatch a single child task using relative project paths and the parent's saved configuration."""
    job=Path(job);portable_configure(read_json(Path('portable_config.json')));resolve_desktop_lock()
    request=read_json(job/'request.json');task=request['task'];payload=request['payload']
    if task=='derived_indexes':
        copied=portable_migrate_all() if CFG['compare_chroma'] else []
        result={'chroma':copied,'bm25':[portable_build_bm25(p) for p in portable_scope()]}
    elif task=='query_reference':result=portable_query_reference(payload['questions'])
    elif task=='retrieval':result=portable_retrieval_worker(payload)
    elif task=='generation':result=portable_generation_worker(payload)
    elif task=='verify':result=portable_offline_check_body()
    elif task=='final_test':result=portable_final_test_worker(payload)
    elif task=='diagnostics':result=portable_unsupported_probes(portable_load_release())
    elif task=='cpu_profile':result=portable_cpu_profile(portable_scope())
    elif task=='export':result=portable_export_bundle(payload.get('kind','validation'))
    else:raise ValueError('Unknown worker task: '+task)
    atomic_json(job/'result.json',report_json_safe(result))


**Helper: `portable_show_selection`**

Present the selected shard plus workflow mode; completed shards are never silently advanced to a new download.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [56]:
def portable_show_selection():
    """Present the selected shard plus workflow mode; completed shards are never silently advanced to a new download."""
    global SHARD_PICKER,SHARD_GATE,SHARD_PROCEED,SHARD_SELECTION_NOTE,SHARD_PANEL,MODE_PICKER,RERANK_PICKER,CPU_PROFILE_PICKER
    saved=PROJECT_ROOT/'desktop_selection.json';last=int(read_json(saved)['shard']) if saved.exists() else 1
    SHARD_GATE={'number':None,'used':False,'cancelled':False,'project_root':str(PROJECT_ROOT),'id':uuid.uuid4().hex}
    SHARD_PICKER=widgets.Dropdown(options=[(shard_selection_label(n),n) for n in range(1,42)],value=last,description='Shard:')
    MODE_PICKER=widgets.Dropdown(options=[('Auto: reuse/compare existing; process shard 1 if new','auto'),
        ('Validation only: saved model + database','validate'),('Process selected shard, then compare','process'),
        ('Compare existing indexed shards only','compare'),('Final held-out test: freeze current release','final_test'),
        ('Export validation bundle','export_validation'),('Export full resume bundle','export_resume')],value='auto',description='Workflow:',layout=widgets.Layout(width='100%'))
    RERANK_PICKER=widgets.Checkbox(value=bool(DESKTOP_CONFIG['enable_reranker']),description='Include optional local reranker (extra inference cost)')
    CPU_PROFILE_PICKER=widgets.Checkbox(value=False,description='Run optional batch/thread microbenchmark')
    SHARD_PROCEED=widgets.Button(description='Proceed',button_style='success',icon='play')
    SHARD_SELECTION_NOTE=widgets.HTML('<b>Choose the workflow, then Proceed.</b> Auto keeps the saved shard selection; existing vectors are reused.')
    def approve(button):
        if SHARD_GATE['number'] is not None:return
        SHARD_GATE.update(number=int(SHARD_PICKER.value),mode=MODE_PICKER.value,used=True,
            reranker=bool(RERANK_PICKER.value),cpu_profile=bool(CPU_PROFILE_PICKER.value))
        atomic_json(saved,{'shard':int(SHARD_PICKER.value)})
        for widget in [SHARD_PICKER,MODE_PICKER,RERANK_PICKER,CPU_PROFILE_PICKER,SHARD_PROCEED]:widget.disabled=True
        SHARD_SELECTION_NOTE.value='<b>Confirmed.</b> Module cells may now continue.'
    SHARD_PROCEED.on_click(approve)
    SHARD_PANEL=widgets.VBox([widgets.HTML('<h3>Wikipedia RAG · choose a workflow</h3>'),SHARD_PICKER,MODE_PICKER,
        RERANK_PICKER,CPU_PROFILE_PICKER,SHARD_SELECTION_NOTE,SHARD_PROCEED]);display(SHARD_PANEL)


**Helper: `portable_wait_selection`**

Keep interactive Proceed behaviour while supporting unattended Kaggle version runs without a frontend.

**Why here:** All later modules share these checks and the same immutable corpus/answer policy; definitions do not start processing.


In [57]:
async def portable_wait_selection():
    """Keep interactive Proceed behaviour while supporting unattended Kaggle version runs without a frontend."""
    if ENVIRONMENT['batch'] or DESKTOP_CONFIG.get('_headless'):
        path=PROJECT_ROOT/'desktop_selection.json';number=int(read_json(path)['shard']) if path.exists() else 1
        return {'number':number,'mode':DESKTOP_CONFIG.get('_headless_mode','auto'),'reranker':False,'cpu_profile':False}
    if not globals().get('SHARD_GATE') or SHARD_GATE.get('used'):portable_show_selection()
    from jupyter_ui_poll import ui_events
    import asyncio
    try:
        kernel=get_ipython().kernel
        if hasattr(kernel,'do_one_iteration'):
            async with ui_events() as poll:
                while not SHARD_GATE['used']:await poll(10);await asyncio.sleep(.05)
        else:
            while not SHARD_GATE['used']:await asyncio.sleep(.05)
    except BaseException:
        SHARD_GATE['cancelled']=True;raise
    return dict(SHARD_GATE)


**Selector helper.** Read display-only progress without loading models or contacting the dataset host. This reads display metadata without downloading data or loading a model.


In [58]:
def shard_selection_record(number):
    """Read display-only progress without loading models or contacting the dataset host."""
    path=PROJECT_ROOT/'parts'/f'shard_{int(number)-1:03d}'/'desktop_manifest.json'
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


**Selector helper.** Treat a shard as finished only when every persisted workflow stage is finished. This reads display metadata without downloading data or loading a model.


In [59]:
def shard_selection_complete(record):
    """Treat a shard as finished only when every persisted workflow stage is finished."""
    return (record.get('embedding')=='complete' and record.get('indexing')=='complete'
        and record.get('training') in {'complete','skipped'} and record.get('evaluation')=='complete')


**Selector helper.** Clearly distinguish completed, partially processed and never-downloaded shard choices. This reads display metadata without downloading data or loading a model.


In [60]:
def shard_selection_label(number):
    """Clearly distinguish completed, partially processed and never-downloaded shard choices."""
    record=shard_selection_record(number)
    if shard_selection_complete(record):state='complete · validate saved output'
    elif record.get('embedding')=='in progress':state=f'resume embedding · {record.get("cursor",0):,} articles committed'
    elif record.get('indexing')=='complete':state='indexed · finish remaining stages'
    elif record.get('embedding')=='complete':state='embedded · indexing pending'
    elif record.get('download')=='complete':state='downloaded · embedding pending'
    else:state='not downloaded'
    return f'Shard {number:02d}/41 · {state}'


**Compatibility helpers.** Preserve the latest notebook’s previous report/assessment functions for reading historical artifacts. The portable workflow does not invoke the old all-in-one assessment or CLI.


**Helper: `completed_desktop_output`**

Return the current matching saved output without loading QA source or restarting completed training.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [61]:
def completed_desktop_output(number):
    """Return the current matching saved output without loading QA source or restarting completed training."""
    record=shard_record(number);pointer=ROOT/'active_release.json'
    if record.get('indexing')!='complete' or record.get('training') not in {'complete','skipped'} or record.get('evaluation')!='complete' or not pointer.exists():return None
    release=read_json(pointer);parts=open_desktop_collection()
    if release['corpus']!=corpus_snapshot(parts):return None
    proof=ROOT/release['report']/'offline_verification.json'
    if not proof.exists() or not read_json(proof).get('passed'):return None
    report=read_json(ROOT/release['report']/'assessment.json')
    versions={x:metadata.version(x) for x in ['torch','transformers','sentence-transformers','lancedb']}
    if report['spec']['software']!=versions or report['spec']['dev_limit']!=CFG['dev_questions']:return None
    if record['training']=='complete':verify_saved_reader(ROOT/record['trained_model'])
    if report['spec']['code']!=digest(inspect.getsource(answer_desktop)+inspect.getsource(generate_merge_desktop)+inspect.getsource(pack_desktop_evidence)+inspect.getsource(finish_grounded_output)):
        raise ValueError('Inference code changed. Existing comparisons cannot be silently reused.')
    for name,value in release['configuration'].items():
        if CFG[name]!=value:raise ValueError('Inference setting changed: '+name+'. Restore it or create a separate evaluation run.')
    manifest=ROOT/release['generator']/'saved_model.json'
    if file_sha(manifest)!=release['model_manifest_sha256']:raise ValueError('Saved model manifest checksum mismatch.')
    check_inventory(ROOT/release['generator'],read_json(manifest)['files'],True)
    check_inventory(ROOT/release['report'],report['files'],True)
    if file_sha(ROOT/release['report']/'assessment.json')!=release['report_sha256']:raise ValueError('Saved report checksum mismatch.')
    return release


**Helper: `desktop_cli`**

The browser backend loads local configuration and saved artifacts; it never invokes the training pipeline.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [62]:
def desktop_cli():
    """The browser backend loads local configuration and saved artifacts; it never invokes the training pipeline."""
    import argparse
    parser=argparse.ArgumentParser();parser.add_argument('mode',choices=['serve']);parser.add_argument('project_root')
    args=parser.parse_args();root=Path(args.project_root).resolve();config=read_json(root/'desktop_config.json')
    config['project_root']=str(root);config['offline']=True;configure_desktop(config);resolve_desktop_lock()
    serve_desktop_qa()


**Helper: `development_questions`**

Choose a deterministic CPU-sized development subset; reserved final-test questions are never executed here.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [63]:
def development_questions(numbers):
    """Choose a deterministic CPU-sized development subset; reserved final-test questions are never executed here."""
    rows=all_matched_qa(numbers);dev=[r for r in rows if r['split']=='dev']
    # Interleave shards to retain earlier-shard coverage as the vector collection grows.
    grouped={n:sorted([r for r in dev if r['shard']==n],key=lambda r:digest(r['id'])) for n in numbers}
    result=[];index=0
    while len(result)<CFG['dev_questions']:
        added=False
        for n in sorted(grouped):
            if index<len(grouped[n]):result.append(grouped[n][index]);added=True
            if len(result)>=CFG['dev_questions']:break
        if not added:break
        index+=1
    return result,dict(collections.Counter(r['split'] for r in rows))


**Helper: `evaluation_spec`**

A larger corpus or a changed reader creates a new comparison identity, never stale reused scores.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [64]:
def evaluation_spec(parts,readers,questions):
    """A larger corpus or a changed reader creates a new comparison identity, never stale reused scores."""
    return {'protocol':'development-preview-v1','project':LOCK['identity'],'corpus':corpus_snapshot(parts),
        'readers':{k:v['id'] for k,v in readers.items()},'questions':digest(questions),'dev_limit':CFG['dev_questions'],
        'k_values':CFG['k_values'],'methods':['concat','generate_merge'],'input_cap':CFG['input_cap'],
        'max_new_tokens':CFG['max_new_tokens'],'max_cosine_distance':CFG['max_cosine_distance'],
        'require_answer_occurrence':CFG['require_answer_occurrence'],'nprobes':CFG['nprobes'],
        'code':digest(inspect.getsource(answer_desktop)+inspect.getsource(generate_merge_desktop)+inspect.getsource(pack_desktop_evidence)+inspect.getsource(finish_grounded_output)),
        'software':{x:metadata.version(x) for x in ['torch','transformers','sentence-transformers','lancedb']}}


**Helper: `run_desktop_assessment`**

Compare on matching cumulative corpus scope; preserve final-test references for later independent assessment.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [65]:
def run_desktop_assessment(parts,updated,previous):
    """Compare on matching cumulative corpus scope; preserve final-test references for later independent assessment."""
    numbers=[p['number'] for p in parts];questions,counts=development_questions(numbers)
    readers=reader_candidates(updated,previous);spec=evaluation_spec(parts,readers,questions)
    run_id=digest(spec)[:24];folder=ROOT/'runs'/run_id;folder.mkdir(parents=True,exist_ok=True)
    receipt=folder/'assessment.json'
    if receipt.exists():
        saved=read_json(receipt)
        if saved['spec']!=spec:raise ValueError('Assessment identity mismatch.')
        check_inventory(folder,saved['files'],True);return saved,folder,readers
    atomic_json(folder/'protocol.json',spec)
    records=run_development_grid(folder,parts,readers,questions) if questions else []
    summary,figures=export_desktop_matrices(folder,records)
    chosen=select_development_reader(summary,readers,len(questions))
    report={'run_id':run_id,'spec':spec,'evaluation_scope':'DEVELOPMENT PREVIEW — reserved final test not executed',
        'coverage':numbers,'expected_shards':len(LOCK['files']),'question_count':len(questions),'qa_counts':counts,
        'selected':chosen,'summary':summary,'figures':figures,'hallucination_index':None,'quality_approved':False,
        'limitations':['Automatic title/alias/context matching is not independent human QA validation.',
            'Reference EM/F1 and lexical proxies do not establish factual correctness.',
            'No calibrated confidence or corpus-negative specificity is claimed.',
            'Full-corpus coverage, independently adjudicated hallucination and untouched final-test acceptance remain pending.']}
    report['files']=file_inventory(folder,{'assessment.json'})
    atomic_json(receipt,report);return report,folder,readers


**Helper: `run_development_grid`**

Cache each measured answer after generation; new visitor validation does not trigger this grid again.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [66]:
def run_development_grid(folder,parts,readers,questions):
    """Cache each measured answer after generation; new visitor validation does not trigger this grid again."""
    path=Path(folder)/'predictions.jsonl';cache=read_evaluation_cache(path);records=[]
    embedder=load_desktop_embedder()
    try:
        for label,reader in readers.items():
            tokenizer,model=load_generator(reader['path']);model.eval();model.requires_grad_(False)
            runtime=types.SimpleNamespace(tokenizer=tokenizer,model=model,embedder=embedder,parts=parts)
            total=len(questions)*len(CFG['k_values'])*2
            with tqdm(total=total,desc=label+' · development answers',unit='answer') as bar:
                for question in questions:
                    for method in ['concat','generate_merge']:
                        for k in CFG['k_values']:
                            key=digest([question['id'],reader['id'],method,k])
                            if key not in cache:
                                output=answer_desktop(question['question'],k,method,runtime)
                                score=score_desktop_prediction(question,output,label,k,method)
                                row={'key':key,'score':score,'output':output,'question':question}
                                with open(path,'a',encoding='utf-8') as handle:
                                    handle.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+'\n');handle.flush();os.fsync(handle.fileno())
                                cache[key]=row
                            records.append(cache[key]);bar.update(1);guard_resources()
            del model,runtime;gc.collect()
    finally:
        del embedder;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    return records


**Helper: `save_deployment`**

Copy actual selected weights into a versioned final-model bundle and bind reports/vectors to that exact release.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [67]:
def save_deployment(report,folder,readers,parts):
    """Copy actual selected weights into a versioned final-model bundle and bind reports/vectors to that exact release."""
    reader=readers[report['selected']['reader']];release_id=report['run_id']
    target=ROOT/'final_models'/release_id/'generator';manifest=target/'saved_model.json'
    if not manifest.exists():
        target.mkdir(parents=True,exist_ok=True)
        for source in Path(reader['path']).rglob('*'):
            if not source.is_file() or '.cache' in source.parts:continue
            relative=source.relative_to(reader['path']);dest=target/relative
            if dest.is_file() and file_sha(dest)==file_sha(source):continue
            guard_resources(source.stat().st_size);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,dest)
        atomic_json(manifest,{'reader_id':reader['id'],'files':file_inventory(target,{'saved_model.json'})})
    saved=read_json(manifest);check_inventory(target,saved['files'],True)
    release={'id':release_id,'project':LOCK['identity'],'selected':report['selected'],
        'generator':target.relative_to(ROOT).as_posix(),'model_manifest_sha256':file_sha(manifest),
        'embedder':'models/embedder','report':folder.relative_to(ROOT).as_posix(),
        'report_sha256':file_sha(folder/'assessment.json'),'corpus':corpus_snapshot(parts),
        'configuration':{k:CFG[k] for k in ['input_cap','max_new_tokens','max_question_chars','max_cosine_distance',
            'require_answer_occurrence','nprobes','k_values']}}
    atomic_json(ROOT/'releases'/(release_id+'.json'),release)
    atomic_json(ROOT/'active_release.json',release)
    return release


**Helper: `saved_pipeline_offline_check`**

Verify local saved-model execution, allowing Windows event-loop IPC; not an accuracy test or OS firewall.

**Why here:** Historical artifacts may refer to these functions; preserving them avoids silently changing old report semantics.


In [68]:
def saved_pipeline_offline_check():
    """Verify local saved-model execution, allowing Windows event-loop IPC; not an accuracy test or OS firewall."""
    import os,sys,subprocess,json
    script=offline_socket_guard_script()+'''import desktop_rag as d
from pathlib import Path
c=d.read_json(Path('desktop_config.json'));c['offline']=True;d.configure_desktop(c);d.resolve_desktop_lock()
r=d.load_saved_deployment();s=r.release['selected']
h=r.parts[0]['table'].search().select(['passage_id','article_id','title','url','text','source_shard']).limit(1).to_list()[0]
h['_distance']=0.0
o=d.answer_desktop('What does this source describe?',s['k'],s['method'],r,hits=[h])
assert o['model_calls']>0 and o['output_tokens']>0,'No saved generator call completed'
print('OFFLINE_PROOF='+json.dumps({'passed':True,'answer':o['answer'],'shards':[p['number'] for p in r.parts],
    'model_calls':o['model_calls'],'network_policy':'loopback-only Python sockets; HF offline; saved local models'}))
'''
    env=os.environ.copy();env.update(HF_HUB_OFFLINE='1',HF_DATASETS_OFFLINE='1',PYTHONIOENCODING='utf-8')
    try:
        result=subprocess.run([sys.executable,'-c',script],cwd=ROOT,env=env,capture_output=True,text=True,encoding='utf-8',timeout=600)
    except subprocess.TimeoutExpired:
        return {'passed':False,'error':'Saved-model verification exceeded 600 seconds; no earlier stage was restarted.'}
    (ROOT/'offline_verification.log').write_text('STDOUT\n'+result.stdout+'\nSTDERR\n'+result.stderr,encoding='utf-8')
    if result.returncode:return {'passed':False,'error':result.stderr[-4000:],'log':'offline_verification.log'}
    line=next((x for x in result.stdout.splitlines() if x.startswith('OFFLINE_PROOF=')),None)
    return json.loads(line.split('=',1)[1]) if line else {'passed':False,'error':'No offline proof was returned.'}


**Restore input artifacts now.** Existing project work is retained. If the project is empty, one attached `wiki_rag_*.zip` bundle is verified and restored before the dropdown reads shard status.


In [69]:
RESTORE_STATUS=portable_restore_inputs(ENVIRONMENT)
print(RESTORE_STATUS)


{'status': 'existing project retained'}


## Disk capacity · Resume a budget-paused conversion

`BudgetPause: Configured output budget reached` means the configured project-size cap was reached.
It does not mark existing vector/index work incomplete or delete any data. The original defaults are retained:
**50 decimal GB on a laptop, 18 decimal GB on Kaggle**; these are notebook policy limits, not claims about
the storage quota currently provided by either platform. The complete project includes both LanceDB and Chroma,
raw files, models, temporary export bundles and reports. It may exceed the original 50 GB cap.

Use **Measure disk usage**, review actual physical free space, then enter a suitable project limit and
**Save limits**. No limit is increased automatically. Keep a positive physical free-space reserve.
A project limit of **0** explicitly disables only the project-size cap; the free-space guard remains active.
Limits are saved separately for laptop and Kaggle and restored on later runs. No files are removed automatically.
After saving limits, rerun **Module 1 and following modules**, choose **Compare existing indexed shards only**
to finish the interrupted Chroma/BM25 conversion, and keep the same project/cache paths.

Do not run the old and new notebooks as concurrent writers. Completed receipt-confirmed Chroma batches are
skipped. A batch committed just before a process interruption can be safely replayed by upsert.
Large source-file verification and scanning earlier batches still take time. This revision retains the
**same migration batch (512), collection names, receipt format and source-vector identities**.


**Storage helper: `portable_storage_snapshot`**

Measure project files and physical free space without touching corpus/index/model contents.


In [70]:
def portable_storage_snapshot(root):
    """Measure project files and physical free space without touching corpus/index/model contents."""
    root=Path(root).resolve();groups=collections.Counter();count=0
    for path in root.rglob('*'):
        if not path.is_file():continue
        try:size=path.stat().st_size
        except FileNotFoundError:continue
        relative=path.relative_to(root);group=relative.parts[0] if len(relative.parts)>1 else '(root files)'
        groups[group]+=size;count+=1
    disk=shutil.disk_usage(root)
    return {'project':str(root),'project_bytes':sum(groups.values()),'files':count,
        'free_bytes':disk.free,'disk_bytes':disk.total,'groups':dict(sorted(groups.items()))}


**Storage helper: `portable_resource_path`**

Keep resource choices local to this platform; Kaggle and laptop allocations are independent.


In [71]:
def portable_resource_path():
    """Keep resource choices local to this platform; Kaggle and laptop allocations are independent."""
    name='kaggle' if ENVIRONMENT['kaggle'] else 'local'
    return PROJECT_ROOT/('resource_limits_'+name+'.json')


**Storage helper: `portable_restore_resource_limits`**

Restore only an explicitly saved resource policy; retain original defaults on first use.


In [72]:
def portable_restore_resource_limits():
    """Restore only an explicitly saved resource policy; retain original defaults on first use."""
    path=portable_resource_path()
    if not path.exists():return {'status':'using notebook defaults'}
    saved=read_json(path)
    for name in ['output_budget_bytes','free_reserve_bytes']:
        value=saved[name]
        if isinstance(value,bool) or not isinstance(value,int) or value<0:raise ValueError('Invalid saved resource limit: '+name)
    if saved['free_reserve_bytes']<=0:raise ValueError('Keep a positive physical disk reserve.')
    DESKTOP_CONFIG.update({key:saved[key] for key in ['output_budget_bytes','free_reserve_bytes']})
    return {'status':'restored saved resource limits',**saved}


**Storage helper: `portable_save_resource_limits`**

Save user-chosen limits under the writer lock; never increase a limit automatically.


In [73]:
def portable_save_resource_limits(budget_gb,reserve_gb):
    """Save user-chosen limits under the writer lock; never increase a limit automatically."""
    values=[budget_gb,reserve_gb]
    if any(isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v) for v in values):
        raise ValueError('Enter finite numeric limits in decimal GB.')
    if budget_gb<0 or reserve_gb<=0:raise ValueError('Budget must be nonnegative and free-disk reserve positive.')
    saved={'output_budget_bytes':int(budget_gb*1e9),'free_reserve_bytes':int(reserve_gb*1e9)}
    if saved['free_reserve_bytes']<=0:raise ValueError('Physical reserve is too small.')
    handle=acquire_local_lock(PROJECT_ROOT/'desktop_writer.lock')
    try:
        snapshot=portable_storage_snapshot(PROJECT_ROOT)
        atomic_json(portable_resource_path(),saved);DESKTOP_CONFIG.update(saved)
        globals()['PORTABLE_RUN']=None;globals()['DESKTOP_RUN']=None
    finally:release_local_lock(handle)
    return {**saved,'project_bytes':snapshot['project_bytes'],'free_bytes':snapshot['free_bytes'],
        'note':'Limits saved. Rerun Module 1 and subsequent modules; saved conversion receipts remain unchanged.'}


**Storage helper: `portable_show_storage_controls`**

Display real project usage and allow a deliberate budget adjustment before resuming compute.


In [74]:
def portable_show_storage_controls():
    """Display real project usage and allow a deliberate budget adjustment before resuming compute."""
    import ipywidgets as widgets
    from IPython.display import display
    budget=widgets.FloatText(value=DESKTOP_CONFIG['output_budget_bytes']/1e9,description='Project limit GB:')
    reserve=widgets.FloatText(value=DESKTOP_CONFIG['free_reserve_bytes']/1e9,description='Keep free GB:')
    refresh=widgets.Button(description='Measure disk usage',icon='refresh')
    apply=widgets.Button(description='Save limits',button_style='info');output=widgets.Output()
    def show(button):
        refresh.disabled=apply.disabled=True
        with output:
            output.clear_output(wait=True)
            try:
                print('Measuring saved project files…',flush=True);snapshot=portable_storage_snapshot(PROJECT_ROOT)
                print(f'Project: {snapshot["project_bytes"]/1e9:.2f} GB | Free on drive: {snapshot["free_bytes"]/1e9:.2f} GB')
                print(f'Configured project limit: {budget.value:g} GB | Keep free: {reserve.value:g} GB')
                display(pd.DataFrame([{'folder':k,'GB':round(v/1e9,3)} for k,v in snapshot['groups'].items()]))
                print('Project limit counts all files under the project root, including raw data, both vector stores and bundles.')
                print('A limit of 0 disables only the project-size cap; the physical free-disk guard stays active.')
            except Exception as error:print(type(error).__name__+': '+str(error))
            finally:refresh.disabled=apply.disabled=False
    def save(button):
        with output:
            output.clear_output(wait=True)
            try:
                print(portable_save_resource_limits(budget.value,reserve.value)['note'])
                print('This changes no embeddings, indexes, migration batches or training checkpoints.')
            except Exception as error:print(type(error).__name__+': '+str(error))
    refresh.on_click(show);apply.on_click(save)
    display(widgets.VBox([widgets.HTML('<h3>Disk usage and resume budget</h3><p>Measure available space before increasing a limit. No files are deleted.</p>'),
        widgets.HBox([budget,reserve]),widgets.HBox([refresh,apply]),output]))
    return {'budget':budget,'reserve':reserve,'refresh':refresh,'apply':apply,'output':output}


In [133]:
# Restore only explicitly saved limits. This cell performs no corpus/model/index computation.
print(portable_restore_resource_limits())
if not ENVIRONMENT['batch'] and not DESKTOP_CONFIG.get('_headless'):
    STORAGE_CONTROLS=portable_show_storage_controls()
else:
    print('Project limit GB:',DESKTOP_CONFIG['output_budget_bytes']/1e9,
          '| Keep free GB:',DESKTOP_CONFIG['free_reserve_bytes']/1e9)


{'status': 'restored saved resource limits', 'output_budget_bytes': 100000000000, 'free_reserve_bytes': 2000000000}


**Display the workflow selector.** Choose your shard and workflow, then Proceed. In unattended Kaggle batch versions, the saved choice is used automatically so a headless run cannot wait forever for a browser click.


In [134]:
if not ENVIRONMENT['batch'] and not DESKTOP_CONFIG.get('_headless'):portable_show_selection()


**Execute Module 1.** Freeze this run’s selection/configuration and verify saved project identity. The table displays all 41 shard statuses. No embedding or training starts in this cell.


In [135]:
CHOICE=await portable_wait_selection()
INITIAL_STATUS=portable_initialize(CHOICE)
display(pd.DataFrame(INITIAL_STATUS))


Execution: cpu | Project: C:\Users\Anirban\OfflineWikipediaRAG_Desktop | Threads: 8
Workflow: compare | Selected shard: 4 | Existing indexed shards: [1, 2, 3, 4]
No existing passage embeddings will be recomputed.


,shard,download,embedding,indexing,training,evaluation,passages,training_note,model_version
0,1,complete,complete,complete,complete,complete,858306,,7b8e17b45a04df44efb67bf8
1,2,complete,complete,complete,skipped,complete,733455,Only 0 eligible packed new training QA; minimu...,
2,3,complete,complete,complete,complete,complete,683466,,c9b95b6765ffdc414618d6a6
3,4,complete,complete,complete,complete,complete,693214,,c8e0c0bedc865fe3845c0154
4,5,pending,pending,pending,pending,pending,0,,
5,6,pending,pending,pending,pending,pending,0,,
6,7,pending,pending,pending,pending,pending,0,,
7,8,pending,pending,pending,pending,pending,0,,
8,9,pending,pending,pending,pending,pending,0,,
9,10,pending,pending,pending,pending,pending,0,,


## Module 2 · Verify/download model assets and one selected source shard

Prepared files are verified and reused. A new Wikipedia Parquet file is downloaded **only** in Process mode when the selected
shard still needs embedding. Compare mode uses existing vectors; Validation/Export/Final-test modes skip acquisition entirely.
The optional reranker is downloaded only when explicitly enabled. All asset revisions are pinned before use.


**Helper: `prepare_desktop_model`**

Download a pinned safetensors model once and reuse verified local assets on later runs.

**Why here:** Acquisition precedes compute, but completed artifacts should only be verified, not downloaded again.


In [136]:
def prepare_desktop_model(role):
    """Download a pinned safetensors model once and reuse verified local assets on later runs."""
    folder=ROOT/'models'/role;receipt=folder/'desktop_model.json';record=LOCK['models'][role]
    if receipt.exists():
        saved=read_json(receipt)
        if saved['source']!=record:raise ValueError('Model revision differs from the project lock.')
        check_inventory(folder,saved['files'],True);return folder
    if CFG['offline']:raise FileNotFoundError('Missing model assets: '+role)
    folder.mkdir(parents=True,exist_ok=True)
    snapshot_download(record['repo'],revision=record['revision'],local_dir=str(folder),
        allow_patterns=['*.json','*.safetensors','*.model','*.txt','1_Pooling/*'],ignore_patterns=['*.bin','onnx/*','openvino/*'])
    if not (folder/'config.json').is_file() or not list(folder.rglob('*.safetensors')):
        raise ValueError('Model download is incomplete: '+role)
    atomic_json(receipt,{'source':record,'files':file_inventory(folder,{'desktop_model.json'})})
    return folder


**Helper: `download_desktop_shard`**

Download only the chosen Wikipedia Parquet file; existing verified files and Hub partial downloads are reused.

**Why here:** Acquisition precedes compute, but completed artifacts should only be verified, not downloaded again.


In [137]:
def download_desktop_shard(number):
    """Download only the chosen Wikipedia Parquet file; existing verified files and Hub partial downloads are reused."""
    entry=LOCK['files'][number-1]
    folder=ROOT/'raw';path=folder/entry['path'];path.parent.mkdir(parents=True,exist_ok=True)
    if path.is_file() and path.stat().st_size==entry['size'] and file_sha(path)==entry['sha256']:
        save_shard_record(number,download='complete');return path
    if CFG['offline']:raise FileNotFoundError('Selected shard is missing; prepare it online first.')
    guard_resources(entry['size']*2)
    print(f'Download/resume ONLY Wikipedia shard {number}/41: {entry["size"]/2**30:.2f} GiB',flush=True)
    save_shard_record(number,download='in progress')
    path=Path(hf_hub_download(LOCK['spec']['dataset'],entry['path'],repo_type='dataset',
        revision=LOCK['spec']['revision'],local_dir=str(folder),force_download=path.exists()))
    if path.stat().st_size!=entry['size'] or file_sha(path)!=entry['sha256']:raise ValueError('Downloaded shard checksum mismatch.')
    with pq.ParquetFile(path) as parquet:
        if not {'id','title','url','text'}.issubset(parquet.schema_arrow.names):raise ValueError('Unexpected Wikipedia schema.')
        save_shard_record(number,download='complete',source_rows=parquet.metadata.num_rows)
    return path


**Helper: `portable_prepare_reranker`**

Optional local reranker: pin its revision once and download only when explicitly enabled.

**Why here:** Acquisition precedes compute, but completed artifacts should only be verified, not downloaded again.


In [138]:
def portable_prepare_reranker():
    """Optional local reranker: pin its revision once and download only when explicitly enabled."""
    folder=ROOT/'extra_models'/'reranker';marker=folder/'source.json'
    if marker.exists():
        saved=read_json(marker);check_inventory(folder,saved['files'],True);return folder
    if CFG['offline']:raise FileNotFoundError('Optional reranker weights are not available offline.')
    repo='cross-encoder/ms-marco-MiniLM-L6-v2';revision=HfApi().model_info(repo).sha
    folder.mkdir(parents=True,exist_ok=True)
    snapshot_download(repo,revision=revision,local_dir=str(folder),allow_patterns=['*.json','*.txt','*.safetensors','*.model'])
    if not list(folder.glob('*.safetensors')):raise ValueError('Reranker safetensors were not supplied; no unsafe fallback is used.')
    atomic_json(marker,{'repo':repo,'revision':revision,'files':file_inventory(folder,{'source.json'})})
    return folder


**Helper: `portable_process_assets`**

Verify prepared model files; download source data only for an explicitly selected unfinished shard.

**Why here:** Acquisition precedes compute, but completed artifacts should only be verified, not downloaded again.


In [139]:
def portable_process_assets():
    """Verify prepared model files; download source data only for an explicitly selected unfinished shard."""
    for role in ['generator','embedder']:prepare_desktop_model(role)
    if CFG['enable_reranker']:portable_prepare_reranker()
    n=PORTABLE_RUN['number']
    if PORTABLE_RUN['mode']=='process' and shard_record(n).get('embedding')!='complete':download_desktop_shard(n)
    return {'status':'model assets verified; compatible downloads reused'}


**Execute Module 2.** Download/resume only missing assets needed for this workflow; download progress appears below.


In [140]:
ASSET_STATUS=portable_run_module('assets',portable_process_assets,modes=['process','compare'])
print(ASSET_STATUS)


assets · starting
assets · complete · 0.4 s
{'status': 'model assets verified; compatible downloads reused'}


## Module 3 · Resume full selected-shard embedding

Only Process mode executes this cell. Completed shards return immediately. For an unfinished shard, the article progress bar starts
at the committed cursor. The original 220/32 tokenizer-offset chunker and normalized 384-dimensional MiniLM vectors are unchanged.
An uncommitted transaction may be recomputed after interruption; completed compatible passages are not recomputed.


**Helper: `load_desktop_embedder`**

The embedding model stays frozen across generator updates so previous vectors remain compatible.

**Why here:** Corpus vectors are expensive; resume uses committed article transactions, and the model is released after the stage.


In [141]:
def load_desktop_embedder():
    """The embedding model stays frozen across generator updates so previous vectors remain compatible."""
    model=SentenceTransformer(str(ROOT/'models/embedder'),device=DEVICE,local_files_only=True)
    model.eval();model.requires_grad_(False)
    if model.get_sentence_embedding_dimension()!=384:raise ValueError('Expected 384-dimensional MiniLM embeddings.')
    return model


**Helper: `committed_article_cursor`**

Recover the true committed cursor from a streamed scalar column, without loading the vector database into RAM.

**Why here:** Corpus vectors are expensive; resume uses committed article transactions, and the model is released after the stage.


In [142]:
def committed_article_cursor(table):
    """Recover the true committed cursor from a streamed scalar column, without loading the vector database into RAM."""
    cursor=0
    for batch in table.search().select(['cursor_end']).limit(None).to_batches():
        if batch.num_rows:cursor=max(cursor,max(batch.column(0).to_pylist()))
    return cursor


**Helper: `embed_desktop_shard`**

Resume whole-article transactions; a stale JSON receipt after a Lance commit cannot duplicate rows.

**Why here:** Corpus vectors are expensive; resume uses committed article transactions, and the model is released after the stage.


In [143]:
def embed_desktop_shard(number,embedder):
    """Resume whole-article transactions; a stale JSON receipt after a Lance commit cannot duplicate rows."""
    record=shard_record(number)
    if record.get('project') not in {None,LOCK['identity']}:raise ValueError('Incompatible vector partition.')
    if record.get('embedding')=='complete':return record
    raw=download_desktop_shard(number);table=desktop_table(number,True);cursor=committed_article_cursor(table)
    saved_cursor=int(record.get('cursor',0))
    if saved_cursor>cursor and record.get('passages',0)!=table.count_rows():
        raise ValueError('Saved passage count differs from committed vectors; inspect the partition before resuming.')
    cursor=max(cursor,saved_cursor);offset=0
    with pq.ParquetFile(raw) as parquet:
        total=parquet.metadata.num_rows
        if cursor>total:raise ValueError('Stored cursor exceeds the source shard length.')
        bar=tqdm(total=total,initial=cursor,desc=f'Shard {number}/41 · articles',unit='article')
        try:
            for batch in parquet.iter_batches(batch_size=CFG['article_batch']):
                end=offset+batch.num_rows
                if end<=cursor:offset=end;continue
                payload=[]
                for j,item in enumerate(batch.to_pylist()):
                    if offset+j>=cursor:payload.extend(desktop_chunks(item,offset+j,number,embedder))
                guard_resources(sum(len(r['text'].encode())+2000 for r in payload)*2)
                commit_vectors(table,payload,end,embedder)
                record=save_shard_record(number,project=LOCK['identity'],embedding='complete' if end==total else 'in progress',
                    cursor=end,source_rows=total,passages=table.count_rows(),indexing='pending')
                bar.update(end-max(cursor,offset));bar.set_postfix(passages=record['passages']);cursor=end;offset=end
                guard_resources()
        finally:bar.close()
    return save_shard_record(number,project=LOCK['identity'],embedding='complete',cursor=total,source_rows=total,passages=table.count_rows())


**Helper: `portable_process_embedding`**

Resume only a selected unfinished shard, leaving all previously completed partitions untouched.

**Why here:** Corpus vectors are expensive; resume uses committed article transactions, and the model is released after the stage.


In [144]:
def portable_process_embedding():
    """Resume only a selected unfinished shard, leaving all previously completed partitions untouched."""
    n=PORTABLE_RUN['number'];record=shard_record(n)
    if record.get('embedding')=='complete':return {'status':'reused','shard':n,'passages':record['passages']}
    embedder=load_desktop_embedder()
    try:return embed_desktop_shard(n,embedder)
    finally:
        del embedder;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


**Execute Module 3.** Embed/resume only the selected unfinished shard, or explicitly skip this stage in other workflows.


In [145]:
EMBED_STATUS=portable_run_module('embedding',portable_process_embedding,requires=['assets'],modes=['process'])
print(EMBED_STATUS)


embedding · starting
embedding · complete · 0.0 s
{'status': 'skipped for workflow compare'}


## Module 4 · Verify/build LanceDB and create derived Chroma/BM25 indexes

Original LanceDB partitions remain intact. New selected-shard embeddings receive the existing IVF-PQ index (or exact search for tiny tables).
All complete source partitions are then streamed into independent derived indexes. **Copying saved vectors is not re-embedding.**

Chroma uses cosine HNSW. Every transferred batch is read back and compared for exact IDs/text/metadata and float32 vectors within
a strict absolute tolerance of **2e-6** (cosine indexing can renormalize them slightly) before
its receipt advances. Upsert makes a post-commit/pre-receipt interruption replay safely. A final count check guards duplicate IDs.
SQLite FTS5 stores BM25 text separately, with its scan cursor and inserts committed in the same transaction.
Both derived stages resume and report actual counters; the fresh child process closes database handles before later stages.


**Helper: `index_desktop_shard`**

Index only this completed embedding partition; rerunning an interrupted native index never re-embeds it.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [146]:
def index_desktop_shard(number):
    """Index only this completed embedding partition; rerunning an interrupted native index never re-embeds it."""
    record=shard_record(number);folder=ROOT/'parts'/f'shard_{number-1:03d}'
    if record.get('indexing')=='complete':
        check_inventory(folder,record['files'],CFG['verify_vector_hashes']);return record
    if record.get('embedding')!='complete':raise ValueError('Embedding must complete before indexing.')
    table=desktop_table(number);count=table.count_rows()
    if count==0:raise ValueError('This shard contains no searchable passages.')
    save_shard_record(number,indexing='in progress')
    print('Native index construction is on CPU; it may not expose a reliable completion percentage.',flush=True)
    native_desktop_stage(lambda:table.optimize(cleanup_older_than=timedelta(0),delete_unverified=True),'Compact committed vector fragments')
    if count>=256:
        from lancedb.index import IvfPq
        native_desktop_stage(lambda:table.create_index('vector',config=IvfPq(distance_type='cosine',
            num_partitions=max(1,min(256,int(math.sqrt(count/256))))),replace=True),'Build native IVF-PQ index')
        kind='IVF_PQ'
    else:kind='FLAT (exact scan for small partition)'
    files=file_inventory(folder,{'desktop_manifest.json'})
    return save_shard_record(number,indexing='complete',index_type=kind,passages=count,files=files)


**Helper: `native_desktop_stage`**

Show measured elapsed seconds for native operations without inventing a completion percentage or ETA.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [147]:
def native_desktop_stage(operation,label):
    """Show measured elapsed seconds for native operations without inventing a completion percentage or ETA."""
    stop=threading.Event()
    def watch():
        with tqdm(total=None,desc=label,unit='elapsed s') as bar:
            while not stop.wait(1):bar.update(1)
    monitor=threading.Thread(target=watch,daemon=True);monitor.start()
    try:return operation()
    finally:stop.set();monitor.join(timeout=2)


**Helper: `retrieve_desktop`**

Merge cosine-distance neighbours from all named shards; distance is not calibrated answer confidence.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [148]:
def retrieve_desktop(question,k,embedder,parts):
    """Merge cosine-distance neighbours from all named shards; distance is not calibrated answer confidence."""
    vector=embedder.encode(question,normalize_embeddings=True,convert_to_numpy=True).astype(np.float32)
    fields=['passage_id','article_id','title','url','text','source_shard','char_start','char_end','_distance']
    rows=[]
    for part in parts:
        query=part['table'].search(vector).distance_type('cosine').select(fields).limit(k)
        if part['record'].get('index_type')=='IVF_PQ':query=query.nprobes(CFG['nprobes']).refine_factor(4)
        rows.extend(query.to_list())
    return sorted(rows,key=lambda r:r['_distance'])[:k]


**Helper: `portable_hardware`**

Include the measurement environment in benchmark identities so laptop/Kaggle timings are not mixed.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [149]:
def portable_hardware():
    """Include the measurement environment in benchmark identities so laptop/Kaggle timings are not mixed."""
    import platform,psutil
    try:ram=psutil.virtual_memory().total
    except (OSError,psutil.Error):ram=None
    return {'system':platform.system(),'machine':platform.machine(),'processor':platform.processor(),
        'logical_cpus':os.cpu_count(),'ram_bytes':ram,'device':DEVICE,
        'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None,'threads':CFG['cpu_threads'],
        'versions':{n:metadata.version(n) for n in ['torch','transformers','sentence-transformers','lancedb','chromadb','numpy']}}


**Helper: `portable_lance_index_module`**

Time only a genuinely new native index build; reading an existing index is not a build-time benchmark.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [150]:
def portable_lance_index_module():
    """Time only a genuinely new native index build; reading an existing index is not a build-time benchmark."""
    number=PORTABLE_RUN['number'];existing=shard_record(number).get('indexing')=='complete';began=time.perf_counter()
    result=index_desktop_shard(number)
    if not existing:
        atomic_json(ROOT/'portable_diagnostics'/'lance_index'/f'shard_{number-1:03d}.json',
            {'seconds':time.perf_counter()-began,'hardware':portable_hardware(),'index_type':result['index_type']})
    return result


**Helper: `portable_batches`**

Stream bounded batches from a sealed source table; never read all vectors into a DataFrame.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [151]:
def portable_batches(part,batch_size=None):
    """Stream bounded batches from a sealed source table; never read all vectors into a DataFrame."""
    size=int(batch_size or CFG['migration_batch']);pending=[]
    if size<1:raise ValueError('Batch size must be positive.')
    fields=vector_schema().names
    for batch in part['table'].search().select(fields).limit(None).to_batches():
        for row in batch.to_pylist():
            pending.append(row)
            if len(pending)==size:yield pending;pending=[]
    if pending:yield pending


**Helper: `portable_vector_rows`**

Validate stored vectors and extract typed metadata without invoking any embedding model.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [152]:
def portable_vector_rows(rows):
    """Validate stored vectors and extract typed metadata without invoking any embedding model."""
    vectors=np.asarray([r['vector'] for r in rows],dtype=np.float32)
    if vectors.shape!=(len(rows),384) or not np.isfinite(vectors).all():raise ValueError('Invalid source vectors.')
    if not np.allclose(np.linalg.norm(vectors,axis=1),1.,atol=1e-4,rtol=0):raise ValueError('Expected normalized source embeddings.')
    ids=[str(r['passage_id']) for r in rows]
    if len(ids)!=len(set(ids)):raise ValueError('Duplicate passage IDs in source batch.')
    metadata=[{k:v for k,v in r.items() if k not in {'text','vector','passage_id'}} for r in rows]
    return ids,vectors,[r['text'] for r in rows],metadata


**Helper: `portable_chroma_client`**

Use an embedded persistent Chroma client with telemetry off; all query embeddings are supplied explicitly.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [153]:
def portable_chroma_client():
    """Use an embedded persistent Chroma client with telemetry off; all query embeddings are supplied explicitly."""
    import chromadb
    from chromadb.config import Settings
    return chromadb.PersistentClient(path=str(ROOT/'derived'/'chroma'),
        settings=Settings(anonymized_telemetry=False,allow_reset=False))


**Helper: `portable_chroma_collection`**

Use cosine HNSW with a source/configuration-specific name so experiments never overwrite existing data.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [154]:
def portable_chroma_collection(part,client,create=False):
    """Use cosine HNSW with a source/configuration-specific name so experiments never overwrite existing data."""
    key=portable_part_key(part)
    settings={'space':'cosine','ef_construction':128,'ef_search':128,'max_neighbors':16,
        'num_threads':int(CFG['cpu_threads'])}
    identity_settings={k:v for k,v in settings.items() if k not in {'num_threads','ef_search'}}
    name='wiki_'+str(part['number']).zfill(3)+'_'+digest([key,identity_settings])[:16]
    if create:
        return client.get_or_create_collection(name=name,embedding_function=None,
            metadata={'source_key':key,'protocol':'copy-f32-v1'},configuration={'hnsw':settings})
    collection=client.get_collection(name=name,embedding_function=None)
    if collection.metadata.get('source_key')!=key:raise ValueError('Chroma source identity mismatch.')
    return collection


**Helper: `portable_verify_chroma`**

Check every copied ID/text/metadata exactly and vectors within cosine-normalization float32 tolerance.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [155]:
def portable_verify_chroma(collection,rows):
    """Check every copied ID/text/metadata exactly and vectors within cosine-normalization float32 tolerance."""
    ids,vectors,texts,metadata=portable_vector_rows(rows)
    saved=collection.get(ids=ids,include=['embeddings','documents','metadatas'])
    positions={value:i for i,value in enumerate(saved['ids'])}
    if set(positions)!=set(ids):raise ValueError('Chroma copy omitted or duplicated passage IDs.')
    max_difference=0.
    for j,identity in enumerate(ids):
        i=positions[identity]
        if saved['documents'][i]!=texts[j] or saved['metadatas'][i]!=metadata[j]:raise ValueError('Copied text/metadata differs.')
        copied=np.asarray(saved['embeddings'][i],dtype=np.float32)
        max_difference=max(max_difference,float(np.max(np.abs(copied-vectors[j]))))
        if not np.allclose(copied,vectors[j],atol=2e-6,rtol=0):
            raise ValueError('Copied embedding differs from the original float32 vector.')
    return {'max_abs_difference':max_difference,'atol':2e-6,'note':'Chroma cosine indexes may renormalize float32 vectors.'}


**Helper: `portable_migrate_part`**

Idempotently upsert existing vectors; replay only an unsealed batch after an interruption.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [156]:
def portable_migrate_part(part,client):
    """Idempotently upsert existing vectors; replay only an unsealed batch after an interruption."""
    source=ROOT/'parts'/f'shard_{part["number"]-1:03d}'
    check_inventory(source,part['record']['files'],True)
    collection=portable_chroma_collection(part,client,create=True)
    path=ROOT/'derived'/'receipts'/(collection.name+'.json')
    spec={'source':portable_part_key(part),'collection':collection.name,'batch':CFG['migration_batch'],
        'protocol':'verified-f32-copy-v1','chroma_version':metadata.version('chromadb')}
    saved=read_json(path) if path.exists() else {'spec':spec,'batches':{},'status':'in progress','seconds':0.}
    if saved['spec']!=spec:raise ValueError('Migration geometry/version changed; restore the original settings.')
    if saved['status']=='complete':
        if collection.count()!=part['record']['passages']:raise ValueError('Completed Chroma collection count changed.')
        sample=part['table'].search().select(vector_schema().names).limit(3).to_list()
        if sample:portable_verify_chroma(collection,sample)
        return saved
    began=time.perf_counter();offset=0
    with tqdm(total=part['record']['passages'],desc=f'Copy/verify shard {part["number"]} → Chroma',unit='passage') as bar:
        for rows in portable_batches(part):
            ids,vectors,texts,metadatas=portable_vector_rows(rows);key=str(offset);end=offset+len(rows)
            fingerprint=digest([ids,hashlib.sha256(vectors.tobytes()).hexdigest(),texts,metadatas])
            if key in saved['batches']:
                if saved['batches'][key]!=fingerprint:raise ValueError('Source scan changed during a resumable migration.')
            else:
                guard_resources(sum(len(t.encode('utf-8'))+2000 for t in texts)*4)
                collection.upsert(ids=ids,embeddings=vectors,documents=texts,metadatas=metadatas)
                verified=portable_verify_chroma(collection,rows)
                if CFG.get('_test_migration_interrupt')==offset:raise BudgetPause('Test interruption after Chroma commit, before receipt.')
                saved['batches'][key]=fingerprint;saved['verified_rows']=end
                saved['vector_tolerance']=verified['atol'];saved['max_abs_vector_difference']=max(saved.get('max_abs_vector_difference',0.),verified['max_abs_difference'])
                saved['seconds']+=time.perf_counter()-began;began=time.perf_counter();atomic_json(path,saved)
            offset=end;bar.update(len(rows));portable_progress('Chroma shard '+str(part['number']),end,part['record']['passages']);guard_resources()
    if offset!=part['record']['passages'] or collection.count()!=offset:raise ValueError('Chroma final row count differs from source.')
    saved.update(status='complete',verified_rows=offset,verified_all_values=True)
    atomic_json(path,saved);return saved


**Helper: `portable_migrate_all`**

Run in a child process so Chroma closes cleanly before another process validates or packages its files.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [157]:
def portable_migrate_all():
    """Run in a child process so Chroma closes cleanly before another process validates or packages its files."""
    client=portable_chroma_client()
    return [portable_migrate_part(part,client) for part in portable_scope()]


**Helper: `portable_lexical_path`**

Put lexical indexes outside the original sealed partitions; source file inventories stay valid.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [158]:
def portable_lexical_path(part):
    """Put lexical indexes outside the original sealed partitions; source file inventories stay valid."""
    path=ROOT/'derived'/'lexical'/(portable_part_key(part)+'.sqlite3');path.parent.mkdir(parents=True,exist_ok=True)
    return path


**Helper: `portable_build_bm25`**

Persist an FTS5 BM25 index with its scan cursor in the same SQLite transaction; no new embeddings.

**Why here:** Derived indexes can be rebuilt from stored source vectors/text without changing corpus identity or fine-tuning.


In [159]:
def portable_build_bm25(part):
    """Persist an FTS5 BM25 index with its scan cursor in the same SQLite transaction; no new embeddings."""
    import sqlite3
    path=portable_lexical_path(part);key=portable_part_key(part)
    with sqlite3.connect(path) as db:
        db.execute('CREATE TABLE IF NOT EXISTS state (key TEXT PRIMARY KEY,value TEXT NOT NULL)')
        db.execute('CREATE VIRTUAL TABLE IF NOT EXISTS docs USING fts5(passage_id UNINDEXED,text,title,tokenize="porter unicode61")')
        values=dict(db.execute('SELECT key,value FROM state'))
        if values.get('source',key)!=key:raise ValueError('Lexical source identity changed.')
        cursor=int(values.get('cursor',0));offset=0
        if values.get('status')=='complete':
            if db.execute('SELECT count(*) FROM docs').fetchone()[0]!=part['record']['passages']:raise ValueError('Lexical count changed.')
            return {'shard':part['number'],'status':'reused','path':path.relative_to(ROOT).as_posix()}
        with tqdm(total=part['record']['passages'],initial=cursor,desc=f'Shard {part["number"]} · BM25',unit='passage') as bar:
            for rows in portable_batches(part):
                end=offset+len(rows)
                if end<=cursor:offset=end;continue
                pending=rows[max(0,cursor-offset):]
                with db:
                    db.executemany('INSERT INTO docs(passage_id,text,title) VALUES (?,?,?)',
                        [(r['passage_id'],r['text'],r['title']) for r in pending])
                    db.executemany('INSERT OR REPLACE INTO state VALUES (?,?)',[('source',key),('cursor',str(end))])
                bar.update(len(pending));offset=end;portable_progress('BM25 shard '+str(part['number']),end,part['record']['passages']);guard_resources()
        if offset!=part['record']['passages']:raise ValueError('Lexical source count differs.')
        db.execute('INSERT OR REPLACE INTO state VALUES (?,?)',('status','complete'))
    return {'shard':part['number'],'status':'complete','path':path.relative_to(ROOT).as_posix()}


**Execute Module 4a.** Reuse/verify the selected original Lance index. An interrupted native-index build can restart from saved embeddings.


In [160]:
LANCE_STATUS=portable_run_module('lance_index',portable_lance_index_module,
    requires=['embedding'],modes=['process'])
print(LANCE_STATUS)


lance_index · starting
lance_index · complete · 0.0 s
{'status': 'skipped for workflow compare'}


**Execute Module 4b.** In comparison/processing workflows, copy and verify every completed source partition into Chroma and build/reuse BM25. No encoder is invoked by this conversion.


In [161]:
DERIVED_STATUS=portable_run_module('derived_indexes',lambda:portable_spawn('derived_indexes'),
    requires=['lance_index'],modes=['process','compare'])
print(DERIVED_STATUS)


derived_indexes · starting


derived_indexes · worker elapsed: 0s [00:00, ?s/s]

derived_indexes · complete · 609.2 s
{'chroma': [{'spec': {'source': '88ef5a17350a4a1885f63763', 'collection': 'wiki_001_0355d7b1912e99c3', 'batch': 512, 'protocol': 'verified-f32-copy-v1', 'chroma_version': '1.5.9'}, 'batches': {'0': '978fb0a54cb63e45fd4f82c44cae9cae7c59098ee4e2a0a277246f1cc702809b', '512': '7ec193c4f54e1f34f4c1684e053929950dd2e49c6f273fe923203121bb54eccb', '1024': '171bc8db17aecc8d04b23adc70f15b618fc49029d018e22545abfd7290ae87ce', '1536': '3cb515d75bdcb1c146761f22dff8d4e08406178cc09517c45a65602a518d5398', '2048': 'dbfe36b8801d49e88aca05605903aac74d486b8b778d79829a8697ab33b36da4', '2560': 'c876285fb32b9bf26f9ddfc5ba8b5de46573e48a0476d0bc3241a10f286a9b48', '3072': '9eedee03c648ffc31134e517aa26c729ae7c12ba55d01a6fdb098048c3d19cbd', '3584': 'f40a2785c8cd2c146af5d1ed2a4b9bc1c76267a022b1d3211c549d4f6f34da19', '4096': '14e502f1d807ea26d838b04b656fa514c7c009b05732d8e7a28f73bd73f22bdf', '4608': 'b7012ab0c19af7afaf63a4ddc4fa1782d4c2041133f42e03ba3484d14009bef8', '5120': '79b16

## Module 5 · Reuse/create grounded QA and article-disjoint splits

The original eligibility protocol and per-shard receipts are retained. A reference alias must occur in stored evidence and context
overlap must pass the declared threshold. Repeated questions spanning split boundaries are excluded. Corpus text is fully indexed,
but only eligible real labelled QA is used for supervised training. Insufficient labels are reported, never fabricated.


**Helper: `prepare_labelled_qa`**

Download supervised SQuAD v2 once; Wikipedia text alone is not a question/answer training label.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [162]:
def prepare_labelled_qa():
    """Download supervised SQuAD v2 once; Wikipedia text alone is not a question/answer training label."""
    folder=ROOT/'qa_source';receipt=folder/'source.json'
    if receipt.exists():
        record=read_json(receipt)
        if record['source']!=LOCK['qa']:raise ValueError('QA source revision changed.')
        check_inventory(folder,record['files'],True)
        return folder
    if CFG['offline']:raise FileNotFoundError('Labelled QA data is not prepared locally.')
    from datasets import load_dataset
    data=load_dataset(LOCK['qa']['repo'],revision=LOCK['qa']['revision'],cache_dir=str(Path(CFG['cache_root'])/'qa'))
    folder.mkdir(parents=True,exist_ok=True)
    for split in ['train','validation']:
        with open(folder/(split+'.jsonl'),'w',encoding='utf-8') as handle:
            for row in data[split]:handle.write(json.dumps(dict(row),ensure_ascii=False)+'\n')
    atomic_json(receipt,{'source':LOCK['qa'],'files':file_inventory(folder,{'source.json'})})
    return folder


**Helper: `article_key`**

Normalize title punctuation/underscores consistently before global article-level split assignment.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [163]:
def article_key(title):
    """Normalize title punctuation/underscores consistently before global article-level split assignment."""
    return unicodedata.normalize('NFKC',urllib.parse.unquote(str(title))).replace('_',' ').strip().casefold()


**Helper: `qa_article_split`**

Assign a stable article-level split before seeing any model scores; future shards follow the same rule.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [164]:
def qa_article_split(title):
    """Assign a stable article-level split before seeing any model scores; future shards follow the same rule."""
    number=int(digest(['article-split-v1',article_key(title),CFG['seed']])[:8],16)%10
    return 'train' if number<8 else 'dev' if number==8 else 'reserved_test'


**Helper: `build_shard_qa`**

Match real labelled QA to this shard and require its reference alias to occur in a stored source passage.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [165]:
def build_shard_qa(number):
    """Match real labelled QA to this shard and require its reference alias to occur in a stored source passage."""
    folder=ROOT/'qa'/f'shard_{number-1:03d}';receipt=folder/'receipt.json'
    spec={'project':LOCK['identity'],'source':LOCK['qa'],'seed':CFG['seed'],'protocol':'matched-answerable-article-split-v1',
        'context_overlap':CFG['qa_context_overlap'],
        'code':digest(inspect.getsource(qa_article_split)+inspect.getsource(match_qa_row)+inspect.getsource(alias_in_text))}
    if receipt.exists():
        saved=read_json(receipt)
        if saved['spec']!=spec:raise ValueError('QA protocol changed; retain the old run or use a separate project.')
        check_inventory(folder,saved['files'],True);return saved
    source=prepare_labelled_qa();candidates=collections.defaultdict(list);audit=collections.Counter()
    for source_split in ['train','validation']:
        with open(source/(source_split+'.jsonl'),encoding='utf-8') as handle:
            for line in handle:
                row=json.loads(line)
                if not row['answers']['text']:audit['context_only_unanswerable_excluded']+=1;continue
                candidates[article_key(row['title'])].append(row)
    table=desktop_table(number);by_article=collections.defaultdict(list)
    for batch in tqdm(table.search().select(['article_id','title']).limit(None).to_batches(),desc='Match source article titles'):
        for row in batch.to_pylist():
            key=article_key(row['title'])
            if key in candidates:by_article[key].append(row['article_id'])
    rows=[];seen=set()
    for key,ids in tqdm(by_article.items(),desc='Verify labelled QA against stored evidence'):
        distinct=sorted(set(ids))
        if len(distinct)!=1:audit['ambiguous_article_title_excluded']+=len(candidates[key]);continue
        article_id=distinct[0];literal=article_id.replace("'","''")
        passages=table.search().where("article_id = '"+literal+"'").limit(None).to_list()
        for item in candidates[key]:
            row=match_qa_row(item,passages,number)
            fingerprint=digest(normalize_answer(item['question']))
            if row and fingerprint not in seen:rows.append(row);seen.add(fingerprint)
            else:audit['unmatched_or_duplicate_excluded']+=1
    folder.mkdir(parents=True,exist_ok=True);atomic_json(folder/'rows.json',rows)
    saved={'spec':spec,'counts':dict(collections.Counter(r['split'] for r in rows)),'audit':dict(audit),
        'files':file_inventory(folder,{'receipt.json'})};atomic_json(receipt,saved)
    save_shard_record(number,qa_counts=saved['counts']);return saved


**Helper: `match_qa_row`**

Use original stored Wikipedia text as context, excluding labels unsupported by the pinned shard snapshot.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [166]:
def match_qa_row(item,passages,number):
    """Use original stored Wikipedia text as context, excluding labels unsupported by the pinned shard snapshot."""
    aliases=[str(x) for x in item['answers']['text'] if normalize_answer(x)]
    original=set(normalize_answer(item.get('context','')).split())
    matches=[p for p in passages if any(alias_in_text(a,p['text']) for a in aliases)
        and len(original&set(normalize_answer(p['text']).split()))/max(1,len(original))>=CFG['qa_context_overlap']]
    if not matches:return None
    chosen=max(matches,key=lambda p:len(set(normalize_answer(item['question']).split())&set(normalize_answer(p['text']).split())))
    split=qa_article_split(item['title'])
    return {'id':str(item['id']),'question':str(item['question']),'answers':aliases,'title':chosen['title'],
        'article_key':article_key(chosen['title']),'article_id':chosen['article_id'],'passage_id':chosen['passage_id'],
        'context':chosen['text'],'url':chosen['url'],'shard':number,'split':split,
        'label_source':'SQuAD v2 alias verified in pinned Wikipedia passage; not independently human-reviewed'}


**Helper: `all_matched_qa`**

Read immutable per-shard QA and deduplicate globally so a question cannot appear across split boundaries.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [167]:
def all_matched_qa(numbers):
    """Read immutable per-shard QA and deduplicate globally so a question cannot appear across split boundaries."""
    rows=[];seen={};conflicts=set()
    for number in sorted(numbers):
        folder=ROOT/'qa'/f'shard_{number-1:03d}';receipt=read_json(folder/'receipt.json')
        check_inventory(folder,receipt['files'],True)
        for row in read_json(folder/'rows.json'):
            key=normalize_answer(row['question'])
            if key in seen:
                if seen[key]!=row['split']:conflicts.add(key)
                continue
            seen[key]=row['split'];rows.append(row)
    return [r for r in rows if normalize_answer(r['question']) not in conflicts]


**Helper: `alias_in_text`**

Match normalized aliases on word boundaries, excluding empty or accidental substring matches.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [168]:
def alias_in_text(alias,text):
    """Match normalized aliases on word boundaries, excluding empty or accidental substring matches."""
    value=normalize_answer(alias)
    return bool(value) and re.search(r'(?<!\w)'+re.escape(value)+r'(?!\w)',normalize_answer(text)) is not None


**Helper: `portable_question_bank`**

Interleave article-disjoint QA across all covered shards; report the query cap separately from corpus coverage.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [169]:
def portable_question_bank(numbers,split='dev',limit=None):
    """Interleave article-disjoint QA across all covered shards; report the query cap separately from corpus coverage."""
    rows=[r for r in all_matched_qa(numbers) if r['split']==split];groups={}
    for n in numbers:groups[n]=sorted([r for r in rows if r['shard']==n],key=lambda r:digest(r['id']))
    ordered=[];index=0;limit=int(limit or len(rows))
    while len(ordered)<limit:
        batch=[groups[n][index] for n in sorted(groups) if index<len(groups[n])]
        if not batch:break
        ordered.extend(batch[:limit-len(ordered)]);index+=1
    return ordered,{'available':len(rows),'used':len(ordered),'by_shard':dict(collections.Counter(r['shard'] for r in ordered)),
        'split':split,'article_disjoint':True,'automatic_label_matching':True}


**Helper: `portable_prepare_qa`**

Use cached shard-matched QA receipts and preserve the original article split and frozen training limits.

**Why here:** Grounded supervision and deterministic article splits are needed before training or choosing a retrieval configuration.


In [170]:
def portable_prepare_qa():
    """Use cached shard-matched QA receipts and preserve the original article split and frozen training limits."""
    parts=portable_scope();PORTABLE_RUN['parts']=parts
    return [{'shard':part['number'],**build_shard_qa(part['number'])['counts']} for part in parts]


**Execute Module 5.** Verify existing matched QA, prepare missing shard QA when allowed, and display available train/dev/reserved counts.


In [171]:
QA_STATUS=portable_run_module('qa',portable_prepare_qa,requires=['derived_indexes'],modes=['process','compare'])
display(pd.DataFrame(QA_STATUS if isinstance(QA_STATUS,list) else [QA_STATUS]))


qa · starting
qa · complete · 0.1 s


,shard,train,reserved_test,dev
0,1,4521.0,745.0,188.0
1,2,NaN,NaN,NaN
2,3,932.0,194.0,NaN
3,4,821.0,NaN,300.0


## Module 6 · Continue Flan-T5-small training only when needed

Completed training or an explicit label-insufficiency skip is preserved. A new stage warm-starts the saved training head,
using eligible new-shard training QA plus bounded training-only replay. The deployment winner and training head remain separate.
An interrupted stage restores model, optimizer, scheduler and RNG from a sealed checkpoint; changed data/settings/software are refused.
Compatible CPU/CUDA Torch build suffixes are accepted, but cross-device results are not promised to be bit-identical.
Checkpoints are saved every 25 optimizer steps or the configured elapsed interval at the next optimizer boundary.


**Helper: `load_generator`**

Load local saved safetensors only; no remote inference endpoint is used.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [172]:
def load_generator(path):
    """Load local saved safetensors only; no remote inference endpoint is used."""
    tokenizer=AutoTokenizer.from_pretrained(str(path),local_files_only=True)
    model=AutoModelForSeq2SeqLM.from_pretrained(str(path),local_files_only=True,
        use_safetensors=True,attn_implementation='eager').to(DEVICE)
    return tokenizer,model


**Helper: `training_rows`**

Exclude over-budget or clipped-away labels explicitly rather than silently truncating supervised targets.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [173]:
def training_rows(rows,tokenizer):
    """Exclude over-budget or clipped-away labels explicitly rather than silently truncating supervised targets."""
    kept=[];excluded=collections.Counter()
    for row in rows:
        evidence=[{'title':row['title'],'text':row['context']}]
        selected,prompt=pack_desktop_evidence(row['question'],evidence,tokenizer)
        target=next((a for a in row['answers'] if any(alias_in_text(a,s['text']) for s in selected)),None)
        if target is None:excluded['answer_outside_packed_evidence']+=1;continue
        labels=tokenizer(text_target=target,truncation=False)['input_ids']
        if len(labels)>CFG['max_new_tokens']:excluded['target_over_budget']+=1;continue
        kept.append({**tokenizer(prompt,truncation=False),'labels':labels})
    return kept,dict(excluded)


**Helper: `base_reader`**

Fingerprint the actual downloaded pretrained generator for comparison and initial warm start.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [174]:
def base_reader():
    """Fingerprint the actual downloaded pretrained generator for comparison and initial warm start."""
    path=ROOT/'models/generator';record=read_json(path/'desktop_model.json')
    check_inventory(path,record['files'],True)
    return {'id':'base-'+digest(record)[:16],'path':str(path),'manifest':record}


**Helper: `latest_training_reader`**

The training head is distinct from the deployment winner; incremental training continues its recorded lineage.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [175]:
def latest_training_reader():
    """The training head is distinct from the deployment winner; incremental training continues its recorded lineage."""
    p=ROOT/'training_head.json'
    if not p.exists():return base_reader(),[]
    saved=read_json(p);reader=verify_saved_reader(ROOT/saved['path'])
    if reader['id']!=saved['id']:raise ValueError('Training head changed or is corrupt.')
    return reader,saved['trained_shards']


**Helper: `promote_training_lineage`**

Recover a completed stage whose head-pointer write was interrupted, without rolling back later training.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [176]:
def promote_training_lineage(reader):
    """Recover a completed stage whose head-pointer write was interrupted, without rolling back later training."""
    manifest=reader['manifest'];p=ROOT/'training_head.json'
    current,shards=latest_training_reader()
    if current['id']==reader['id'] or manifest['spec']['shard'] in shards:return
    if current['id']!=manifest['parent']:raise ValueError('Training lineage diverged; do not replace an unrelated later head.')
    atomic_json(p,{'path':Path(reader['path']).relative_to(ROOT).as_posix(),'id':reader['id'],
        'trained_shards':manifest['trained_shards']})


**Helper: `incremental_training_spec`**

Freeze a stage's data, warm-start parent, software and optimizer geometry before taking any optimizer steps.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [177]:
def incremental_training_spec(number,parent,new,replay):
    """Freeze a stage's data, warm-start parent, software and optimizer geometry before taking any optimizer steps."""
    return {'protocol':'desktop-incremental-v1','project':LOCK['identity'],'shard':number,'parent':parent['id'],
        'data':digest(new+replay),'epochs':CFG['train_epochs'],'lr':CFG['train_lr'],'batch':CFG['train_batch'],
        'accum':CFG['train_accum'],'max_steps':CFG['train_max_steps'],'input_cap':CFG['input_cap'],
        'output_cap':CFG['max_new_tokens'],'seed':CFG['seed'],
        'software':{x:metadata.version(x) for x in ['torch','transformers','accelerate','datasets']},
        'code':digest(inspect.getsource(training_rows)+inspect.getsource(qa_article_split))}


**Helper: `checked_incremental_checkpoint`**

Resume only a full model/optimizer/scheduler/RNG checkpoint that was sealed after Trainer finished saving.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [178]:
def checked_incremental_checkpoint(folder):
    """Resume only a full model/optimizer/scheduler/RNG checkpoint that was sealed after Trainer finished saving."""
    matches=[]
    for p in Path(folder).glob('checkpoint-*'):
        marker=p/'desktop_checkpoint.json'
        if not marker.exists():continue
        record=read_json(marker);check_inventory(p,record['files'],True)
        state=read_json(p/'trainer_state.json')
        if state['global_step']!=record['step']:raise ValueError('Checkpoint step differs from its sealed receipt.')
        for name in ['optimizer.pt','scheduler.pt','rng_state.pth','model.safetensors']:
            if not (p/name).is_file():raise ValueError('Incomplete optimizer checkpoint: '+name)
        matches.append((record['step'],p))
    return max(matches,key=lambda x:x[0]) if matches else (0,None)


**Helper: `incremental_callbacks`**

Publish checkpoint completion after saving, and request extra checkpoints after a measured time interval.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [179]:
def incremental_callbacks(number,folder,resume_step):
    """Publish checkpoint completion after saving, and request extra checkpoints after a measured time interval."""
    from transformers import TrainerCallback
    class CheckpointReceipt(TrainerCallback):
        last_save=time.perf_counter()
        def on_train_begin(self,args,state,control,**kwargs):
            if resume_step and state.global_step!=resume_step:raise ValueError('Trainer did not restore the expected optimizer step.')
        def on_log(self,args,state,control,logs=None,**kwargs):
            atomic_json(Path(folder)/'training_telemetry.json',report_json_safe(state.log_history))
        def on_step_end(self,args,state,control,**kwargs):
            if time.perf_counter()-self.last_save>CFG['checkpoint_seconds']:control.should_save=True
            if CFG['time_budget_hours'] and time.perf_counter()-STARTED>CFG['time_budget_hours']*3600:
                control.should_save=True;control.should_training_stop=True
            return control
        def on_save(self,args,state,control,**kwargs):
            p=Path(folder)/f'checkpoint-{state.global_step}'
            for name in ['optimizer.pt','scheduler.pt','rng_state.pth','model.safetensors']:
                if not (p/name).is_file():raise ValueError('Trainer omitted a required full-checkpoint file: '+name)
            atomic_json(p/'desktop_checkpoint.json',{'step':state.global_step,'files':file_inventory(p,{'desktop_checkpoint.json'})})
            save_shard_record(number,training='in progress',training_step=state.global_step,epoch=float(state.epoch or 0),last_checkpoint=p.relative_to(ROOT).as_posix())
            self.last_save=time.perf_counter()
            if CFG.get('_test_interrupt_step')==state.global_step:raise BudgetPause('Explicit test interruption after verified checkpoint.')
    return [CheckpointReceipt()]


**Helper: `execute_incremental_training`**

A new shard warm-starts weights; an interrupted stage restores its own optimizer/scheduler/RNG state.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [180]:
def execute_incremental_training(number,parent,rows,development,folder):
    """A new shard warm-starts weights; an interrupted stage restores its own optimizer/scheduler/RNG state."""
    from datasets import Dataset
    from transformers import Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    tokenizer,model=load_generator(parent['path'])
    train,audit=training_rows(rows,tokenizer);dev,dev_audit=training_rows(development,tokenizer)
    atomic_json(Path(folder)/'data_audit.json',{'eligible_train':len(rows),'used_train':len(train),'excluded_train':audit,
        'used_dev':len(dev),'excluded_dev':dev_audit,'full_Wikipedia_shard_processed':True})
    if len(train)<CFG['min_new_train']:raise ValueError('Too few supervised rows remain after token-budget validation.')
    restored,checkpoint=checked_incremental_checkpoint(folder)
    args=Seq2SeqTrainingArguments(output_dir=str(folder),num_train_epochs=CFG['train_epochs'],learning_rate=CFG['train_lr'],
        per_device_train_batch_size=CFG['train_batch'],per_device_eval_batch_size=CFG['train_batch'],
        gradient_accumulation_steps=CFG['train_accum'],save_strategy='steps',save_steps=CFG['checkpoint_steps'],
        save_total_limit=2,logging_steps=1,eval_strategy='epoch' if dev else 'no',report_to='none',
        optim='adamw_torch',fp16=False,bf16=False,use_cpu=DEVICE=='cpu',dataloader_num_workers=0,
        dataloader_pin_memory=DEVICE!='cpu',seed=CFG['seed'],data_seed=CFG['seed'],max_steps=CFG['train_max_steps'],
        save_only_model=False,load_best_model_at_end=False)
    trainer=Seq2SeqTrainer(model=model,args=args,train_dataset=Dataset.from_list(train),
        eval_dataset=Dataset.from_list(dev) if dev else None,processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer,model=model),callbacks=incremental_callbacks(number,folder,restored))
    try:
        trainer.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)
        if CFG['train_max_steps']<0 and float(trainer.state.epoch or 0)+1e-5<CFG['train_epochs']:
            raise BudgetPause('Training paused at a saved optimizer boundary before completing the configured epochs.')
        trained=Path(folder)/'model';trainer.save_model(str(trained));tokenizer.save_pretrained(trained)
        atomic_json(Path(folder)/'training_history.json',trainer.state.log_history)
        return trained,trainer.state.global_step,float(trainer.state.epoch or 0)
    finally:
        del trainer,model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


**Helper: `plan_incremental_training`**

Select only new-shard training QA plus a bounded deterministic replay of earlier training QA.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [181]:
def plan_incremental_training(number,available):
    """Select only new-shard training QA plus a bounded deterministic replay of earlier training QA."""
    parent,trained_shards=latest_training_reader();all_rows=all_matched_qa(available)
    new=[r for r in all_rows if r['shard']==number and r['split']=='train']
    old=[r for r in all_rows if r['shard'] in trained_shards and r['shard']!=number and r['split']=='train']
    count=min(len(old),CFG['replay_max'],max(1,int(len(new)*CFG['replay_fraction']))) if new else 0
    replay=sorted(old,key=lambda r:digest(['replay',r['id'],CFG['seed']]))[:count]
    dev=[r for r in all_rows if r['split']=='dev'];dev=sorted(dev,key=lambda r:digest(r['id']))[:CFG['dev_questions']]
    return parent,trained_shards,new,replay,dev


**Helper: `train_desktop_shard`**

Reuse a completed shard's saved model, or create exactly one new incremental training stage for that shard.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [182]:
def train_desktop_shard(number,available):
    """Reuse a completed shard's saved model, or create exactly one new incremental training stage for that shard."""
    record=shard_record(number)
    if record.get('training')=='complete':
        reader=verify_saved_reader(ROOT/record['trained_model']);promote_training_lineage(reader);return reader
    parent,trained_shards,new,replay,dev=plan_incremental_training(number,available)
    if record.get('training')=='skipped':return parent
    spec=incremental_training_spec(number,parent,new,replay);stage_id=digest(spec)[:24]
    folder=ROOT/'training'/f'shard_{number-1:03d}'/stage_id;folder.mkdir(parents=True,exist_ok=True)
    old_spec=record.get('training_spec')
    if old_spec and old_spec!=spec:raise ValueError('Unfinished training data/parent/settings changed. Restore its original lineage/settings; do not silently restart it.')
    tokenizer=AutoTokenizer.from_pretrained(parent['path'],local_files_only=True)
    usable,_=training_rows(new,tokenizer)
    if len(usable)<CFG['min_new_train']:
        save_shard_record(number,training='skipped',training_reason=f'Only {len(usable)} eligible packed new training QA; minimum {CFG["min_new_train"]}.',training_spec=spec)
        return parent
    save_shard_record(number,training='in progress',training_spec=spec,training_new_qa=len(new),training_replay_qa=len(replay))
    model,steps,epoch=execute_incremental_training(number,parent,new+replay,dev,folder)
    manifest={'id':stage_id,'project':LOCK['identity'],'spec':spec,'parent':parent['id'],
        'steps':steps,'epoch':epoch,'trained_shards':sorted(set(trained_shards+[number])),
        'files':file_inventory(model,{'reader_manifest.json'})}
    atomic_json(model/'reader_manifest.json',manifest)
    relative=model.relative_to(ROOT).as_posix()
    save_shard_record(number,training='complete',trained_model=relative,model_version=stage_id,training_step=steps,epoch=epoch)
    reader=verify_saved_reader(model);promote_training_lineage(reader);return reader


**Helper: `generate_desktop`**

Use deterministic local generation with explicit input/output budgets and measured token counts.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [183]:
def generate_desktop(prompt,tokenizer,model):
    """Use deterministic local generation with explicit input/output budgets and measured token counts."""
    encoded=tokenizer(prompt,return_tensors='pt',truncation=False)
    if encoded.input_ids.shape[1]>CFG['input_cap']:raise ValueError('Generator prompt exceeds its declared input budget.')
    encoded={k:v.to(DEVICE) for k,v in encoded.items() if k in {'input_ids','attention_mask'}}
    if DEVICE!='cpu':torch.cuda.synchronize()
    started=time.perf_counter()
    with torch.inference_mode():output=model.generate(**encoded,max_new_tokens=CFG['max_new_tokens'],do_sample=False,num_beams=1)
    if DEVICE!='cpu':torch.cuda.synchronize()
    return {'answer':tokenizer.decode(output[0],skip_special_tokens=True).strip(),
        'input_tokens':int(encoded['input_ids'].shape[1]),'output_tokens':int(output.shape[1]-1),
        'generation_ms':(time.perf_counter()-started)*1000}


**Helper: `portable_train_selected`**

Keep completed training; on an interrupted stage accept compatible Torch CPU/CUDA wheel suffixes only.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [184]:
def portable_train_selected(number,available):
    """Keep completed training; on an interrupted stage accept compatible Torch CPU/CUDA wheel suffixes only."""
    record=shard_record(number)
    if record.get('training') in {'complete','skipped'}:
        print('Training already finished/skipped for this shard; retaining the current training head.')
        return latest_training_reader()[0]
    old=record.get('training_spec')
    if old:
        parent,_,new,replay,dev=plan_incremental_training(number,available)
        current=incremental_training_spec(number,parent,new,replay)
        compatible=lambda version:version.split('+',1)[0]
        a=copy.deepcopy(old);b=copy.deepcopy(current)
        for spec in [a,b]:spec['software']['torch']=compatible(spec['software']['torch'])
        if a!=b:raise ValueError('Unfinished optimizer stage has incompatible settings/data/software. Validate saved outputs or restore its original environment.')
        # Retain the exact old folder identity and metadata; no optimizer-state relabeling is performed.
        folder=ROOT/'training'/f'shard_{number-1:03d}'/digest(old)[:24]
        model,steps,epoch=execute_incremental_training(number,parent,new+replay,dev,folder)
        manifest={'id':digest(old)[:24],'project':LOCK['identity'],'spec':old,'parent':parent['id'],
            'steps':steps,'epoch':epoch,'trained_shards':sorted(set(parent.get('manifest',{}).get('trained_shards',[])+[number])),
            'files':file_inventory(model,{'reader_manifest.json'})}
        atomic_json(model/'reader_manifest.json',manifest)
        save_shard_record(number,training='complete',trained_model=model.relative_to(ROOT).as_posix(),
            model_version=manifest['id'],training_step=steps,epoch=epoch)
        reader=verify_saved_reader(model);promote_training_lineage(reader);return reader
    return train_desktop_shard(number,available)


**Helper: `portable_readers`**

Compare the pretrained reader, current training head and its immediate saved parent; never roll back the lineage.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [185]:
def portable_readers():
    """Compare the pretrained reader, current training head and its immediate saved parent; never roll back the lineage."""
    updated,_=latest_training_reader();previous=base_reader()
    parent=updated.get('manifest',{}).get('parent')
    if parent and not parent.startswith('base-'):
        for manifest in (ROOT/'training').glob('shard_*/*/model/reader_manifest.json'):
            if read_json(manifest).get('id')==parent:previous=verify_saved_reader(manifest.parent);break
    return reader_candidates(updated,previous)


**Helper: `portable_train_module`**

Continue one shard's recorded training lineage; completed shards are never automatically retrained.

**Why here:** Only Process mode may train; comparison/validation can load existing readers without spending optimizer steps.


In [186]:
def portable_train_module():
    """Continue one shard's recorded training lineage; completed shards are never automatically retrained."""
    available=[p['number'] for p in portable_scope()]
    reader=portable_train_selected(PORTABLE_RUN['number'],available)
    return {'status':shard_record(PORTABLE_RUN['number'])['training'],'model_version':reader['id']}


**Execute Module 6.** Resume/create the selected shard’s training stage, or reuse completed training. Compare/Validation modes skip training entirely.


In [187]:
TRAINING_STATUS=portable_run_module('training',portable_train_module,requires=['qa'],modes=['process'])
print(TRAINING_STATUS)


training · starting
training · complete · 0.0 s
{'status': 'skipped for workflow compare'}


## Module 7 · Measure retrieval, then compare saved readers and fusion

Dense, BM25 and RRF hybrid retrieval use the same source scope, query vectors and declared candidate budget.
Each strategy runs in its own process for sampled RSS and latency measurements. Lookup latency includes source-row fetches;
query encoding is recorded separately. All displayed `k` rows share a max-k lookup measurement, so they are not independent per-k timing experiments.

The exact dense reference covers up to 8 questions; ID ties can affect ANN agreement. Article hit/MRR and alias diagnostics
are matched-reference proxies. Development selection chooses a retrieval method, then the reader/k/fusion grid is run on up to 24 questions.
Every generated answer is persisted so interruptions resume at the next uncached answer. Earlier-shard results are exported for forgetting checks.


**Helper: `em_f1`**

Take the best exact match/token F1 over valid answer aliases, using multiset token overlap.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [188]:
def em_f1(answer, references):
    """Take the best exact match/token F1 over valid answer aliases, using multiset token overlap."""
    predicted, exact, best = normalize_answer(answer), 0.0, 0.0
    for reference in references:
        expected = normalize_answer(reference)
        exact = max(exact, float(predicted == expected))
        left, right = predicted.split(), expected.split()
        common = sum((collections.Counter(left) & collections.Counter(right)).values())
        if not left or not right:
            f1 = float(left == right)
        else:
            f1 = 2 * common / (len(left) + len(right))
        best = max(best, f1)
    return exact, best


**Helper: `matrix_plot`**

Display and save a labeled diagnostic matrix; unavailable human or negative-class measurements remain N/A rather than fabricated zeros.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [189]:
def matrix_plot(table, name, title, percent=True, annotations=True):
    """Display and save a labeled diagnostic matrix; unavailable human or negative-class measurements remain N/A rather than fabricated zeros."""
    table.to_csv(RUN_FOLDER/(name+'.csv'))
    fig,ax=plt.subplots(figsize=(max(6,len(table.columns)*.75),min(15,max(3.5,len(table)*.28))))
    if table.empty:
        ax.text(.5,.5,'Pending: no eligible measured observations',ha='center',va='center')
        ax.axis('off')
    else:
        values=table.to_numpy(dtype=float)
        color=plt.get_cmap('viridis').copy();color.set_bad('#e5e7eb')
        picture=ax.imshow(np.ma.masked_invalid(values),aspect='auto',cmap=color,vmin=0,vmax=100 if percent else None)
        ax.set_xticks(range(len(table.columns)),[str(x) for x in table.columns],rotation=45,ha='right')
        ax.set_yticks(range(len(table)),[' / '.join(map(str,x)) if isinstance(x,tuple) else str(x) for x in table.index])
        if annotations and values.size<=240:
            for i in range(len(table)):
                for j in range(len(table.columns)):
                    value=values[i,j]
                    ax.text(j,i,'N/A' if not np.isfinite(value) else f'{value:.1f}' if percent else f'{value:.0f}',
                            ha='center',va='center',fontsize=8)
        fig.colorbar(picture,ax=ax,label='Percent' if percent else 'Count / value')
    ax.set_title(title);fig.tight_layout()
    fig.savefig(RUN_FOLDER/(name+'.png'),dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)


**Helper: `read_evaluation_cache`**

Recover only a torn final JSONL record; malformed earlier records or duplicate prediction IDs are errors.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [190]:
def read_evaluation_cache(path):
    """Recover only a torn final JSONL record; malformed earlier records or duplicate prediction IDs are errors."""
    path=Path(path)
    if not path.exists():return {}
    data=path.read_bytes();lines=data.splitlines(keepends=True);cache={};valid=0
    for index,line in enumerate(lines):
        try:row=json.loads(line)
        except (json.JSONDecodeError,UnicodeDecodeError):
            if index!=len(lines)-1:raise ValueError('Corrupt evaluation cache before its final record.')
            path.write_bytes(data[:valid]);break
        if row['key'] in cache:raise ValueError('Duplicate evaluation cache record.')
        cache[row['key']]=row;valid+=len(line)
    else:
        if data and not data.endswith(b'\n'):
            with open(path,'ab') as f:f.write(b'\n')
    return cache


**Helper: `reader_candidates`**

Compare distinct saved weights, avoiding duplicate readers when tuning was skipped or no previous model exists.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [191]:
def reader_candidates(updated,previous):
    """Compare distinct saved weights, avoiding duplicate readers when tuning was skipped or no previous model exists."""
    readers={};seen=set()
    for label,reader in [('pretrained',base_reader()),('previous',previous),('updated',updated)]:
        if reader['id'] in seen:continue
        seen.add(reader['id']);readers[label]=reader
    return readers


**Helper: `score_desktop_prediction`**

Reference overlap and article-ID retrieval diagnostics are measured; factual hallucination remains unmeasured.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [192]:
def score_desktop_prediction(question,output,label,k,method):
    """Reference overlap and article-ID retrieval diagnostics are measured; factual hallucination remains unmeasured."""
    exact,f1=em_f1(output['answer'],question['answers']);hits=output.get('retrieved',[])
    ranks=[i+1 for i,h in enumerate(hits) if h['article_id']==question['article_id'] and h['source_shard']==question['shard']]
    evidence=' '.join(s['text'] for s in output.get('evidence',[]))
    return {'question_id':question['id'],'question_shard':question['shard'],'reader':label,'k':k,'method':method,
        'EM':exact,'F1':f1,'article_hit':float(bool(ranks)),'article_MRR':1/min(ranks) if ranks else 0.,
        'alias_in_evidence':float(any(normalize_answer(a) in normalize_answer(evidence) for a in question['answers'])),
        'abstained':float(output.get('abstained',False)),'total_ms':output.get('total_ms',0.),
        'generation_ms':output.get('generation_ms',0.),'model_calls':output.get('model_calls',0),
        'unsupported_sentence_proxy':output.get('unsupported_sentence_proxy'),'hallucination_index':None}


**Helper: `export_desktop_matrices`**

Reuse the earlier notebook's matrix renderer and save numeric CSVs plus portable PNG figures.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [193]:
def export_desktop_matrices(folder,records):
    """Reuse the earlier notebook's matrix renderer and save numeric CSVs plus portable PNG figures."""
    global RUN_FOLDER
    RUN_FOLDER=Path(folder);scores=pd.DataFrame([r['score'] for r in records]);files=[]
    if scores.empty:return [],[]
    scores.to_csv(RUN_FOLDER/'per_question_scores.csv',index=False)
    columns=['EM','F1','article_hit','article_MRR','alias_in_evidence','abstained','total_ms','model_calls']
    summary=scores.groupby(['reader','method','k'],as_index=False)[columns].mean()
    summary.to_csv(RUN_FOLDER/'comparison_summary.csv',index=False)
    for metric in ['F1','EM','article_hit','article_MRR','alias_in_evidence','total_ms']:
        table=summary.pivot(index=['reader','method'],columns='k',values=metric)
        percent=metric!='total_ms';table=table*100 if percent else table
        matrix_plot(table,'matrix_'+metric,metric+' · DEVELOPMENT PREVIEW',percent=percent)
        files.append('matrix_'+metric+'.png')
    return summary.to_dict(orient='records'),files


**Helper: `select_development_reader`**

Select solely on development F1, with deterministic smaller-k/concat tie breaks; no test-driven selection.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [194]:
def select_development_reader(summary,readers,question_count):
    """Select solely on development F1, with deterministic smaller-k/concat tie breaks; no test-driven selection."""
    if not summary:
        label=next(reversed(readers))
        return {'reader':label,'reader_id':readers[label]['id'],'k':3,'method':'concat',
            'reason':'No matched development QA; fallback configuration is explicitly unvalidated.'}
    best=sorted(summary,key=lambda r:(-r['F1'],r['k'],r['method']!='concat',r['reader']))[0]
    return {**{k:best[k] for k in ['reader','k','method']},'reader_id':readers[best['reader']]['id'],
        'reason':f'Selected on {question_count} development questions; not a final held-out result.'}


**Helper: `portable_fetch_passages`**

Look up original source rows, retaining full stored vectors for consistent distance calculations.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [195]:
def portable_fetch_passages(part,ids):
    """Look up original source rows, retaining full stored vectors for consistent distance calculations."""
    if not ids:return []
    literals=','.join("'"+str(x).replace("'","''")+"'" for x in ids)
    rows=part['table'].search().where('passage_id IN ('+literals+')').limit(None).to_list()
    mapping={r['passage_id']:r for r in rows}
    if len(mapping)!=len(set(ids)):raise ValueError('Retrieved passage is missing from the source partition.')
    return [mapping[x] for x in ids]


**Helper: `portable_bm25`**

Search each shard by BM25; fuse per-shard ranks rather than compare incompatible BM25 magnitudes.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [196]:
def portable_bm25(question,parts,k):
    """Search each shard by BM25; fuse per-shard ranks rather than compare incompatible BM25 magnitudes."""
    import sqlite3
    tokens=list(dict.fromkeys(re.findall(r'[^\W_]+',question.casefold())))[:32]
    if not tokens:return []
    expression=' OR '.join('"'+x+'"' for x in tokens);lists=[]
    for part in parts:
        path=portable_lexical_path(part)
        if not path.exists():raise FileNotFoundError('Build the BM25 sidecar first: '+str(path))
        with sqlite3.connect(path.as_uri()+'?mode=ro',uri=True) as db:
            ids=[r[0] for r in db.execute('SELECT passage_id FROM docs WHERE docs MATCH ? ORDER BY bm25(docs) LIMIT ?',
                (expression,int(k)))]
        lists.append(portable_fetch_passages(part,ids))
    return portable_rrf(lists,k)


**Helper: `portable_rrf`**

Combine rank positions with RRF; its score is neither a cosine distance nor a confidence probability.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [197]:
def portable_rrf(rankings,k):
    """Combine rank positions with RRF; its score is neither a cosine distance nor a confidence probability."""
    scores=collections.defaultdict(float);rows={}
    for ranking in rankings:
        for rank,row in enumerate(ranking,1):
            identity=row['passage_id'];scores[identity]+=1/(60+rank);rows[identity]=row
    ordered=sorted(scores,key=lambda x:(-scores[x],x))[:int(k)]
    return [{**rows[x],'_rank_score':scores[x]} for x in ordered]


**Helper: `portable_dense`**

Query identical vectors with Lance IVF-PQ, Chroma cosine HNSW, or an exact Lance reference scan.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [198]:
def portable_dense(vector,parts,k,backend,client=None,exact=False):
    """Query identical vectors with Lance IVF-PQ, Chroma cosine HNSW, or an exact Lance reference scan."""
    hits=[]
    for part in parts:
        if backend=='chroma':
            collection=portable_chroma_collection(part,client)
            result=collection.query(query_embeddings=[vector.tolist()],n_results=min(k,collection.count()),
                include=['distances','documents','metadatas','embeddings'])
            ids=result['ids'][0]
            rows=[{**meta,'passage_id':identity,'text':text,'vector':embedding} for identity,text,meta,embedding in
                zip(ids,result['documents'][0],result['metadatas'][0],result['embeddings'][0])]
            for row,distance in zip(rows,result['distances'][0]):hits.append({**row,'_distance':float(distance)})
        else:
            query=part['table'].search(vector).distance_type('cosine').limit(k)
            if exact:query=query.bypass_vector_index()
            elif part['record'].get('index_type')=='IVF_PQ':query=query.nprobes(CFG['nprobes']).refine_factor(4)
            hits.extend(query.to_list())
    return sorted(hits,key=lambda r:(r['_distance'],r['passage_id']))[:k]


**Helper: `portable_retrieve`**

Use one frozen query encoder for all strategies; gold answers never influence ranking.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [199]:
def portable_retrieve(question,k,embedder,parts,strategy,client=None,reranker=None,vector=None):
    """Use one frozen query encoder for all strategies; gold answers never influence ranking."""
    vector=np.asarray(vector if vector is not None else embedder.encode(question,normalize_embeddings=True,
        convert_to_numpy=True),dtype=np.float32)
    candidates=max(int(k),CFG['retrieval_candidates'])
    if strategy=='bm25':hits=portable_bm25(question,parts,candidates)
    elif strategy in {'lance_dense','chroma_dense','exact'}:
        backend='chroma' if strategy=='chroma_dense' else 'lance'
        hits=portable_dense(vector,parts,candidates,backend,client,exact=strategy=='exact')
    elif strategy in {'lance_hybrid','chroma_hybrid','hybrid_rerank'}:
        backend='chroma' if strategy!='lance_hybrid' else 'lance'
        hits=portable_rrf([portable_dense(vector,parts,candidates,backend,client),
            portable_bm25(question,parts,candidates)],candidates)
        if strategy=='hybrid_rerank':
            if reranker is None:raise ValueError('The optional local reranker has not been loaded.')
            scores=reranker.predict([(question,h['text']) for h in hits],show_progress_bar=False)
            hits=[h for _,h in sorted(zip(map(float,scores),hits),key=lambda pair:-pair[0])]
    else:raise ValueError('Unknown retrieval strategy: '+str(strategy))
    result=[]
    for hit in hits[:int(k)]:
        value=np.asarray(hit['vector'],dtype=np.float32);den=float(np.linalg.norm(vector)*np.linalg.norm(value))
        distance=1-float(np.dot(vector,value))/den if den else 2.
        result.append({**{a:b for a,b in hit.items() if a!='vector'},'_distance':distance,'retrieval_strategy':strategy})
    return result


**Helper: `portable_answer`**

Keep Flan-T5-small and the original evidence-budget policy; swap only the retrieval adapter.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [200]:
def portable_answer(question,runtime,k=None,method=None,strategy=None):
    """Keep Flan-T5-small and the original evidence-budget policy; swap only the retrieval adapter."""
    selected=runtime.release['selected'];k=int(k or selected['k']);method=method or selected['method']
    strategy=strategy or runtime.release.get('retrieval_strategy','lance_dense');started=time.perf_counter()
    hits=portable_retrieve(question,k,runtime.embedder,runtime.parts,strategy,
        getattr(runtime,'chroma',None),getattr(runtime,'reranker',None))
    retrieval_ms=(time.perf_counter()-started)*1000
    output=answer_desktop(question,k,method,runtime,hits=hits)
    output.update(retrieval_strategy=strategy,retrieval_ms=retrieval_ms,total_ms=(time.perf_counter()-started)*1000)
    return output


**Helper: `portable_load_reranker`**

Load a prepared optional cross-encoder without contacting a remote inference service.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [201]:
def portable_load_reranker():
    """Load a prepared optional cross-encoder without contacting a remote inference service."""
    from sentence_transformers import CrossEncoder
    folder=ROOT/'extra_models'/'reranker';saved=read_json(folder/'source.json')
    check_inventory(folder,saved['files'],True)
    return CrossEncoder(str(folder),device=DEVICE,local_files_only=True,max_length=512)


**Helper: `portable_rss_sampler`**

Sample process RSS in an isolated worker; this is observed process memory, not an allocated index-size estimate.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [202]:
def portable_rss_sampler():
    """Sample process RSS in an isolated worker; this is observed process memory, not an allocated index-size estimate."""
    import psutil
    try:process=psutil.Process();start=process.memory_info().rss
    except (OSError,psutil.Error):process=None;start=None
    state={'start':start,'peak':start};stop=threading.Event()
    def sample():
        while not stop.wait(.02):
            if process is not None:
                try:state['peak']=max(state['peak'],process.memory_info().rss)
                except (OSError,psutil.Error):pass
    thread=threading.Thread(target=sample,daemon=True);thread.start()
    return state,stop,thread


**Helper: `portable_query_reference`**

Compute query vectors once and exact-neighbour references for a declared subset of questions.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [203]:
def portable_query_reference(questions):
    """Compute query vectors once and exact-neighbour references for a declared subset of questions."""
    parts=portable_scope();embedder=load_desktop_embedder();rows=[]
    for index,q in enumerate(questions):
        began=time.perf_counter();vector=embedder.encode(q['question'],normalize_embeddings=True,convert_to_numpy=True).astype(np.float32)
        encode_ms=(time.perf_counter()-began)*1000;exact=None
        if index<CFG['exact_reference_queries']:
            exact=[r['passage_id'] for r in portable_dense(vector,parts,max(CFG['k_values']),'lance',exact=True)]
        rows.append({'question':q,'vector':vector.tolist(),'query_encode_ms':encode_ms,'exact_ids':exact})
        portable_progress('Query vectors / exact reference',index+1,len(questions));guard_resources()
    return rows


**Helper: `portable_retrieval_worker`**

Measure one strategy per fresh process, using identical query vectors and source scope.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [204]:
def portable_retrieval_worker(payload):
    """Measure one strategy per fresh process, using identical query vectors and source scope."""
    strategy=payload['strategy'];rows=payload['references'];parts=portable_scope()
    state,stop,thread=portable_rss_sampler();client=None;reranker=None;results=[]
    try:
        if strategy.startswith('chroma') or strategy=='hybrid_rerank':client=portable_chroma_client()
        if strategy=='hybrid_rerank':reranker=portable_load_reranker()
        if rows:
            first=rows[0];portable_retrieve(first['question']['question'],max(CFG['k_values']),None,parts,strategy,
                client,reranker,np.asarray(first['vector'],np.float32))
        for index,record in enumerate(rows):
            question=record['question'];times=[];hits=[]
            for repeat in range(CFG['query_repeats']):
                began=time.perf_counter();hits=portable_retrieve(question['question'],max(CFG['k_values']),None,
                    parts,strategy,client,reranker,np.asarray(record['vector'],np.float32))
                times.append((time.perf_counter()-began)*1000)
            for k in CFG['k_values']:
                top=hits[:k];ranks=[i+1 for i,h in enumerate(top) if h['article_id']==question['article_id'] and h['source_shard']==question['shard']]
                exact=record['exact_ids'];agreement=None
                if exact is not None and strategy in {'lance_dense','chroma_dense'}:
                    agreement=len({h['passage_id'] for h in top}&set(exact[:k]))/max(1,len(exact[:k]))
                results.append({'strategy':strategy,'question_id':question['id'],'shard':question['shard'],'k':k,
                    'article_hit':float(bool(ranks)),'article_MRR':1/min(ranks) if ranks else 0.,
                    'gold_passage_hit':float(any(h['passage_id']==question['passage_id'] for h in top)),
                    'alias_in_retrieved':float(any(alias_in_text(a,h['text']) for a in question['answers'] for h in top)),
                    'ANN_recall_at_k':agreement,'lookup_ms':float(np.median(times)),
                    'lookup_repeats_ms':times,'query_encode_ms':record['query_encode_ms']})
            portable_progress(strategy,index+1,len(rows));guard_resources()
    finally:stop.set();thread.join(timeout=2)
    return {'strategy':strategy,'rows':results,'process_rss_start_bytes':state['start'],'process_peak_rss_bytes':state['peak'],
        'sampled_rss_increase_bytes':max(0,state['peak']-state['start']) if state['peak'] is not None else None,'hardware':portable_hardware(),
        'timing_note':'Warm lookup fetches max(k) candidates; k rows share that measured lookup. Query encoding is reported separately. Exact ties may affect ANN ID agreement.'}


**Helper: `portable_benchmark_spec`**

Version benchmark results by semantic inputs, software, code and hardware; keep old reports untouched.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [205]:
def portable_benchmark_spec(parts,questions):
    """Version benchmark results by semantic inputs, software, code and hardware; keep old reports untouched."""
    return {'protocol':'portable-retrieval-v1','project':LOCK['identity'],'corpus':corpus_snapshot(parts),
        'question_ids':[q['id'] for q in questions],'hardware':portable_hardware(),
        'reranker':read_json(ROOT/'extra_models/reranker/source.json') if CFG['enable_reranker'] else None,
        'settings':{k:CFG[k] for k in ['benchmark_dev_questions','generation_dev_questions','retrieval_candidates',
            'query_repeats','k_values','nprobes','exact_reference_queries','compare_chroma','enable_reranker',
            'max_cosine_distance','require_answer_occurrence','input_cap','max_new_tokens']},
        'code':digest(inspect.getsource(portable_retrieve)+inspect.getsource(portable_bm25)+inspect.getsource(portable_dense)+
            inspect.getsource(portable_generation_worker)+inspect.getsource(answer_desktop))}


**Helper: `portable_index_costs`**

Report available build/ingest observations; historical Lance build time stays missing rather than being reconstructed.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [206]:
def portable_index_costs(parts):
    """Report available build/ingest observations; historical Lance build time stays missing rather than being reconstructed."""
    rows=[]
    for part in parts:
        key=portable_part_key(part);lance_path=ROOT/'parts'/f'shard_{part["number"]-1:03d}'/'lance'
        receipts=[]
        for path in (ROOT/'derived'/'receipts').glob('wiki_'+str(part['number']).zfill(3)+'_*.json'):
            receipt=read_json(path)
            if receipt['spec']['source']==key:receipts.append(receipt)
        native=ROOT/'portable_diagnostics'/'lance_index'/f'shard_{part["number"]-1:03d}.json'
        native=read_json(native) if native.exists() else {}
        rows.append({'shard':part['number'],'passages':part['record']['passages'],
            'lance_partition_bytes':sum(p.stat().st_size for p in lance_path.rglob('*') if p.is_file()),
            'lance_new_index_build_seconds':native.get('seconds'),
            'chroma_copy_verify_ingest_seconds':max((r.get('seconds',0.) for r in receipts),default=None),
            'note':'Chroma duration includes source scan, insertion/HNSW work and verification. Historical Lance timing is unavailable; these are not identical build workloads.'})
    return {'rows':rows,'chroma_shared_store_bytes':sum(p.stat().st_size for p in (ROOT/'derived'/'chroma').rglob('*') if p.is_file()),
        'memory_note':'Peak memory is sampled in isolated query workers; unavailable resource APIs yield null, never zero.'}


**Helper: `portable_retrieval_matrices`**

Export the measured strategy-versus-k heatmaps; missing ANN measurements remain unavailable.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [207]:
def portable_retrieval_matrices(folder,summary):
    """Export the measured strategy-versus-k heatmaps; missing ANN measurements remain unavailable."""
    global RUN_FOLDER
    if not summary:return []
    RUN_FOLDER=Path(folder);frame=pd.DataFrame(summary);figures=[]
    for metric in ['article_hit','article_MRR','gold_passage_hit','ANN_recall_at_k']:
        if not frame[metric].notna().any():continue
        table=frame.pivot(index='strategy',columns='k',values=metric)*100
        name='matrix_retrieval_'+metric;matrix_plot(table,name,metric+' · DEVELOPMENT RETRIEVAL',percent=True)
        figures.append(name+'.png')
    return figures


**Helper: `portable_retrieval_suite`**

Compare dense, lexical and hybrid methods without multiplying the generator grid by every strategy.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [208]:
def portable_retrieval_suite(parts):
    """Compare dense, lexical and hybrid methods without multiplying the generator grid by every strategy."""
    questions,coverage=portable_question_bank([p['number'] for p in parts],limit=CFG['benchmark_dev_questions'])
    spec=portable_benchmark_spec(parts,questions);folder=ROOT/'portable_runs'/digest(spec)[:24];folder.mkdir(parents=True,exist_ok=True)
    atomic_json(folder/'protocol.json',spec);path=folder/'query_reference.json'
    if not path.exists():atomic_json(path,portable_spawn('query_reference',{'questions':questions}))
    references=read_json(path);strategies=['lance_dense','bm25','lance_hybrid']
    if CFG['compare_chroma']:strategies+=['chroma_dense','chroma_hybrid']
    if CFG['enable_reranker']:strategies+=['hybrid_rerank']
    results=[];errors=[]
    for strategy in strategies:
        target=folder/('retrieval_'+strategy+'.json')
        try:
            if not target.exists():atomic_json(target,portable_spawn('retrieval',{'strategy':strategy,'references':references}))
            results.append(read_json(target))
        except Exception as error:
            record_desktop_error('retrieval benchmark '+strategy,error);errors.append({'strategy':strategy,'error':str(error)})
    rows=[r for result in results for r in result['rows']];summary=[]
    for strategy in strategies:
        result=next((r for r in results if r['strategy']==strategy),None)
        for k in CFG['k_values']:
            group=[r for r in rows if r['strategy']==strategy and r['k']==k]
            if not group:continue
            item={'strategy':strategy,'k':k,'questions':len(group)}
            for name in ['article_hit','article_MRR','gold_passage_hit','alias_in_retrieved','ANN_recall_at_k']:
                values=[r[name] for r in group if r[name] is not None];item[name]=float(np.mean(values)) if values else None
            latencies=[x for r in group for x in r['lookup_repeats_ms']]
            item.update(lookup_p50_ms=float(np.percentile(latencies,50)),lookup_p95_ms=float(np.percentile(latencies,95)),
                sampled_process_peak_RSS_MB=result['process_peak_rss_bytes']/1e6 if result['process_peak_rss_bytes'] is not None else None,
                sampled_process_RSS_increase_MB=result['sampled_rss_increase_bytes']/1e6 if result['sampled_rss_increase_bytes'] is not None else None)
            summary.append(item)
    candidates=[r for r in summary if r['k']==CFG['retrieval_select_k']]
    if questions and not candidates:raise RuntimeError('Every retrieval benchmark failed. Inspect saved worker logs; no winner is published.')
    chosen=sorted(candidates,key=lambda r:(-r['article_hit'],-r['article_MRR'],r['lookup_p50_ms'],r['strategy']))[0]['strategy'] if candidates else 'lance_dense'
    report={'spec':spec,'coverage':coverage,'summary':summary,'selected_strategy':chosen,'errors':errors,
        'selection':'Development article hit rate, then MRR, then warm p50 lookup at k='+str(CFG['retrieval_select_k']),
        'no_questions':not bool(questions),'measurement_note':'Separate strategy processes; sampled total process RSS includes runtime/source fetch overhead. Index build timings are separate ingest measurements.'}
    report['index_costs']=portable_index_costs(parts)
    report['figures']=portable_retrieval_matrices(folder,summary)
    atomic_json(folder/'retrieval_report.json',report);write_report_csv(folder/'retrieval_comparison.csv',summary)
    return report,folder,questions


**Helper: `portable_generation_worker`**

Run the required reader × k × fusion ablation on one selected retrieval strategy; checkpoint every answer.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [209]:
def portable_generation_worker(payload):
    """Run the required reader × k × fusion ablation on one selected retrieval strategy; checkpoint every answer."""
    folder=ROOT/payload['folder'];parts=portable_scope();questions=payload['questions'];strategy=payload['strategy']
    readers=payload['readers'];cache=read_evaluation_cache(folder/'predictions.jsonl');records=[]
    embedder=load_desktop_embedder();client=portable_chroma_client() if strategy.startswith('chroma') or strategy=='hybrid_rerank' else None
    reranker=portable_load_reranker() if strategy=='hybrid_rerank' else None
    total=len(readers)*len(questions)*len(CFG['k_values'])*2;done=0
    for label,reader in readers.items():
        tokenizer,model=load_generator(ROOT/reader['relative_path']);model.eval();model.requires_grad_(False)
        runtime=types.SimpleNamespace(tokenizer=tokenizer,model=model,embedder=embedder,parts=parts,chroma=client,reranker=reranker,
            release={'selected':{'k':3,'method':'concat'},'retrieval_strategy':strategy})
        for question in questions:
            for method in ['concat','generate_merge']:
                for k in CFG['k_values']:
                    key=digest([question['id'],reader['id'],method,k,strategy])
                    if key not in cache:
                        output=portable_answer(question['question'],runtime,k,method,strategy)
                        score=score_desktop_prediction(question,output,label,k,method)
                        row={'key':key,'score':score,'output':output,'question':question}
                        append_report_jsonl(folder/'predictions.jsonl',row);cache[key]=row
                    records.append(cache[key]);done+=1;portable_progress(label+' · saved/generated answers',done,total);guard_resources()
        del runtime,model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    summary,figures=export_desktop_matrices(folder,records)
    per_shard=[]
    for label in readers:
        for shard in sorted({q['shard'] for q in questions}):
            group=[r['score'] for r in records if r['score']['reader']==label and r['score']['question_shard']==shard]
            if group:per_shard.append({'reader':label,'shard':shard,'answer_count':len(group),
                'EM':float(np.mean([x['EM'] for x in group])),'F1':float(np.mean([x['F1'] for x in group]))})
    write_report_csv(folder/'earlier_shard_regression.csv',per_shard)
    return {'summary':summary,'figures':figures,'per_shard':per_shard,'questions':len(questions),
        'selected':select_development_reader(summary,readers,len(questions)),'retrieval_strategy':strategy}


**Helper: `portable_assess`**

Create a new assessment namespace; old 12-question reports and original deployment pointers are preserved.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [210]:
def portable_assess(parts):
    """Create a new assessment namespace; old 12-question reports and original deployment pointers are preserved."""
    retrieval,_,questions=portable_retrieval_suite(parts);readers=portable_readers()
    questions=questions[:CFG['generation_dev_questions']]
    portable_readers_spec={label:{'id':r['id'],'relative_path':Path(r['path']).relative_to(ROOT).as_posix()} for label,r in readers.items()}
    spec={'protocol':'portable-answer-v1','retrieval':retrieval['spec'],'strategy':retrieval['selected_strategy'],
        'readers':portable_readers_spec,'questions':digest(questions),'generation_limit':CFG['generation_dev_questions'],
        'code':digest(inspect.getsource(portable_answer)+inspect.getsource(finish_grounded_output)+inspect.getsource(generate_merge_desktop))}
    identity=digest(spec)[:24];folder=ROOT/'portable_assessments'/identity;folder.mkdir(parents=True,exist_ok=True)
    marker=folder/'assessment.json'
    if marker.exists():
        saved=read_json(marker);check_inventory(folder,saved['files'],True);return saved,folder,readers
    atomic_json(folder/'protocol.json',spec)
    generation=portable_spawn('generation',{'folder':folder.relative_to(ROOT).as_posix(),'questions':questions,
        'strategy':retrieval['selected_strategy'],'readers':portable_readers_spec})
    retrieval_folder=ROOT/'portable_runs'/digest(retrieval['spec'])[:24]
    for name in retrieval.get('figures',[]):
        shutil.copy2(retrieval_folder/name,folder/name)
        csv=(retrieval_folder/name).with_suffix('.csv')
        if csv.exists():shutil.copy2(csv,folder/csv.name)
    report={'run_id':identity,'spec':spec,'evaluation_scope':'DEVELOPMENT PREVIEW — final test remains reserved',
        'coverage':[p['number'] for p in parts],'expected_shards':len(LOCK['files']),
        'question_count':len(questions),'qa_counts':dict(collections.Counter(q['shard'] for q in questions)),
        'selected':generation['selected'],'summary':generation['summary'],'figures':generation['figures']+retrieval.get('figures',[]),
        'retrieval_strategy':retrieval['selected_strategy'],'retrieval_comparison':retrieval,'per_shard':generation['per_shard'],
        'quality_approved':False,'hallucination_index':None,
        'limitations':['Automatically matched references require human review; model pretraining contamination cannot be excluded.',
            'Retrieval selection and generator selection both use development data; these are not final-test results.',
            'Query counts are declared compute limits, never corpus/article/passage limits.',
            'Lexical answer-occurrence and unsupported-sentence diagnostics do not prove factual correctness.',
            'No independently adjudicated factual hallucination rate is available.']}
    write_report_csv(folder/'retrieval_comparison.csv',retrieval['summary'])
    atomic_json(folder/'retrieval_comparison.json',retrieval)
    report['files']=file_inventory(folder,{'assessment.json'});atomic_json(marker,report)
    return report,folder,readers


**Helper: `portable_assessment_module`**

Retain model lineage and score caches while recording a new, broader comparison report.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [211]:
def portable_assessment_module():
    """Retain model lineage and score caches while recording a new, broader comparison report."""
    report,folder,readers=portable_assess(portable_scope())
    PORTABLE_RUN.update(report=report,folder=folder,readers=readers)
    return {'release_candidate':report['run_id'],'retrieval_strategy':report['retrieval_strategy'],
        'answer_questions':report['question_count'],'retrieval_questions':report['retrieval_comparison']['coverage']['used']}


**Helper: `write_report_csv`**

Preserve numeric fractions; nested diagnostics remain JSON strings in CSV and structured data in JSON.

**Why here:** Measurements must use compatible data and explicit score definitions before a configuration is selected.


In [212]:
def write_report_csv(path,rows,columns=None):
    """Preserve numeric fractions; nested diagnostics remain JSON strings in CSV and structured data in JSON."""
    flattened=[{k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in row.items()} for row in rows]
    flattened=[{k:"'"+v if isinstance(v,str) and v.startswith(('=','+','-','@')) else v for k,v in row.items()} for row in flattened]
    pd.DataFrame(flattened,columns=columns if not rows else None).to_csv(path,index=False)


**Execute Module 7.** Run/reuse the staged development comparison. This may take time; worker counters and per-answer caches show/resume real progress.


In [213]:
COMPARISON_STATUS=portable_run_module('comparison',portable_assessment_module,requires=['training'],modes=['process','compare'])
print(COMPARISON_STATUS)
if 'report' in PORTABLE_RUN:
    display(pd.DataFrame(PORTABLE_RUN['report']['retrieval_comparison']['summary']))
    display(pd.DataFrame(PORTABLE_RUN['report']['summary']))


comparison · starting


query_reference · worker elapsed: 0s [00:00, ?s/s]

retrieval · worker elapsed: 0s [00:00, ?s/s]

retrieval · worker elapsed: 0s [00:00, ?s/s]

retrieval · worker elapsed: 0s [00:00, ?s/s]

retrieval · worker elapsed: 0s [00:00, ?s/s]

... (2 outputs left out)

generation · worker elapsed: 0s [00:00, ?s/s]

comparison · complete · 6277.5 s
{'release_candidate': '622880b84231acbd2e5a8691', 'retrieval_strategy': 'chroma_hybrid', 'answer_questions': 24, 'retrieval_questions': 40}


,strategy,k,questions,article_hit,article_MRR,gold_passage_hit,alias_in_retrieved,ANN_recall_at_k,lookup_p50_ms,lookup_p95_ms,sampled_process_peak_RSS_MB,sampled_process_RSS_increase_MB
0,lance_dense,1,40,0.300,0.300000,0.175,0.200,1.0000,107.37540,166.174855,743.526400,207.298560
1,lance_dense,3,40,0.475,0.375000,0.325,0.375,0.8750,107.37540,166.174855,743.526400,207.298560
2,lance_dense,5,40,0.475,0.375000,0.400,0.400,0.8250,107.37540,166.174855,743.526400,207.298560
3,lance_dense,10,40,0.500,0.378571,0.450,0.450,0.8125,107.37540,166.174855,743.526400,207.298560
4,bm25,1,40,0.350,0.350000,0.300,0.325,NaN,5840.40630,8951.268215,743.407616,207.749120
5,bm25,3,40,0.350,0.350000,0.300,0.325,NaN,5840.40630,8951.268215,743.407616,207.749120
6,bm25,5,40,0.600,0.412500,0.500,0.550,NaN,5840.40630,8951.268215,743.407616,207.749120
7,bm25,10,40,0.650,0.418403,0.600,0.675,NaN,5840.40630,8951.268215,743.407616,207.749120
8,lance_hybrid,1,40,0.375,0.375000,0.250,0.300,NaN,5736.22675,8243.855700,819.113984,282.447872
9,lance_hybrid,3,40,0.625,0.491667,0.500,0.525,NaN,5736.22675,8243.855700,819.113984,282.447872


,reader,method,k,EM,F1,article_hit,article_MRR,alias_in_evidence,abstained,total_ms,model_calls
0,pretrained,concat,1,0.208333,0.229167,0.333333,0.333333,0.291667,0.166667,8609.898417,0.958333
1,pretrained,concat,3,0.250000,0.287500,0.541667,0.430556,0.416667,0.083333,7201.789563,1.000000
2,pretrained,concat,5,0.250000,0.287500,0.625000,0.449306,0.416667,0.083333,6891.372233,1.000000
3,pretrained,concat,10,0.250000,0.287500,0.625000,0.449306,0.416667,0.083333,6522.557658,1.000000
4,pretrained,generate_merge,1,0.000000,0.000000,0.333333,0.333333,0.291667,0.083333,6269.377638,1.916667
5,pretrained,generate_merge,3,0.000000,0.000000,0.541667,0.430556,0.333333,0.041667,6733.502679,3.083333
6,pretrained,generate_merge,5,0.000000,0.000000,0.625000,0.449306,0.333333,0.041667,6706.395325,3.125000
7,pretrained,generate_merge,10,0.000000,0.000000,0.625000,0.449306,0.333333,0.041667,6800.351892,3.125000
8,previous,concat,1,0.166667,0.244048,0.333333,0.333333,0.291667,0.125000,6630.590321,0.958333
9,previous,concat,3,0.250000,0.344048,0.541667,0.430556,0.416667,0.000000,6085.498925,1.000000


## Module 8 · Save/reload the selected deployment

Save actual selected Flan-T5-small weights, frozen encoder identity, retrieval strategy, report checksums and exact vector scope.
The new `portable_active_release.json` pointer leaves the previous desktop deployment/report history intact.
Validation/Export/Final-test workflows load an existing release. Runtime paths are resolved relative to the current project root.


**Helper: `portable_save_release`**

Save actual Flan-T5-small weights and bind the chosen retrieval adapter to the exact vector snapshot.

**Why here:** A visitor must use the saved selected model and its recorded index scope, rather than whichever in-memory model was last trained.


In [214]:
def portable_save_release(report,folder,readers,parts):
    """Save actual Flan-T5-small weights and bind the chosen retrieval adapter to the exact vector snapshot."""
    reader=readers[report['selected']['reader']];identity=report['run_id']
    target=ROOT/'final_models'/identity/'generator';marker=target/'saved_model.json'
    if not marker.exists():
        target.mkdir(parents=True,exist_ok=True)
        for source in Path(reader['path']).rglob('*'):
            if not source.is_file() or '.cache' in source.parts:continue
            dest=target/source.relative_to(reader['path'])
            if dest.exists() and file_sha(dest)==file_sha(source):continue
            guard_resources(source.stat().st_size);dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,dest)
        atomic_json(marker,{'reader_id':reader['id'],'files':file_inventory(target,{'saved_model.json'})})
    check_inventory(target,read_json(marker)['files'],True)
    config=read_json(target/'config.json')
    if config.get('model_type')!='t5':raise ValueError('The generator must remain a T5 model from the pinned Flan-T5-small lineage.')
    release={'id':identity,'protocol':'portable-release-v1','project':LOCK['identity'],'selected':report['selected'],
        'retrieval_strategy':report['retrieval_strategy'],'generator':target.relative_to(ROOT).as_posix(),
        'model_manifest_sha256':file_sha(marker),'embedder':'models/embedder','report':folder.relative_to(ROOT).as_posix(),
        'report_sha256':file_sha(folder/'assessment.json'),'corpus':corpus_snapshot(parts),
        'configuration':{k:CFG[k] for k in ['input_cap','max_new_tokens','max_question_chars','max_cosine_distance',
            'require_answer_occurrence','nprobes','k_values','retrieval_candidates','cpu_threads']}}
    if report['retrieval_strategy']=='hybrid_rerank':
        release.update(reranker='extra_models/reranker',reranker_manifest_sha256=file_sha(ROOT/'extra_models/reranker/source.json'))
    atomic_json(ROOT/'portable_releases'/(identity+'.json'),release)
    atomic_json(ROOT/'portable_active_release.json',release)
    return release


**Helper: `portable_load_release`**

Load only the saved deployed reader/encoder/index scope, supporting both legacy and new release formats.

**Why here:** A visitor must use the saved selected model and its recorded index scope, rather than whichever in-memory model was last trained.


In [215]:
def portable_load_release():
    """Load only the saved deployed reader/encoder/index scope, supporting both legacy and new release formats."""
    release=portable_release_pointer();parts,report=portable_validate_release(release)
    # Hardware threads stay local; saved inference semantics and candidate count stay frozen.
    CFG.update({k:v for k,v in release['configuration'].items() if k!='cpu_threads'})
    tokenizer,model=load_generator(ROOT/release['generator']);model.eval();model.requires_grad_(False)
    strategy=release.get('retrieval_strategy','lance_dense');client=None;reranker=None
    if strategy.startswith('chroma') or strategy=='hybrid_rerank':client=portable_chroma_client()
    if strategy=='hybrid_rerank':reranker=portable_load_reranker()
    return types.SimpleNamespace(tokenizer=tokenizer,model=model,embedder=load_desktop_embedder(),parts=parts,
        release=release,report=report,chroma=client,reranker=reranker)


**Helper: `load_saved_deployment`**

Resolve the portable saved deployment, falling back to a verified legacy desktop release.

**Why here:** A visitor must use the saved selected model and its recorded index scope, rather than whichever in-memory model was last trained.


In [216]:
def load_saved_deployment():
    """Resolve the portable saved deployment, falling back to a verified legacy desktop release."""
    return portable_load_release()


**Helper: `portable_save_module`**

Publish a new release only after comparisons; validation/export modes load the existing release unchanged.

**Why here:** A visitor must use the saved selected model and its recorded index scope, rather than whichever in-memory model was last trained.


In [217]:
def portable_save_module():
    """Publish a new release only after comparisons; validation/export modes load the existing release unchanged."""
    if PORTABLE_RUN['mode'] in {'compare','process'}:
        release=portable_save_release(PORTABLE_RUN['report'],PORTABLE_RUN['folder'],PORTABLE_RUN['readers'],portable_scope())
    else:release=portable_release_pointer()
    PORTABLE_RUN['release']=release
    return {'release':release['id'],'selected':release['selected'],
        'retrieval_strategy':release.get('retrieval_strategy','lance_dense')}


**Execute Module 8.** Publish the selected measured deployment, or load an existing saved deployment for validation/export.


In [218]:
RELEASE_STATUS=portable_run_module('saved_release',portable_save_module,requires=['comparison'])
print(RELEASE_STATUS)


saved_release · starting
saved_release · complete · 0.8 s
{'release': '622880b84231acbd2e5a8691', 'selected': {'reader': 'previous', 'k': 3, 'method': 'concat', 'reader_id': 'c9b95b6765ffdc414618d6a6', 'reason': 'Selected on 24 development questions; not a final held-out result.'}, 'retrieval_strategy': 'chroma_hybrid'}


## Module 9 · Verify offline execution and preserve complete reports

The saved reader is reloaded in a fresh process. External Python socket connections are blocked while local/Unix IPC remains allowed,
including the Windows socket pair needed by asyncio/LanceDB. This fixes the earlier blanket-socket-blocking failure.
HF offline flags and local-only model loads remain active. This is a Python-level execution check, **not an OS firewall or accuracy test**.

The latest supplied report workspace is retained: measured matrices, training loss/perplexity where actually logged, manual dev samples,
answer calculations, full error logs and exportable CSV/JSON/PNG/HTML. Missing historical measurements remain unavailable.


**Helper: `offline_socket_guard_script`**

Allow local IPC while denying external Python socket connections in the child process.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [219]:
def offline_socket_guard_script():
    """Allow local IPC while denying external Python socket connections in the child process."""
    return '''import sys,socket,ipaddress,json
def _local_host(host):
    if host is None:return True
    if isinstance(host,bytes):host=host.decode('ascii',errors='replace')
    if str(host).lower().rstrip('.')=='localhost':return True
    try:address=ipaddress.ip_address(str(host).split('%',1)[0])
    except ValueError:return False
    mapped=getattr(address,'ipv4_mapped',None)
    return address.is_loopback or bool(mapped and mapped.is_loopback)
def _local_socket_only(event,args):
    if event in {'socket.connect','socket.sendto','socket.sendmsg'}:
        sock,target=args[0],args[-1]
        if sock.family==getattr(socket,'AF_UNIX',-1):return
        if target is None:return
        host=target[0] if isinstance(target,tuple) else target
    elif event in {'socket.getaddrinfo','socket.gethostbyname','socket.gethostbyaddr'}:
        host=args[0]
    else:return
    if not _local_host(host):raise RuntimeError('External Python socket access blocked in saved-pipeline verification')
sys.addaudithook(_local_socket_only)
_original_connect=socket.socket.connect;_original_connect_ex=socket.socket.connect_ex
def _checked_connect(sock,address):
    _local_socket_only('socket.connect',(sock,address))
    return _original_connect(sock,address)
def _checked_connect_ex(sock,address):
    _local_socket_only('socket.connect',(sock,address))
    return _original_connect_ex(sock,address)
socket.socket.connect=_checked_connect;socket.socket.connect_ex=_checked_connect_ex
'''


**Helper: `portable_offline_check_body`**

Perform a normal retrieval plus a forced saved-reader execution probe; neither is a factual accuracy test.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [220]:
def portable_offline_check_body():
    """Perform a normal retrieval plus a forced saved-reader execution probe; neither is a factual accuracy test."""
    runtime=portable_load_release();chosen=runtime.release['selected']
    hit=runtime.parts[0]['table'].search().select(['passage_id','article_id','title','url','text','source_shard']).limit(1).to_list()[0]
    normal=portable_answer('What information does '+hit['title']+' provide?',runtime)
    if not normal['retrieved']:raise RuntimeError('The saved retrieval adapter returned no source passages for its own source-title probe.')
    hit['_distance']=0.
    forced=answer_desktop('What does this source describe?',chosen['k'],chosen['method'],runtime,hits=[hit])
    if forced['model_calls']<1 or forced['output_tokens']<1:raise RuntimeError('The saved generator was not executed.')
    return {'passed':True,'release':runtime.release['id'],'retrieval_strategy':runtime.release.get('retrieval_strategy','lance_dense'),
        'shards':[p['number'] for p in runtime.parts],'retrieval_returned':len(normal['retrieved']),
        'forced_generator_calls':forced['model_calls'],'accuracy_test':False,
        'network_policy':'HF offline/local_files_only; Python sockets restricted to loopback/Unix IPC, not an OS firewall'}


**Helper: `portable_offline_verify`**

Allow Windows internal loopback IPC; save complete subprocess output and a fresh-platform proof.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [221]:
def portable_offline_verify():
    """Allow Windows internal loopback IPC; save complete subprocess output and a fresh-platform proof."""
    portable_export_runtime();release=portable_release_pointer()
    identity=digest([release['id'],portable_hardware(),inspect.getsource(portable_offline_check_body),
        inspect.getsource(offline_socket_guard_script)])[:24]
    folder=ROOT/'portable_verification'/identity;folder.mkdir(parents=True,exist_ok=True);marker=folder/'proof.json'
    if marker.exists() and read_json(marker).get('passed'):return read_json(marker)
    script=offline_socket_guard_script()+'''import portable_rag as p
from pathlib import Path
c=p.read_json(Path('portable_config.json'));c['offline']=True
p.portable_configure(c);p.resolve_desktop_lock()
print('PORTABLE_PROOF='+p.json.dumps(p.portable_offline_check_body()))
'''
    env=os.environ.copy();env.update(HF_HUB_OFFLINE='1',HF_DATASETS_OFFLINE='1',PYTHONIOENCODING='utf-8')
    result=subprocess.run([sys.executable,'-c',script],cwd=ROOT,env=env,capture_output=True,text=True,encoding='utf-8',timeout=600)
    (folder/'execution.log').write_text(result.stdout+'\nSTDERR\n'+result.stderr,encoding='utf-8')
    line=next((s for s in result.stdout.splitlines() if s.startswith('PORTABLE_PROOF=')),None)
    proof=json.loads(line.split('=',1)[1]) if result.returncode==0 and line else {'passed':False,'error':result.stderr[-4000:]}
    proof.update(log=(folder/'execution.log').relative_to(ROOT).as_posix(),platform=portable_hardware())
    atomic_json(marker,proof)
    if not proof['passed']:raise RuntimeError('Offline execution check failed. Saved compute retained. '+proof.get('error',''))
    return proof


**Helper: `web_sources_html`**

Show only evidence actually supplied to generation, with escaped text and validated Wikipedia links, labeled as sources rather than verified claims.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [222]:
def web_sources_html(spans):
    """Show only evidence actually supplied to generation, with escaped text and validated Wikipedia links, labeled as sources rather than verified claims."""
    if not spans:
        return '<p>No source evidence was supplied to the model.</p>'
    cards = ['<h3>Evidence used for this answer</h3>']
    for number, span in enumerate(spans, 1):
        title, text = html.escape(str(span.get('title', 'Source'))), html.escape(str(span.get('text', '')))
        url = str(span.get('url', ''))
        parsed = urllib.parse.urlsplit(url)
        domain = (parsed.hostname or '').lower()
        safe = parsed.scheme == 'https' and (domain == 'wikipedia.org' or domain.endswith('.wikipedia.org'))
        link = f'<a href="{html.escape(url, quote=True)}" target="_blank" rel="noopener noreferrer">{title}</a>' if safe else title
        cards.append(f'<details open><summary>[S{number}] {link}</summary><blockquote>{text}</blockquote></details>')
    cards.append('<p>These are source passages supplied to the model. Check them to verify its answer.</p>')
    return '\n'.join(cards)


**Helper: `desktop_html`**

Professional offline workspace with separate Q&A, run reports, shard samples and coverage tabs.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [223]:
def desktop_html():
    """Extend the saved report workspace with controls without changing any corpus or comparison identity."""
    markup=portable_markup()
    anchor='<div class="toolbar"><button id="ask"'
    markup=markup.replace(anchor,qa_controls_markup()+anchor,1)
    anchor='</aside></div></section>'
    markup=markup.replace(anchor,'</aside></div><div id="liveComparisons" class="hidden"></div></section>',1)
    return markup.replace('__STYLE__',qa_report_css()+qa_controls_css()).replace('__SCRIPT__',desktop_javascript())


**Helper: `desktop_javascript`**

Reuse the supplied offline report UI and display the deployed retrieval strategy.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [224]:
def desktop_javascript():
    """Reuse the supplied offline report UI and display the deployed retrieval strategy."""
    return 'const $=id=>document.getElementById(id);\nconst esc=v=>String(v??\'Unavailable\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst pct=v=>v==null?\'Unavailable\':(Number(v)*100).toFixed(2)+\'%\';\nconst number=(v,d=1)=>v==null?\'Unavailable\':Number(v).toFixed(d);\nconst table=(headers,rows)=>\'<div class="tablewrap"><table><thead><tr>\'+headers.map(h=>\'<th>\'+esc(h)+\'</th>\').join(\'\')+\'</tr></thead><tbody>\'+rows.map(r=>\'<tr>\'+r.map(v=>\'<td>\'+esc(v)+\'</td>\').join(\'\')+\'</tr>\').join(\'\')+\'</tbody></table></div>\';\nlet lastAnswer=null,sampleId=null,samples=[],reportId=\'\',errorOffset=0,errorTotal=0;\nasync function api(path,options){const response=await fetch(path,options);const data=await response.json();if(!response.ok)throw Error(data.error||\'Request failed\');return data;}\nfunction page(name){document.querySelectorAll(\'.page\').forEach(x=>x.classList.toggle(\'hidden\',x.id!==name));document.querySelectorAll(\'.tab\').forEach(x=>x.classList.toggle(\'active\',x.dataset.page===name));}\ndocument.querySelectorAll(\'.tab\').forEach(x=>x.onclick=()=>page(x.dataset.page));\nfunction download(name,text,type){const url=URL.createObjectURL(new Blob([text],{type}));const link=document.createElement(\'a\');link.href=url;link.download=name;link.click();setTimeout(()=>URL.revokeObjectURL(url),1000);}\nfunction csvCell(value){let s=value==null?\'\':String(value);if(/^[=+@\\-]/.test(s))s="\'"+s;return \'"\'+s.replace(/"/g,\'""\')+\'"\';}\nfunction calculationHtml(c){const r=c.reference,l=c.lexical_proxy;let html=\'<details open><summary>Reference Exact Match / token F1</summary><p class="small">\'+esc(r.normalization)+\'</p><p class="small">\'+esc(r.formula)+\'</p><p><b>Normalized answer:</b> \'+esc(r.normalized_answer)+\'</p>\';\nhtml+=r.aliases.length?table([\'Reference\',\'Shared tokens\',\'Predicted tokens\',\'Reference tokens\',\'Precision\',\'Recall\',\'EM\',\'F1\'],r.aliases.map(x=>[x.reference,x.common_tokens,x.predicted_token_count,x.reference_token_count,pct(x.precision),pct(x.recall),pct(x.EM),pct(x.F1)])):\'<p class="small">No gold/user reference supplied. EM/F1 are unavailable.</p>\';html+=\'</details><details><summary>Lexical unsupported-sentence proxy</summary><p class="small">\'+esc(l.formula)+\'</p>\'+table([\'Raw-answer sentence\',\'Shared unique words\',\'Unique words\',\'Overlap\',\'Below 0.8?\'],l.sentences.map(x=>[x.sentence,x.supported_unique_tokens,x.unique_tokens,pct(x.overlap),x.flagged_below_0_8]))+\'</details>\';\nhtml+=\'<details><summary>Retrieval gate / factual hallucination</summary><p class="small">\'+esc(c.retrieval.formula)+\'</p>\'+table([\'Setting\',\'Observed / configured\'],[[\'Maximum cosine distance\',c.retrieval.gate],[\'Returned passages\',c.retrieval.returned],[\'Supplied passages\',c.retrieval.supplied],[\'Answer occurrence gate\',c.retrieval.require_answer_occurrence],[\'Factual hallucination index\',\'Not measured\']])+\'</details>\';return html;}\nfunction renderAnswer(o){lastAnswer=o;$(\'answer\').textContent=o.answer;$(\'sources\').innerHTML=o.sources_html;$(\'calculations\').className=\'calculation\';$(\'calculations\').innerHTML=calculationHtml(o.calculations);const ref=o.reference_scores||{};\nconst rows=[[\'Retrieval strategy\',o.retrieval_strategy||\'lance_dense\'],[\'Saved reader\',o.model_version],[\'Release / scope\',o.release_id+\' · shards \'+o.shards.join(\', \')],[\'Retrieval k / fusion\',o.k+\' / \'+o.method],[\'Retrieved / supplied passages\',o.retrieved.length+\' / \'+o.evidence.length],[\'Retrieval distances\',o.retrieved.map(x=>number(x._distance,4)).join(\', \')],[\'Retrieval (embedding + vector lookup)\',number(o.retrieval_ms)+\' ms\'],[\'Generation\',number(o.generation_ms)+\' ms\'],[\'Model pipeline total\',number(o.total_ms)+\' ms\'],[\'Backend request incl. score calculation\',number(o.request_ms)+\' ms\'],[\'Input / generated tokens\',o.input_tokens+\' / \'+o.output_tokens],[\'Generator calls\',o.model_calls],[\'Aggregate generated tokens/s\',number(o.generated_tokens_per_second)],[\'Abstained\',o.abstained],[\'Reference source\',o.reference_kind],[\'Reference Exact Match\',pct(ref.EM)],[\'Reference token F1\',pct(ref.F1)],[\'Raw-answer lexical unsupported proxy\',pct(o.unsupported_sentence_proxy)],[\'Factual hallucination index\',\'Not measured\']];\nif(o.abstention_reason)rows.push([\'Abstention reason\',o.abstention_reason]);$(\'metrics\').className=\'\';$(\'metrics\').innerHTML=table([\'Metric\',\'This question\'],rows);$(\'rawSection\').classList.remove(\'hidden\');$(\'rawAnswer\').textContent=JSON.stringify({raw_model_answer:o.raw_model_answer,intermediate:o.intermediate||[]},null,2);$(\'answerJson\').disabled=false;$(\'answerCsv\').disabled=false;}\n$(\'ask\').onclick=async()=>{const button=$(\'ask\');button.disabled=true;$(\'message\').textContent=\'\';$(\'activity\').textContent=\'Retrieving evidence and generating with the saved local model…\';const began=performance.now();try{const output=await api(\'/api/answer\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({question:$(\'question\').value,reference:$(\'reference\').value,sample_id:sampleId})});renderAnswer(output);$(\'activity\').textContent=\'Completed in \'+((performance.now()-began)/1000).toFixed(1)+\' s. Diagnostics saved locally.\';}catch(e){$(\'message\').textContent=e.message;$(\'activity\').textContent=\'Request failed; inspect Run reports → Full error logs.\';}finally{button.disabled=false;}};\n$(\'question\').addEventListener(\'input\',()=>{sampleId=null;$(\'sampleContext\').textContent=\'\';});$(\'reference\').addEventListener(\'input\',()=>{sampleId=null;$(\'sampleContext\').textContent=\'Using the reference you entered; it is scored after generation.\';});\n$(\'answerJson\').onclick=()=>download(\'answer_\'+lastAnswer.answer_id+\'.json\',JSON.stringify(lastAnswer,null,2),\'application/json\');\n$(\'answerCsv\').onclick=()=>{const keys=[\'question\',\'answer\',\'model_version\',\'total_ms\',\'retrieval_ms\',\'generation_ms\',\'input_tokens\',\'output_tokens\',\'model_calls\',\'abstained\'];const row={...lastAnswer,EM:lastAnswer.reference_scores?.EM,F1:lastAnswer.reference_scores?.F1};keys.push(\'EM\',\'F1\');download(\'answer_\'+row.answer_id+\'.csv\',keys.map(csvCell).join(\',\')+\'\\n\'+keys.map(k=>csvCell(row[k])).join(\',\')+\'\\n\',\'text/csv\');};\nasync function status(){const s=await api(\'/api/status\');$(\'indexed\').textContent=s.searchable.length+\' / \'+s.rows.length;$(\'pending\').textContent=s.rows.filter(x=>x.indexing!==\'complete\').length;$(\'selected\').textContent=s.selected_shard;$(\'version\').textContent=s.model_version;$(\'scope\').textContent=\'Searchable shards: \'+s.searchable.join(\', \')+\' · \'+s.scope;$(\'shardTable\').innerHTML=table([\'Shard\',\'Download\',\'Embedding\',\'Index\',\'Fine-tuning\',\'Evaluation\',\'Searchable\',\'Passages\',\'Training note\'],s.rows.map(x=>[x.shard,x.download,x.embedding,x.indexing,x.training,x.evaluation,s.searchable.includes(x.shard)?\'Yes\':\'No\',x.passages,x.training_note]));$(\'statusStamp\').textContent=\'Status read from saved receipts: \'+new Date().toLocaleString();}\nfunction imageHtml(image){return \'<figure\'+(image.name.startsWith(\'matrix_\')?\' class="matrix"\':\'\')+\'><figcaption>\'+esc(image.name.replace(/_/g,\' \'))+\'</figcaption><img loading="lazy" alt="\'+esc(image.name)+\'" src="data:image/png;base64,\'+image.data+\'"></figure>\';}\nasync function reports(){const r=await api(\'/api/report\'+(reportId?\'?release=\'+encodeURIComponent(reportId):\'\'));reportId=r.release_id;const m=r.metadata,chosen=m.selected_reader;const selected=r.comparison.find(x=>x.reader===chosen.reader&&x.method===chosen.method&&x.k===chosen.k);$(\'reportScope\').textContent=r.evaluation_scope+\' · Viewing report \'+r.release_id+\'; the Main Q&A model remains the active deployment.\';\n$(\'reportRun\').innerHTML=r.available_reports.map(x=>\'<option value="\'+esc(x.release_id)+\'"\'+(x.release_id===reportId?\' selected\':\'\')+\'>\'+esc(x.release_id)+\'</option>\').join(\'\');$(\'reportMetrics\').innerHTML=[[\'Selected EM\',pct(selected?.EM)],[\'Selected F1\',pct(selected?.F1)],[\'Development questions\',m.question_count],[\'Indexed shards\',m.coverage.length+\' / 41\']].map(x=>\'<div class="stat"><small>\'+esc(x[0])+\'</small><strong>\'+esc(x[1])+\'</strong></div>\').join(\'\');$(\'reportMeta\').textContent=\'Reader \'+chosen.reader+\' · \'+chosen.reader_id+\' · k=\'+chosen.k+\' · \'+chosen.method+\' · exported \'+r.created_at;\n$(\'exports\').innerHTML=r.exports.map(x=>\'<a href="/api/export?release=\'+encodeURIComponent(reportId)+\'&file=\'+encodeURIComponent(x.name)+\'" download>\'+esc(x.name)+\' · \'+(x.bytes/1024).toFixed(1)+\' KB</a>\').join(\'\');$(\'perplexityNote\').textContent=r.perplexity_note;$(\'trainingStages\').innerHTML=table([\'Shard\',\'Training\',\'Model\',\'History source\',\'Validation availability\',\'Note\'],r.training_stages.map(x=>[x.shard,x.status,x.model_version,x.history_source,x.validation_note,x.note]));$(\'trainingHistory\').innerHTML=r.training_history.length?table([\'Shard\',\'Step\',\'Epoch\',\'Train loss\',\'Validation loss\',\'Conditional val. perplexity\'],r.training_history.map(x=>[x.shard,x.step,x.epoch,number(x.train_loss,4),number(x.validation_loss,4),number(x.validation_perplexity,3)])):\'<p class="no-data">No saved training loss history is available. Completed training has not been rerun to create a curve.</p>\';\n$(\'comparison\').innerHTML=r.comparison.length?table([\'Reader\',\'Fusion\',\'k\',\'EM\',\'F1\',\'Article hit\',\'Article MRR\',\'Alias in evidence\',\'Mean latency (ms)\'],r.comparison.map(x=>[x.reader,x.method,x.k,pct(x.EM),pct(x.F1),pct(x.article_hit),number(x.article_MRR,3),pct(x.alias_in_evidence),number(x.total_ms)])):\'<p class="no-data">No eligible development questions; reference scores and comparison matrices are unavailable.</p>\';\n$(\'trainingFigures\').innerHTML=r.images.filter(x=>x.name.startsWith(\'training_\')).map(imageHtml).join(\'\')||\'<p class="no-data">No logged points support a training/validation curve for this run.</p>\';$(\'matrixFigures\').innerHTML=r.images.filter(x=>x.name.startsWith(\'matrix_\')).map(imageHtml).join(\'\');$(\'errorFigures\').innerHTML=r.images.filter(x=>x.name===\'answer_error_breakdown.png\').map(imageHtml).join(\'\');errorOffset=0;await errors();}\nasync function errors(){const data=await api(\'/api/errors?release=\'+encodeURIComponent(reportId)+\'&offset=\'+errorOffset+\'&limit=20\');errorTotal=data.total;$(\'errorRows\').innerHTML=data.rows.length?data.rows.map(x=>\'<details><summary>\'+esc(x.reader)+\' · \'+esc(x.method)+\' · k=\'+x.k+\' · \'+esc(x.category)+\' · \'+esc(x.question)+\'</summary><p><b>Expected:</b> \'+esc(x.expected_answers.join(\' | \'))+\'</p><p><b>Model:</b> \'+esc(x.answer)+\'</p><p>EM \'+pct(x.EM)+\' · F1 \'+pct(x.F1)+\' · shard \'+x.question_shard+\'</p><pre>\'+esc(JSON.stringify(x.calculation,null,2))+\'</pre></details>\').join(\'\'):\'<p class="no-data">No saved development predictions.</p>\';$(\'errorPage\').textContent=errorTotal?`${errorOffset+1}–${Math.min(errorOffset+20,errorTotal)} of ${errorTotal} measured answers`:\'0 measured answers\';$(\'errorsPrevious\').disabled=errorOffset===0;$(\'errorsNext\').disabled=errorOffset+20>=errorTotal;}\nasync function logs(){const data=await api(\'/api/logs\');$(\'logNote\').textContent=data.note;$(\'logs\').innerHTML=data.files.length?data.files.map((x,i)=>\'<details><summary>\'+esc(x.name)+\' · \'+x.bytes+\' bytes · full content</summary><button data-log="\'+i+\'">Download full log</button><pre>\'+esc(x.text||\'(empty log)\')+\'</pre></details>\').join(\'\'):\'<p class="no-data">No captured log files are present.</p>\';$(\'logs\').querySelectorAll(\'[data-log]\').forEach(button=>button.onclick=()=>{const log=data.files[Number(button.dataset.log)];download(log.name.replaceAll(\'/\',\'_\'),log.text,\'text/plain\');});$(\'historicalErrors\').innerHTML=table([\'Shard\',\'Saved historical summary\'],data.historical_summaries.map(x=>[x.shard,x.last_error]));}\n$(\'reportRun\').onchange=async()=>{reportId=$(\'reportRun\').value;await reports().catch(showError);};$(\'refreshReports\').onclick=()=>Promise.all([reports(),logs()]).catch(showError);$(\'errorsPrevious\').onclick=()=>{errorOffset=Math.max(0,errorOffset-20);errors().catch(showError);};$(\'errorsNext\').onclick=()=>{errorOffset+=20;errors().catch(showError);};\nfunction sampleCards(){const shard=Number($(\'sampleShard\').value);const rows=samples.filter(x=>x.shard===shard);$(\'sampleCards\').innerHTML=rows.length?rows.map(x=>\'<article class="sample"><div class="tag">Shard \'+x.shard+\' · \'+esc(x.reference_kind.replaceAll(\'_\',\' \'))+\'</div><h3>\'+esc(x.question)+\'</h3><p class="small">\'+esc(x.reference_label)+\'</p><p><b>Saved-model answer:</b> \'+esc(x.output?.answer||x.error||\'Unavailable\')+\'</p><p class="small">\'+esc(x.prediction_origin||x.status)+\'</p><button class="primary" data-sample="\'+esc(x.sample_id)+\'">Use this question in Main Q&A</button><details><summary>Expected answer and source evidence</summary><p><b>Reference:</b> \'+esc(x.answers.join(\' | \')||\'No gold answer; inspect the source excerpt.\')+\'</p><p><b>Source article:</b> \'+esc(x.title)+\'</p><blockquote>\'+esc(x.context)+\'</blockquote></details>\'+(x.calculation?\'<details><summary>Saved sample score calculations</summary>\'+calculationHtml(x.calculation)+\'</details>\':\'\')+\'</article>\').join(\'\'):\'<p class="no-data">No source passage was available for a sample in this shard.</p>\';\n$(\'sampleCards\').querySelectorAll(\'[data-sample]\').forEach(button=>button.onclick=()=>{const sample=samples.find(x=>x.sample_id===button.dataset.sample);sampleId=sample.sample_id;$(\'question\').value=sample.question;$(\'reference\').value=sample.answers[0]||\'\';$(\'sampleContext\').textContent=\'Sample from shard \'+sample.shard+\' · \'+sample.reference_label;page(\'qa\');$(\'question\').focus();});}\nasync function loadSamples(){const data=await api(\'/api/samples\');samples=data.samples;$(\'sampleShard\').innerHTML=data.coverage.map(n=>\'<option value="\'+n+\'"\'+(n===data.selected_shard?\' selected\':\'\')+\'>Shard \'+n+\' / 41</option>\').join(\'\');$(\'sampleLimit\').textContent=\'Up to \'+data.per_shard_limit+\' samples per deployed shard · actual saved final-model predictions\';sampleCards();}\n$(\'sampleShard\').onchange=sampleCards;\nfunction showError(error){$(\'message\').textContent=error.message;$(\'scope\').textContent=\'Local report/status error: \'+error.message;}\nPromise.all([status(),reports(),logs(),loadSamples()]).catch(showError);setInterval(()=>status().catch(()=>{}),15000);\n' + portable_ui_script() + qa_controls_script()


**Helper: `qa_status_payload`**

Expose status and coverage without private filesystem paths or stale process IDs.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [225]:
def qa_status_payload(runtime):
    """Expose status and coverage without private filesystem paths or stale process IDs."""
    selected=read_json(ROOT/'desktop_selection.json')['shard'] if (ROOT/'desktop_selection.json').exists() else 1
    return {'rows':desktop_status(),'searchable':[p['number'] for p in runtime.parts],'selected_shard':selected,
        'model_version':runtime.release['selected']['reader_id'],'scope':runtime.report['evaluation_scope']}


**Helper: `qa_report_payload`**

Keep the report-tab response small; complete calculations/logs remain in verified exports and separate endpoints.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [226]:
def qa_report_payload(runtime,release_id=None):
    """Keep the report-tab response small; complete calculations/logs remain in verified exports and separate endpoints."""
    pointer,folder=resolve_desktop_report(release_id or runtime.release['id']);data=read_json(folder/'report.json')
    result={k:data[k] for k in ['release_id','evaluation_scope','metadata','comparison','training_history','training_stages','perplexity_note','created_at']}
    result['images']=[{'name':name,'data':base64.b64encode((folder/name).read_bytes()).decode('ascii')} for name in pointer['images']]
    result['exports']=[{'name':name,'bytes':info['size']} for name,info in pointer['files'].items()]
    result['available_reports']=available_desktop_reports()
    result['diagnostic_counts']=dict(collections.Counter(r['category'] for r in data['answer_diagnostics']))
    return result


**Helper: `live_desktop_answer`**

Score references only after generation; sample references are validated server-side and never enter prompts.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


**Interactive helper: `qa_backend_status`**

Discover existing derived artifacts without creating a database or restarting conversion.


In [227]:
def qa_backend_status(runtime):
    """Discover existing derived artifacts without creating a database or restarting conversion."""
    import sqlite3
    cached=getattr(runtime,'control_backends',None)
    if cached and time.monotonic()-cached['at']<15:return cached['rows']
    rows={'lance_dense':{'available':True,'reason':'Verified deployed LanceDB scope'}}
    receipts=[]
    for path in (ROOT/'derived/receipts').glob('*.json'):
        try:receipts.append(read_json(path))
        except (ValueError,OSError):continue
    missing_chroma=[];missing_bm25=[]
    for part in runtime.parts:
        key=portable_part_key(part);count=part['record']['passages']
        matches=[r for r in receipts if r.get('spec',{}).get('source')==key
            and r.get('status')=='complete' and r.get('verified_rows')==count
            and r.get('spec',{}).get('chroma_version')==metadata.version('chromadb')]
        if not matches or not (ROOT/'derived/chroma').is_dir():missing_chroma.append(part['number'])
        path=ROOT/'derived/lexical'/(key+'.sqlite3');valid=False
        if path.is_file():
            try:
                with sqlite3.connect(path.as_uri()+'?mode=ro',uri=True) as db:
                    state=dict(db.execute('SELECT key,value FROM state'))
                valid=state.get('source')==key and state.get('status')=='complete' and int(state.get('cursor',-1))==count
            except (sqlite3.Error,ValueError):pass
        if not valid:missing_bm25.append(part['number'])
    for name,missing in [('chroma_dense',missing_chroma),('bm25',missing_bm25)]:
        rows[name]={'available':not missing,'reason':('Incomplete/unavailable for shard(s) '+str(missing)) if missing else 'Completed derived index available'}
    runtime.control_backends={'at':time.monotonic(),'rows':rows}
    return rows


**Interactive helper: `qa_control_options`**

Expose defaults from the frozen release; UI choices never change saved training/index settings.


In [228]:
def qa_control_options(runtime):
    """Expose defaults from the frozen release; UI choices never change saved training/index settings."""
    status=qa_backend_status(runtime);strategy=runtime.release.get('retrieval_strategy','lance_dense')
    required=['chroma_dense','bm25'] if strategy in {'chroma_hybrid','hybrid_rerank'} else (
        ['lance_dense','bm25'] if strategy=='lance_hybrid' else [strategy])
    saved_ok=all(status.get(n,{'available':False})['available'] for n in required)
    choices=[{'value':'saved','label':'Saved deployment · '+strategy,'available':saved_ok,'reason':'Uses the saved selected strategy'}]
    for key,label in [('lance_dense','LanceDB only'),('chroma_dense','Chroma only'),('bm25','BM25 only')]:
        choices.append({'value':key,'label':label,**status[key]})
    choices.append({'value':'compare','label':'Compare all three · sequential',
        'available':all(r['available'] for r in status.values()),'reason':'Requires completed indexes for every deployed shard'})
    selected=runtime.release['selected'];cap=int(CFG['max_new_tokens'])
    return {'backends':choices,'defaults':{'backend':'saved','k':selected['k'],'method':selected['method'],
        'max_new_tokens':cap,'sampling':False,'temperature':1.,'top_p':1.,'seed':int(CFG['seed'])},
        'k_values':sorted(set([1,3,5,10,int(selected['k'])])),
        'token_limits':sorted(set([64,128,256,cap])),'shards':[p['number'] for p in runtime.parts],
        'input_cap':CFG['input_cap'],'model_version':selected['reader_id'],
        'note':'Interactive experiments use one saved reader and the same shard scope. Saved evaluation matrices remain unchanged.'}


**Interactive helper: `qa_parse_controls`**

Validate bounded request-local controls before any encoding, retrieval or model execution.


In [229]:
def qa_parse_controls(payload,runtime):
    """Validate bounded request-local controls before any encoding, retrieval or model execution."""
    supplied=payload.get('controls',{})
    if not isinstance(supplied,dict):raise ValueError('Controls must be an object.')
    options=qa_control_options(runtime);settings=dict(options['defaults'])
    if set(supplied)-set(settings):raise ValueError('Unknown Q&A control.')
    settings.update(supplied)
    choice=next((b for b in options['backends'] if b['value']==settings['backend']),None)
    if not choice or not choice['available']:raise ValueError('Selected backend is unavailable; finish its conversion before validation.')
    for name in ['k','max_new_tokens','seed']:
        if isinstance(settings[name],bool) or not isinstance(settings[name],int):raise ValueError(name+' must be an integer.')
    if settings['k'] not in options['k_values']:raise ValueError('Unsupported retrieval top-k.')
    if not 1<=settings['max_new_tokens']<=max(options['token_limits']):raise ValueError('Answer token limit is outside the supported range.')
    if not 0<=settings['seed']<2**31:raise ValueError('Seed is outside the supported range.')
    if settings['method'] not in {'concat','generate_merge'}:raise ValueError('Unsupported answer method.')
    if not isinstance(settings['sampling'],bool):raise ValueError('Sampling must be true or false.')
    for name,low,high in [('temperature',0.1,2.),('top_p',0.05,1.)]:
        value=settings[name]
        if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or not low<=value<=high:
            raise ValueError('Invalid '+name+'.')
        settings[name]=float(value)
    if not settings['sampling']:settings.update(temperature=1.,top_p=1.)
    return settings


**Interactive helper: `qa_generate_controlled`**

Use request-local generation arguments; leave CFG and the saved model configuration untouched.


In [230]:
def qa_generate_controlled(prompt,tokenizer,model,settings):
    """Use request-local generation arguments; leave CFG and the saved model configuration untouched."""
    encoded=tokenizer(prompt,return_tensors='pt',truncation=False)
    if encoded.input_ids.shape[1]>CFG['input_cap']:raise ValueError('Generator prompt exceeds the declared input budget.')
    encoded={k:v.to(DEVICE) for k,v in encoded.items() if k in {'input_ids','attention_mask'}}
    arguments={'max_new_tokens':settings['max_new_tokens'],'do_sample':settings['sampling'],'num_beams':1}
    if settings['sampling']:arguments.update(temperature=settings['temperature'],top_p=settings['top_p'],top_k=0)
    if DEVICE!='cpu':torch.cuda.synchronize()
    started=time.perf_counter()
    with torch.inference_mode():output=model.generate(**encoded,**arguments)
    if DEVICE!='cpu':torch.cuda.synchronize()
    return {'answer':tokenizer.decode(output[0],skip_special_tokens=True).strip(),
        'input_tokens':int(encoded['input_ids'].shape[1]),'output_tokens':int(output.shape[1]-1),
        'generation_ms':(time.perf_counter()-started)*1000}


**Interactive helper: `qa_answer_controlled`**

Reuse the original evidence/abstention/fusion functions with an isolated generation binding.


In [231]:
def qa_answer_controlled(question,runtime,strategy,settings,vector,encode_ms):
    """Reuse the original evidence/abstention/fusion functions with an isolated generation binding."""
    began=time.perf_counter();client=getattr(runtime,'chroma',None)
    if strategy.startswith('chroma') or strategy=='hybrid_rerank':
        if client is None:
            client=portable_chroma_client();runtime.chroma=client
    setup_ms=(time.perf_counter()-began)*1000;started=time.perf_counter()
    hits=portable_retrieve(question,settings['k'],runtime.embedder,runtime.parts,strategy,
        client,getattr(runtime,'reranker',None),vector=vector)
    retrieval_ms=(time.perf_counter()-started)*1000
    # Functions receive a private global namespace; original functions and CFG retain their exact identities.
    namespace=dict(answer_desktop.__globals__)
    namespace['generate_desktop']=lambda p,t,m:qa_generate_controlled(p,t,m,settings)
    merge=generate_merge_desktop
    namespace['generate_merge_desktop']=types.FunctionType(merge.__code__,namespace,merge.__name__,merge.__defaults__,merge.__closure__)
    answer=types.FunctionType(answer_desktop.__code__,namespace,answer_desktop.__name__,answer_desktop.__defaults__,answer_desktop.__closure__)
    devices=[torch.cuda.current_device()] if DEVICE!='cpu' else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(settings['seed'])
        result=answer(question,settings['k'],settings['method'],runtime,hits=hits)
    result.update(retrieval_strategy=strategy,retrieval_ms=retrieval_ms,query_encode_ms=encode_ms,
        backend_setup_ms=setup_ms,total_ms=(time.perf_counter()-began)*1000+encode_ms)
    return result


**Interactive helper: `qa_request_references`**

Resolve visitor/sample references for post-generation scoring only.


In [232]:
def qa_request_references(payload,runtime,question):
    """Resolve visitor/sample references for post-generation scoring only."""
    reference=str(payload.get('reference','')).strip()
    if len(reference)>4000:raise ValueError('Reference exceeds 4000 characters.')
    refs=[reference] if reference else [];kind='visitor_reference' if reference else 'none'
    if payload.get('sample_id'):
        samples=qa_samples_payload(runtime)['samples']
        sample=next((s for s in samples if s['sample_id']==payload['sample_id']),None)
        if not sample or sample['question']!=question:raise ValueError('Sample question changed; clear the sample selection.')
        refs=sample['answers'];kind=sample['reference_kind']
    return refs,kind


**Interactive helper: `qa_controlled_result`**

Attach the exact interactive settings, saved model identity, evidence and honest metrics to one answer.


In [233]:
def qa_controlled_result(question,runtime,strategy,settings,vector,encode_ms,refs,kind,comparison_id):
    """Attach the exact interactive settings, saved model identity, evidence and honest metrics to one answer."""
    output=qa_answer_controlled(question,runtime,strategy,settings,vector,encode_ms)
    details=answer_calculation_details(output,refs);generation=output.get('generation_ms',0.)
    result={**output,'answer_id':uuid.uuid4().hex,'comparison_id':comparison_id,'at':datetime.now(timezone.utc).isoformat(),
        'question':question,'reference_kind':kind,'release_id':runtime.release['id'],
        'model_version':runtime.release['selected']['reader_id'],'shards':[p['number'] for p in runtime.parts],
        'sources_html':web_sources_html(output.get('evidence',[])),'calculations':details,
        'reference_scores':{k:details['reference'][k] for k in ['EM','F1']} if refs else None,
        'k':settings['k'],'method':settings['method'],'controls':dict(settings),
        'evaluation_scope':'Interactive exploratory result; saved evaluation matrices are unchanged.',
        'retrieved_passages':len(output.get('retrieved',[])),'supplied_passages':len(output.get('evidence',[])),
        'generated_tokens_per_second':output.get('output_tokens',0)*1000/generation if generation>0 else None}
    persist_desktop_answer(result)
    return result


**Interactive helper: `qa_comparison_row`**

Flatten one backend's real measurements; unavailable reference scores remain null.


In [234]:
def qa_comparison_row(answer):
    """Flatten one backend's real measurements; unavailable reference scores remain null."""
    ref=answer.get('reference_scores') or {}
    return {'strategy':answer['retrieval_strategy'],'EM':ref.get('EM'),'F1':ref.get('F1'),
        'retrieval_ms':answer['retrieval_ms'],'generation_ms':answer.get('generation_ms',0.),'total_ms':answer['total_ms'],
        'input_tokens':answer.get('input_tokens',0),'output_tokens':answer.get('output_tokens',0),
        'retrieved_passages':answer['retrieved_passages'],'supplied_passages':answer['supplied_passages'],
        'abstained':answer['abstained'],'lexical_unsupported_proxy':answer.get('unsupported_sentence_proxy'),
        'hallucination_index':None}


**Interactive helper: `qa_comparison_history`**

Persist paired results outside frozen reports; aggregate only matching settings and complete comparisons.


In [235]:
def qa_comparison_history(runtime,settings,question,rows,comparison_id,reference_kind):
    """Persist paired results outside frozen reports; aggregate only matching settings and complete comparisons."""
    spec={'protocol':'interactive-three-way-v1','application':qa_application_version(),'release':runtime.release,'settings':dict(settings),
        'hardware':portable_hardware(),'input_cap':CFG['input_cap'],'gate':CFG['max_cosine_distance'],
        'require_answer_occurrence':CFG['require_answer_occurrence'],'candidates':CFG['retrieval_candidates']}
    key=digest(spec)[:24];folder=ROOT/'qa_sessions'/runtime.release['id']/'comparisons'/key
    atomic_json(folder/'protocol.json',spec)
    event={'comparison_id':comparison_id,'at':datetime.now(timezone.utc).isoformat(),'question':question,
        'reference_kind':reference_kind,'rows':rows}
    path=folder/'runs.jsonl'
    if path.exists() and path.stat().st_size:
        # Trim only an incomplete final telemetry record before appending the next complete comparison.
        with open(path,'rb+') as handle:
            data=handle.read()
            if not data.endswith(b'\n'):handle.seek(0);handle.truncate(data.rfind(b'\n')+1)
    append_report_jsonl(path,event)
    events=read_report_jsonl(path)[-200:];summary=[]
    for strategy in ['lance_dense','chroma_dense','bm25']:
        group=[r for e in events for r in e['rows'] if r['strategy']==strategy]
        result={'strategy':strategy,'questions':len(group),'reference_scored':sum(r['F1'] is not None for r in group)}
        for metric in rows[0]:
            if metric=='strategy':continue
            values=[r[metric] for r in group if r.get(metric) is not None]
            result[metric]=sum(map(float,values))/len(values) if values else None
        summary.append(result)
    write_report_csv(folder/'summary.csv',summary);atomic_json(folder/'summary.json',summary)
    return {'group':key,'completed_requests':len(events),'rows':summary,
        'note':'Means over the latest 200 complete paired requests with identical settings/release/hardware. Repeated questions count again. Interactive exploration, not held-out accuracy. EM/F1 use reference-scored requests only.'}


**Interactive helper: `qa_inline_render`**

Show the same API results and three-way matrices inside a remote Kaggle notebook.


In [236]:
def qa_inline_render(result,show_evidence=True):
    """Show the same API results and three-way matrices inside a remote Kaggle notebook."""
    from IPython.display import HTML,display
    answers=result.get('answers',[result])
    for answer in answers:
        display(HTML('<div style="padding:16px;border:1px solid #94a3b8;border-radius:12px"><h3>'+html.escape(
            answer['retrieval_strategy'])+'</h3><p>'+html.escape(answer['answer'])+'</p></div>'))
        display(pd.DataFrame([qa_comparison_row(answer)]))
        if show_evidence:
            display(HTML(answer['sources_html']))
            display(pd.DataFrame([{k:r.get(k) for k in ['passage_id','article_id','title','source_shard','_distance']} for r in answer.get('retrieved',[])]))
            display(answer['calculations'])
    if result.get('comparison'):
        display(HTML('<h3>Current question · backend metric matrix</h3>'));display(pd.DataFrame(result['matrix']))
        print(result['summary']['note']);display(pd.DataFrame(result['summary']['rows']))
    print('Saved model:',result['model_version'],'| Shards:',result['shards'])
    print('Settings:',result['controls'])


**Interactive helper: `qa_inline_controls`**

Mirror laptop controls using Kaggle-supported widgets; no remote-localhost browser dependency.


In [237]:
def qa_inline_controls(options):
    """Mirror laptop controls using Kaggle-supported widgets; no remote-localhost browser dependency."""
    import ipywidgets as widgets
    d=options['defaults']
    backend_choices=[(b['label'],b['value']) for b in options['backends'] if b['available']]
    default='saved' if any(value=='saved' for _,value in backend_choices) else backend_choices[0][1]
    fields={'backend':widgets.Dropdown(options=backend_choices,value=default,description='Search:'),
        'k':widgets.Dropdown(options=options['k_values'],value=d['k'],description='Passages:'),
        'method':widgets.Dropdown(options=[('Concatenate','concat'),('Generate and merge','generate_merge')],value=d['method'],description='Method:'),
        'max_new_tokens':widgets.Dropdown(options=options['token_limits'],value=d['max_new_tokens'],description='Answer tokens:'),
        'sampling':widgets.Checkbox(value=False,description='Enable sampling'),
        'temperature':widgets.FloatSlider(value=1.,min=.1,max=2.,step=.1,description='Temperature:',disabled=True),
        'top_p':widgets.FloatSlider(value=1.,min=.05,max=1.,step=.05,description='Top-p:',disabled=True),
        'seed':widgets.BoundedIntText(value=d['seed'],min=0,max=2**31-1,description='Seed:')}
    def toggle(change):
        for key in ['temperature','top_p']:fields[key].disabled=not fields['sampling'].value
    fields['sampling'].observe(toggle,names='value')
    return fields


**Interactive helper: `qa_inline_callback`**

Use the same serialized server endpoint as the desktop HTML; generation stays in one worker.


In [238]:
def qa_inline_callback(connection,question,reference,fields,evidence,button,output):
    """Use the same serialized server endpoint as the desktop HTML; generation stays in one worker."""
    button.disabled=True
    with output:
        output.clear_output(wait=True)
        try:
            body=json.dumps({'question':question.value,'reference':reference.value,
                'controls':{k:v.value for k,v in fields.items()}}).encode('utf-8')
            request=urllib.request.Request(connection['url']+'/api/answer',data=body,
                headers={'Content-Type':'application/json'},method='POST')
            with urllib.request.urlopen(request,timeout=3600) as response:result=json.load(response)
            qa_inline_render(result,evidence.value)
            from IPython.display import HTML,display
            data=base64.b64encode(json.dumps(report_json_safe(result),ensure_ascii=False).encode()).decode()
            display(HTML('<a download="qa_result.json" href="data:application/json;base64,'+data+'">Download answer(s) and matrices JSON</a>'))
        except Exception as error:
            record_desktop_error('Kaggle inline Q&A',error);print(type(error).__name__+': '+str(error))
        finally:button.disabled=False


**Interactive helper: `qa_controls_markup`**

Embedded offline backend/generation controls.


In [239]:
def qa_controls_markup():
    'Embedded offline backend/generation controls.'
    return '<div class="qa-control-box">\n  <div class="eyebrow">REQUEST SETTINGS · SAVED MODEL UNCHANGED</div>\n  <div class="qa-control-grid">\n    <div><label for="searchBackend">Search backend</label><select id="searchBackend"><option>Loading…</option></select></div>\n    <div><label for="retrievalK">Passages to retrieve</label><select id="retrievalK"></select></div>\n    <div><label for="answerMethod">Answer method</label><select id="answerMethod"><option value="concat">Concatenate evidence</option><option value="generate_merge">Generate per passage, then merge</option></select></div>\n    <div><label for="answerTokens">Maximum answer tokens</label><select id="answerTokens"></select></div>\n  </div>\n  <label class="qa-check"><input type="checkbox" id="showEvidence" checked> Show source evidence, distances and calculation details</label>\n  <details><summary>Advanced generation settings</summary>\n    <label class="qa-check"><input type="checkbox" id="samplingEnabled"> Enable sampling (default: deterministic greedy generation)</label>\n    <div class="qa-control-grid">\n      <div><label for="temperature">Temperature</label><input id="temperature" type="number" min="0.1" max="2" step="0.1" value="1" disabled></div>\n      <div><label for="topP">Top-p</label><input id="topP" type="number" min="0.05" max="1" step="0.05" value="1" disabled></div>\n      <div><label for="generationSeed">Random seed</label><input id="generationSeed" type="number" min="0" max="2147483647" step="1" value="42"></div>\n    </div>\n    <p class="small">Temperature and top-p apply only when sampling is enabled. A shared seed controls sampling; hardware can still affect reproducibility. Sampling is not a factuality setting.</p>\n  </details>\n  <p id="controlScope" class="small" aria-live="polite"></p>\n  <p id="backendAvailability" class="small"></p>\n  <button id="restoreControls" type="button">Restore saved deployment settings</button>\n</div>\n'


**Interactive helper: `qa_controls_css`**

Responsive comparison cards and bounded controls styling.


In [240]:
def qa_controls_css():
    'Responsive comparison cards and bounded controls styling.'
    return '.qa-control-box{margin:18px 0;padding:18px;background:#f2f6fc;border:1px solid #cbd5e1;border-radius:14px}\n.qa-control-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;margin:12px 0}\n.qa-control-grid select,.qa-control-grid input{width:100%;box-sizing:border-box;min-height:40px}\n.qa-check{display:flex;align-items:center;gap:9px;font-weight:500;margin:12px 0}\n.qa-check input{width:auto;min-height:0;margin:0}\n.qa-control-box input:disabled{opacity:.5}.qa-control-box details{margin:12px 0}\n.qa-compare-cards{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px}\n.qa-compare-card{padding:20px;border:1px solid #cbd5e1;border-radius:15px;background:#fff;min-width:0}\n.qa-compare-card h3{margin-top:0}.qa-compare-answer{white-space:pre-wrap;overflow-wrap:anywhere;font-size:1.04rem;line-height:1.6}\n.qa-compare-card pre{white-space:pre-wrap;overflow-wrap:anywhere}.qa-matrix-wrap{overflow-x:auto}\n.qa-compare-card .source,.qa-compare-card blockquote{overflow-wrap:anywhere}\n.compare-active #qa>.grid{grid-template-columns:1fr}.compare-active #qa>.grid>aside{display:none}\n.compare-active #answer,.compare-active #sources,.compare-active #rawSection{display:none}\n.qa-settings-record{margin:12px 0;font-size:.82rem;color:#475569;overflow-wrap:anywhere}\n@media(max-width:1050px){.qa-compare-cards{grid-template-columns:1fr}}\n@media(max-width:600px){.qa-control-grid{grid-template-columns:1fr}.qa-control-box{padding:12px}}\n'


**Interactive helper: `qa_controls_script`**

Request-local options, backend comparisons and measured matrices.


In [241]:
def qa_controls_script():
    'Request-local options, backend comparisons and measured matrices.'
    return '// The saved deployment and its aggregate reports are never mutated by these controls.\nlet qaControlOptions=null;\nconst originalRenderAnswer=renderAnswer;\n$(\'ask\').disabled=true;\nfunction selectedControls(){return {backend:$(\'searchBackend\').value,k:Number($(\'retrievalK\').value),method:$(\'answerMethod\').value,\n max_new_tokens:Number($(\'answerTokens\').value),sampling:$(\'samplingEnabled\').checked,temperature:Number($(\'temperature\').value),\n top_p:Number($(\'topP\').value),seed:Number($(\'generationSeed\').value)};}\nfunction toggleSampling(){for(const id of [\'temperature\',\'topP\'])$(id).disabled=!$(\'samplingEnabled\').checked;}\nfunction toggleEvidence(){const show=$(\'showEvidence\').checked;\n if(!document.body.classList.contains(\'compare-active\')){\n  $(\'sources\').classList.toggle(\'hidden\',!show);$(\'rawSection\').classList.toggle(\'hidden\',!show||!lastAnswer);\n  $(\'calculations\').classList.toggle(\'hidden\',!show);\n }\n document.querySelectorAll(\'.qa-backend-evidence\').forEach(node=>node.classList.toggle(\'hidden\',!show));\n}\nfunction controlSettingsText(value){return \'Search: \'+value.backend+\' · retrieval k=\'+value.k+\' · \'+value.method+\n \' · max answer tokens=\'+value.max_new_tokens+\' · \'+(value.sampling?\'sampling; temperature=\'+value.temperature+\'; top-p=\'+value.top_p:\'deterministic; temperature/top-p inactive\')+\' · seed=\'+value.seed;}\nfunction restoreSavedControls(){if(!qaControlOptions)return;const d=qaControlOptions.defaults;\n $(\'searchBackend\').value=d.backend;if(!$(\'searchBackend\').selectedOptions[0]||$(\'searchBackend\').selectedOptions[0].disabled){\n  $(\'searchBackend\').value=qaControlOptions.backends.find(x=>x.available).value;}\n $(\'retrievalK\').value=String(d.k);$(\'answerMethod\').value=d.method;$(\'answerTokens\').value=String(d.max_new_tokens);\n $(\'samplingEnabled\').checked=false;$(\'temperature\').value=\'1\';$(\'topP\').value=\'1\';$(\'generationSeed\').value=String(d.seed);toggleSampling();}\nasync function loadControlOptions(){qaControlOptions=await api(\'/api/controls\');const o=qaControlOptions;\n $(\'searchBackend\').innerHTML=o.backends.map(x=>\'<option value="\'+esc(x.value)+\'"\'+(x.available?\'\':\' disabled\')+\'>\'+esc(x.label)+(x.available?\'\':\' — unavailable\')+\'</option>\').join(\'\');\n $(\'retrievalK\').innerHTML=o.k_values.map(x=>\'<option value="\'+x+\'">\'+x+\'</option>\').join(\'\');\n $(\'answerTokens\').innerHTML=o.token_limits.map(x=>\'<option value="\'+x+\'">\'+x+\'</option>\').join(\'\');\n $(\'controlScope\').textContent=\'Saved generator \'+o.model_version+\' · deployed shards \'+o.shards.join(\', \')+\' · prompt budget \'+o.input_cap+\' tokens. \'+o.note;\n $(\'backendAvailability\').textContent=o.backends.filter(x=>!x.available).map(x=>x.label+\': \'+x.reason).join(\' · \');\n restoreSavedControls();$(\'ask\').disabled=false;\n}\nfunction matrixTable(rows,aggregate=false){const columns=[\'Backend\',...(aggregate?[\'Requests\',\'Reference-scored\']:[]),\'EM\',\'F1\',\'Retrieval ms\',\'Generation ms\',\'Total ms\',\'Input tokens\',\'Output tokens\',\'Retrieved\',\'Used in prompt\',\'Abstained\',\'Lexical unsupported proxy\',\'Factual hallucination\'];\n return table(columns,rows.map(x=>[x.strategy,...(aggregate?[x.questions,x.reference_scored]:[]),pct(x.EM),pct(x.F1),\n number(x.retrieval_ms),number(x.generation_ms),number(x.total_ms),number(x.input_tokens,0),number(x.output_tokens,0),\n number(x.retrieved_passages,1),number(x.supplied_passages,1),aggregate?pct(x.abstained):String(x.abstained),\n pct(x.lexical_unsupported_proxy),\'Unmeasured\']));}\nfunction comparisonCard(o){const ref=o.reference_scores||{};return \'<article class="qa-compare-card"><div class="eyebrow">\'+esc(o.retrieval_strategy)+\'</div><h3>Saved-model answer</h3><p class="qa-compare-answer">\'+esc(o.answer)+\'</p>\'+\n \'<p class="small">EM \'+pct(ref.EM)+\' · F1 \'+pct(ref.F1)+\' · \'+number(o.total_ms)+\' ms</p>\'+\n \'<p class="small">Passages retrieved \'+o.retrieved_passages+\' / supplied \'+o.supplied_passages+\' · \'+esc(o.abstention_reason||\'\')+\'</p>\'+\n \'<div class="qa-backend-evidence"><details><summary>Source passages and distances</summary>\'+o.sources_html+\n table([\'Passage\',\'Article\',\'Shard\',\'Cosine distance\'],(o.retrieved||[]).map(x=>[x.passage_id,x.title,x.source_shard,number(x._distance,4)]))+\'</details>\'+\n \'<details><summary>Metric calculations</summary>\'+calculationHtml(o.calculations)+\'</details>\'+\n \'<details><summary>Raw model output / candidates</summary><pre>\'+esc(JSON.stringify({answer:o.raw_model_answer,intermediate:o.intermediate||[]},null,2))+\'</pre></details></div></article>\';}\nrenderAnswer=function(o){lastAnswer=o;document.body.classList.toggle(\'compare-active\',!!o.comparison);const workspace=$(\'liveComparisons\');\n if(!o.comparison){workspace.classList.add(\'hidden\');workspace.innerHTML=\'\';originalRenderAnswer(o);\n  $(\'metrics\').innerHTML+=\'<div class="qa-settings-record">\'+esc(controlSettingsText(o.controls))+\'</div>\'+\n   table([\'Additional measurement\',\'Value\'],[[\'Passages retrieved\',o.retrieved_passages],[\'Passages used in prompt\',o.supplied_passages],\n   [\'Query encoding ms\',number(o.query_encode_ms)],[\'Backend setup ms\',number(o.backend_setup_ms)],[\'Scope\',o.evaluation_scope]]);\n }else{workspace.classList.remove(\'hidden\');workspace.innerHTML=\'<div class="panel"><h2>Same question · three retrieval backends</h2><p class="subtle">\'+esc(o.note)+\'</p>\'+\n  \'<p class="qa-settings-record">\'+esc(controlSettingsText(o.controls))+\' · generator \'+esc(o.model_version)+\' · shards \'+esc(o.shards.join(\', \'))+\'</p>\'+\n  \'<div class="qa-compare-cards">\'+o.answers.map(comparisonCard).join(\'\')+\'</div></div>\'+\n  \'<div class="panel"><h2>Current question · metric matrix</h2><p class="small">Reference EM/F1 are unavailable without a supplied reference. These values do not establish factual accuracy.</p><div class="qa-matrix-wrap">\'+matrixTable(o.matrix)+\'</div></div>\'+\n  \'<div class="panel"><h2>Paired interactive history · mean metric matrix</h2><p class="small">\'+esc(o.summary.note)+\'</p><p class="small">Group \'+esc(o.summary.group)+\' · \'+o.summary.completed_requests+\' completed paired requests</p><div class="qa-matrix-wrap">\'+matrixTable(o.summary.rows,true)+\'</div></div>\';\n  $(\'answerJson\').disabled=false;$(\'answerCsv\').disabled=false;}\n toggleEvidence();\n};\n$(\'ask\').onclick=async()=>{const button=$(\'ask\');button.disabled=true;$(\'message\').textContent=\'\';\n $(\'activity\').textContent=$(\'searchBackend\').value===\'compare\'?\'Comparing LanceDB → Chroma → BM25 sequentially with one saved generator. This can take several times a single answer…\':\'Retrieving from the selected backend and generating locally…\';\n const began=performance.now();try{\n  const output=await api(\'/api/answer\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({question:$(\'question\').value,reference:$(\'reference\').value,sample_id:sampleId,controls:selectedControls()})});\n  renderAnswer(output);$(\'activity\').textContent=\'Completed in \'+((performance.now()-began)/1000).toFixed(1)+\' s. Exact settings and measurements saved locally.\';\n }catch(e){$(\'message\').textContent=e.message;$(\'activity\').textContent=\'Request failed. Missing indexes are never built by the Q&A page; use the notebook to finish conversion.\';}finally{button.disabled=false;}};\n$(\'answerJson\').onclick=()=>{if(lastAnswer)download(\'qa_\'+lastAnswer.answer_id+\'.json\',JSON.stringify(lastAnswer,null,2),\'application/json\');};\nfunction comparisonCsv(o){const answers=o.answers||[o];const keys=[\'question\',\'answer\',\'retrieval_strategy\',\'model_version\',\'release_id\',\'k\',\'method\',\'max_new_tokens\',\'sampling\',\'temperature\',\'top_p\',\'seed\',\'total_ms\',\'retrieval_ms\',\'generation_ms\',\'input_tokens\',\'output_tokens\',\'retrieved_passages\',\'supplied_passages\',\'abstained\',\'EM\',\'F1\',\'unsupported_sentence_proxy\',\'hallucination_index\'];\n const rows=answers.map(a=>({...a,...a.controls,retrieval_strategy:a.retrieval_strategy,EM:a.reference_scores?.EM,F1:a.reference_scores?.F1}));\n return keys.map(csvCell).join(\',\')+\'\\n\'+rows.map(row=>keys.map(k=>csvCell(row[k])).join(\',\')).join(\'\\n\')+\'\\n\';}\n$(\'answerCsv\').onclick=()=>{if(lastAnswer)download(\'qa_\'+lastAnswer.answer_id+\'.csv\',comparisonCsv(lastAnswer),\'text/csv\');};\n$(\'showEvidence\').onchange=toggleEvidence;$(\'samplingEnabled\').onchange=toggleSampling;$(\'restoreControls\').onclick=restoreSavedControls;\nloadControlOptions().catch(e=>{$(\'message\').textContent=\'Unable to load Q&A controls: \'+e.message;});\n'


In [242]:
def live_desktop_answer(payload,runtime):
    """Answer one selected backend or compare three sequentially, without changing the saved deployment."""
    started=time.perf_counter();question=str(payload.get('question','')).strip()
    if not question or len(question)>CFG['max_question_chars']:raise ValueError('Question is empty or too long.')
    settings=qa_parse_controls(payload,runtime);refs,kind=qa_request_references(payload,runtime,question)
    selected=settings['backend'];comparison_id=uuid.uuid4().hex
    strategies=['lance_dense','chroma_dense','bm25'] if selected=='compare' else [
        runtime.release.get('retrieval_strategy','lance_dense') if selected=='saved' else selected]
    began=time.perf_counter()
    vector=runtime.embedder.encode(question,normalize_embeddings=True,convert_to_numpy=True).astype(np.float32)
    encode_ms=(time.perf_counter()-began)*1000
    answers=[qa_controlled_result(question,runtime,strategy,settings,vector,encode_ms,refs,kind,comparison_id) for strategy in strategies]
    if selected!='compare':
        result=answers[0];result['request_ms']=(time.perf_counter()-started)*1000;return result
    rows=[qa_comparison_row(a) for a in answers]
    summary=qa_comparison_history(runtime,settings,question,rows,comparison_id,kind)
    return {'comparison':True,'answer_id':comparison_id,'question':question,'answers':answers,'matrix':rows,
        'summary':summary,'controls':settings,'release_id':runtime.release['id'],
        'model_version':runtime.release['selected']['reader_id'],'shards':[p['number'] for p in runtime.parts],
        'query_encode_ms':encode_ms,'request_ms':(time.perf_counter()-started)*1000,
        'note':'Three sequential answers use one saved generator. Query encoding is shared once; per-backend total includes its encode cost for standalone comparison. Reference scores are not factual accuracy.'}


**Helper: `desktop_http_handler`**

Local-only application with explicit export allowlists and no user-controlled filesystem paths.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [243]:
def desktop_http_handler(runtime,nonce):
    """Local-only application with explicit export allowlists and no user-controlled filesystem paths."""
    from http.server import BaseHTTPRequestHandler
    class Handler(BaseHTTPRequestHandler):
        def log_message(self,*args):pass
        def send(self,status,value,kind='application/json'):
            data=(json.dumps(report_json_safe(value),ensure_ascii=False,allow_nan=False) if kind=='application/json' else value).encode('utf-8')
            self.send_response(status);self.send_header('Content-Type',kind+'; charset=utf-8')
            self.send_header('Content-Length',str(len(data)));self.send_header('Cache-Control','no-store')
            self.send_header('X-Content-Type-Options','nosniff');self.end_headers();self.wfile.write(data)
        def send_file(self,path):
            import mimetypes
            self.send_response(200);self.send_header('Content-Type',mimetypes.guess_type(path.name)[0] or 'application/octet-stream')
            self.send_header('Content-Length',str(path.stat().st_size))
            self.send_header('Content-Disposition','attachment; filename="'+path.name+'"')
            self.send_header('X-Content-Type-Options','nosniff');self.end_headers()
            with open(path,'rb') as handle:shutil.copyfileobj(handle,self.wfile)
        def do_GET(self):
            try:desktop_api_get(self,runtime)
            except (BrokenPipeError,ConnectionResetError):pass
            except (ValueError,FileNotFoundError):self.send(400,{'error':'Invalid or unavailable saved report/export.'})
            except Exception as error:
                record_desktop_error('Q&A report read',error);self.send(500,{'error':'Unable to read saved reports. See captured error logs.'})
        def do_POST(self):desktop_api_post(self,runtime,nonce)
    return Handler


**Helper: `serve_desktop_qa`**

Load saved artifacts once in an independent local process so notebook completion leaves Q&A usable.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [244]:
def serve_desktop_qa():
    """Load saved artifacts once in an independent local process so notebook completion leaves Q&A usable."""
    from http.server import ThreadingHTTPServer
    handle=acquire_local_lock(ROOT/'desktop_qa.lock');server=None
    try:
        runtime=load_saved_deployment();runtime.query_lock=threading.Lock();nonce=secrets.token_urlsafe(32)
        handler=desktop_http_handler(runtime,nonce)
        try:server=ThreadingHTTPServer(('127.0.0.1',CFG['web_port']),handler)
        except OSError:server=ThreadingHTTPServer(('127.0.0.1',0),handler)
        atomic_json(ROOT/'qa_connection.json',{'url':'http://127.0.0.1:'+str(server.server_port),
            'release':runtime.release['id'],'project':LOCK['identity'],'token':nonce,'pid':os.getpid()})
        server.serve_forever(poll_interval=.25)
    finally:
        if server:server.server_close()
        release_local_lock(handle)


**Helper: `launch_desktop_qa`**

Open the real saved-model application in a separate browser tab, reusing a matching live local server.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [245]:
def launch_desktop_qa(release):
    """Open the real saved-model application in a separate browser tab, reusing a matching live local server."""
    existing=existing_qa_connection()
    if existing and existing['release']==release['id']:
        with urllib.request.urlopen(existing['url']+'/health',timeout=3) as response:health=json.load(response)
        if health.get('application')!=qa_application_version():stop_existing_qa();existing=None
    if existing and existing['release']==release['id']:
        if CFG['open_browser']:webbrowser.open(existing['url'],new=2)
        print('Saved-model Q&A:',existing['url']);return existing
    stop_existing_qa()
    connection=ROOT/'qa_connection.json'
    if connection.exists():connection.unlink()
    options={'creationflags':subprocess.CREATE_NEW_PROCESS_GROUP} if os.name=='nt' else {'start_new_session':True}
    env=os.environ.copy();env['PYTHONIOENCODING']='utf-8'
    with open(ROOT/'qa_server.log','a',encoding='utf-8') as log:
        process=subprocess.Popen([sys.executable,'-u',str(ROOT/'portable_rag.py'),'qa',str(ROOT)],
            cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,**options)
    with tqdm(total=None,desc='Load saved model and open browser',unit='s') as bar:
        deadline=time.perf_counter()+600
        while time.perf_counter()<deadline:
            if process.poll() is not None:raise RuntimeError('Local Q&A failed to start. Inspect qa_server.log.')
            saved=existing_qa_connection()
            if saved:
                if CFG['open_browser']:webbrowser.open(saved['url'],new=2)
                print('Saved-model Q&A:',saved['url']);return saved
            time.sleep(1);bar.update(1)
    raise TimeoutError('Q&A startup exceeded ten minutes. Inspect qa_server.log before launching another process.')


**Helper: `reference_score_details`**

Expose the exact normalization, multiset counts and best-alias rules used by the original EM/F1 scorer.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [246]:
def reference_score_details(answer,references):
    """Expose the exact normalization, multiset counts and best-alias rules used by the original EM/F1 scorer."""
    predicted=normalize_answer(answer);tokens=predicted.split();aliases=[]
    for reference in references:
        normalized=normalize_answer(reference);gold=normalized.split()
        counts=collections.Counter(tokens)&collections.Counter(gold);common=sum(counts.values())
        precision=common/len(tokens) if tokens else 0.;recall=common/len(gold) if gold else 0.
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        if not tokens or not gold:f1=float(tokens==gold)
        aliases.append({'reference':reference,'normalized_reference':normalized,'reference_tokens':gold,
            'common_token_counts':dict(counts),'common_tokens':common,'predicted_token_count':len(tokens),
            'reference_token_count':len(gold),'precision':precision,'recall':recall,
            'EM':float(predicted==normalized),'F1':f1})
    exact,f1=em_f1(answer,references) if references else (None,None)
    return {'normalization':'Unicode casefold; remove ASCII punctuation and English a/an/the; collapse whitespace.',
        'normalized_answer':predicted,'predicted_tokens':tokens,'aliases':aliases,'EM':exact,'F1':f1,
        'formula':'EM=max(exact normalized alias equality). P=common/predicted tokens; R=common/reference tokens; F1=max_alias(2PR/(P+R)). Empty-token cases follow the saved scorer.'}


**Helper: `answer_calculation_details`**

Reference overlap and lexical support are diagnostics, not calibrated confidence or factual hallucination.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [247]:
def answer_calculation_details(output,references):
    """Reference overlap and lexical support are diagnostics, not calibrated confidence or factual hallucination."""
    raw=output.get('raw_model_answer',output['answer']);evidence=output.get('evidence',[])
    source_words=set(normalize_answer(' '.join(p['text'] for p in evidence)).split());sentences=[]
    if raw and normalize_answer(raw)!=normalize_answer(ABSTAIN):
        for sentence in re.split(r'(?<=[.!?])\s+',raw):
            words=set(normalize_answer(sentence).split());common=sorted(words&source_words)
            ratio=len(common)/max(1,len(words))
            sentences.append({'sentence':sentence,'unique_tokens':len(words),'supported_unique_tokens':len(common),
                'common_tokens':common,'overlap':ratio,'flagged_below_0_8':ratio<.8})
    return {'reference':reference_score_details(output['answer'],references),
        'lexical_proxy':{'raw_answer':raw,'sentences':sentences,'threshold':.8,
            'reported_value':output.get('unsupported_sentence_proxy'),
            'formula':'Fraction of raw-answer sentences with unique-word overlap below 0.8 against supplied evidence. Unavailable for an empty/already-abstained raw answer.'},
        'retrieval':{'formula':'Cosine distance = 1 - cosine similarity; raw index distances are not probabilities.',
            'gate':CFG['max_cosine_distance'],'returned':len(output.get('retrieved',[])),
            'supplied':len(evidence),'require_answer_occurrence':CFG['require_answer_occurrence']},
        'hallucination_index':{'value':None,'reason':'No independent factual adjudication is available.'}}


**Helper: `report_validation_perplexity`**

Conditional, teacher-forced validation diagnostic: exp(saved eval_loss); no answer-quality interpretation.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [248]:
def report_validation_perplexity(loss):
    """Conditional, teacher-forced validation diagnostic: exp(saved eval_loss); no answer-quality interpretation."""
    if loss is None or not math.isfinite(float(loss)):return None
    try:return math.exp(float(loss))
    except OverflowError:return None


**Helper: `collect_training_report`**

Read saved logs/checkpoint history only; missing legacy measurements do not cause retraining.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [249]:
def collect_training_report(numbers):
    """Read saved logs/checkpoint history only; missing legacy measurements do not cause retraining."""
    rows=[];stages=[]
    for number in numbers:
        record=shard_record(number);relative=record.get('trained_model')
        stage={'shard':number,'status':record['training'],'model_version':record.get('model_version'),
            'note':record.get('training_reason',''),'history_source':None,'validation_note':'No logged validation loss available.'}
        if relative:
            folder=(ROOT/relative).parent;history=[]
            candidates=[folder/'training_history.json',folder/'training_telemetry.json']
            checkpoints=sorted(folder.glob('checkpoint-*/trainer_state.json'),
                key=lambda p:int(p.parent.name.split('-')[-1]),reverse=True)
            for path in candidates+checkpoints:
                if not path.exists():continue
                saved=read_json(path);history=saved if isinstance(saved,list) else saved.get('log_history',[])
                stage['history_source']=path.relative_to(ROOT).as_posix();break
            seen=set()
            for event in history:
                key=digest(report_json_safe(event))
                if key in seen:continue
                seen.add(key)
                row={**stage,'step':event.get('step'),'epoch':event.get('epoch'),
                    'train_loss':event.get('loss'),'validation_loss':event.get('eval_loss'),
                    'validation_perplexity':report_validation_perplexity(event.get('eval_loss')),
                    'learning_rate':event.get('learning_rate'),'grad_norm':event.get('grad_norm')}
                if any(row[k] is not None for k in ['train_loss','validation_loss']):rows.append(row)
            if any(r['shard']==number and r['validation_loss'] is not None for r in rows):
                stage['validation_note']='Conditional teacher-forced perplexity = exp(logged eval_loss); aggregation follows Trainer.'
        stages.append(stage)
    return report_json_safe(rows),stages


**Helper: `collect_desktop_logs`**

Return full contents of an explicit diagnostic-file allowlist, never a visitor-provided filesystem path.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [250]:
def collect_desktop_logs():
    """Return full contents of an explicit diagnostic-file allowlist, never a visitor-provided filesystem path."""
    names=['error_logs/full_errors.log','error_logs/events.jsonl','offline_verification.log',
        'offline_verification.stdout.log','offline_verification.stderr.log','qa_server.log']
    names += [p.relative_to(ROOT).as_posix() for p in (ROOT/'portable_jobs').glob('*/worker.log') if not (p.parent/'result.json').exists()]
    names += [p.relative_to(ROOT).as_posix() for p in (ROOT/'portable_verification').glob('*/execution.log')]
    logs=[]
    for name in names:
        path=ROOT/name
        if path.is_file():logs.append({'name':name,'bytes':path.stat().st_size,'text':path.read_text(encoding='utf-8',errors='replace')})
    summaries=[{'shard':n,'last_error':shard_record(n).get('last_error')} for n in range(1,len(LOCK['files'])+1)
        if shard_record(n).get('last_error')]
    return {'files':logs,'historical_summaries':summaries,
        'note':'Full tracebacks are captured by this version. Earlier last_error summaries cannot reconstruct missing historical logs.'}


**Helper: `prediction_error_row`**

Mutually exclusive diagnostic buckets; these are not a factual hallucination classification.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [251]:
def prediction_error_row(item):
    """Mutually exclusive diagnostic buckets; these are not a factual hallucination classification."""
    score=item['score'];output=item['output'];question=item['question']
    if score['EM']==1:category='Exact reference match'
    elif output.get('abstained'):category='Abstention'
    elif not score['article_hit']:category='Reference article not retrieved'
    elif not score['alias_in_evidence']:category='Reference alias absent from supplied evidence'
    elif score['F1']>0:category='Partial reference overlap'
    else:category='Zero reference token overlap'
    return {**score,'category':category,'question':question['question'],'expected_answers':question['answers'],
        'answer':output['answer'],'raw_model_answer':output.get('raw_model_answer'),
        'abstention_reason':output.get('abstention_reason'),
        'sources':[{'title':p['title'],'shard':p['source_shard'],'passage_id':p.get('passage_id')} for p in output.get('evidence',[])],
        'calculation':reference_score_details(output['answer'],question['answers'])}


**Helper: `manual_shard_questions`**

Use development QA only; shards without labels get source-excerpt inspection prompts, never synthetic gold answers.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [252]:
def manual_shard_questions(parts,per_shard):
    """Use development QA only; shards without labels get source-excerpt inspection prompts, never synthetic gold answers."""
    numbers=[p['number'] for p in parts];rows=all_matched_qa(numbers)
    selected=[]
    for part in parts:
        number=part['number'];dev=sorted([r for r in rows if r['shard']==number and r['split']=='dev'],key=lambda r:r['id'])
        for row in dev[:per_shard]:
            selected.append({**row,'sample_id':digest(['manual-dev',number,row['id']])[:24],
                'reference_kind':'matched_development_answer','reference_label':'Matched development reference; not an untouched final test.'})
        if dev:continue
        fields=['article_id','title','text','url','passage_id','source_shard']
        passages=part['table'].search().select(fields).limit(per_shard*5).to_list();seen=set()
        for passage in passages:
            if passage['article_id'] in seen:continue
            seen.add(passage['article_id'])
            selected.append({'id':passage['passage_id'],'sample_id':digest(['manual-source',number,passage['passage_id']])[:24],
                'question':f'What information does the article titled "{passage["title"]}" provide?',
                'answers':[],'context':passage['text'],'title':passage['title'],'url':passage['url'],
                'shard':number,'split':'source_inspection','reference_kind':'source_excerpt_only',
                'reference_label':'Source excerpt for human inspection; no gold answer or reference EM/F1 is claimed.'})
            if len(seen)>=per_shard:break
    return selected


**Helper: `manual_sample_answers`**

Persist each saved-reader sample after generation; exact winning development predictions are reused where available.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [253]:
def manual_sample_answers(release,parts,per_shard):
    """Persist each saved-reader sample after generation; exact winning development predictions are reused where available."""
    questions=manual_shard_questions(parts,per_shard);predictions=read_evaluation_cache(ROOT/release['report']/'predictions.jsonl')
    chosen=release['selected'];existing={r['question']['id']:r['output'] for r in predictions.values()
        if r['score']['reader']==chosen['reader'] and r['score']['method']==chosen['method'] and r['score']['k']==chosen['k']}
    runtime=None;answers=[];cache_folder=ROOT/'reporting'/release['id']/'sample_cache'
    try:
        with tqdm(total=len(questions),desc='Saved-model manual Q&A samples',unit='sample') as bar:
            for row in questions:
                key=digest([release,row['sample_id'],row['question'],'manual-samples-v1'])
                path=cache_folder/(key+'.json')
                if path.exists():sample=read_json(path)
                else:
                    sample=dict(row)
                    try:
                        if row['reference_kind']=='matched_development_answer' and row['id'] in existing:
                            output=existing[row['id']];origin='saved winning development prediction'
                        else:
                            if runtime is None:runtime=load_saved_deployment()
                            output=portable_answer(row['question'],runtime)
                            origin='saved final model with normal vector retrieval'
                        sample.update(output=output,prediction_origin=origin,
                            calculation=answer_calculation_details(output,row['answers']),status='complete')
                        atomic_json(path,sample)
                    except Exception as error:
                        record_desktop_error('manual sample',error,row['shard'])
                        sample.update(output=None,status='failed',error=type(error).__name__+': '+str(error))
                answers.append(sample);bar.update(1);guard_resources()
    finally:
        if runtime is not None:del runtime;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    return answers


**Helper: `plot_report_training`**

Plot actual logged points; omit unavailable curves and retain empty-series reasons in report data.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [254]:
def plot_report_training(folder,rows):
    """Plot actual logged points; omit unavailable curves and retain empty-series reasons in report data."""
    images=[]
    for kind,metrics,ylabel in [('loss',['train_loss','validation_loss'],'Logged cross-entropy loss'),
        ('perplexity',['validation_perplexity'],'Conditional validation perplexity = exp(eval_loss)')]:
        usable=[r for r in rows if any(r.get(m) is not None for m in metrics)]
        if not usable:continue
        fig,ax=plt.subplots(figsize=(10,4.5))
        for number in sorted(set(r['shard'] for r in usable)):
            for metric in metrics:
                data=[r for r in usable if r['shard']==number and r.get(metric) is not None and r.get('step') is not None]
                if not data:continue
                ax.plot([r['step'] for r in data],[r[metric] for r in data],
                    marker='o' if metric!='train_loss' else None,linewidth=1.6,label=f'Shard {number} · {metric}')
        ax.set(xlabel='Optimizer step within each incremental stage',ylabel=ylabel,title='Saved training / validation history')
        ax.grid(alpha=.2);ax.legend(fontsize=8,loc='best');fig.tight_layout()
        name='training_'+kind+'.png';fig.savefig(Path(folder)/name,dpi=150);plt.close(fig);images.append(name)
    return images


**Helper: `plot_report_errors`**

Count the saved selected configuration's diagnostic outcomes with explicit sample denominators.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [255]:
def plot_report_errors(folder,rows):
    """Count the saved selected configuration's diagnostic outcomes with explicit sample denominators."""
    if not rows:return []
    counts=collections.Counter(r['category'] for r in rows);fig,ax=plt.subplots(figsize=(10,4.5))
    names=list(counts);values=[counts[n] for n in names]
    ax.barh(names,values,color='#267f95');ax.invert_yaxis()
    for i,value in enumerate(values):ax.text(value+.03,i,str(value),va='center')
    ax.set(xlabel='Number of development answers',title=f'Answer diagnostics · selected configuration · n={len(rows)}')
    fig.tight_layout();name='answer_error_breakdown.png';fig.savefig(Path(folder)/name,dpi=150);plt.close(fig)
    return [name]


**Helper: `report_table_html`**

Render model/user values as text; tables never execute content from source passages or questions.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [256]:
def report_table_html(rows,columns):
    """Render model/user values as text; tables never execute content from source passages or questions."""
    head=''.join('<th>'+html.escape(label)+'</th>' for key,label in columns)
    body=[]
    for row in rows:
        values=[]
        for key,label in columns:
            value=row.get(key)
            if isinstance(value,(list,dict)):value=json.dumps(value,ensure_ascii=False)
            values.append('<td>'+html.escape('Unavailable' if value is None else str(value))+'</td>')
        body.append('<tr>'+''.join(values)+'</tr>')
    return '<div class="tablewrap"><table><thead><tr>'+head+'</tr></thead><tbody>'+''.join(body)+'</tbody></table></div>'


**Helper: `standalone_desktop_report`**

One portable HTML file embeds all figures, full logs and measured diagnostics; it does not run the model.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [257]:
def standalone_desktop_report(data,folder,images):
    """One portable HTML file embeds all figures, full logs and measured diagnostics; it does not run the model."""
    summary=report_table_html(data['comparison'],[(x,x) for x in ['reader','method','k','EM','F1','article_hit','article_MRR','total_ms']])
    retrieval=report_table_html(data.get('retrieval_comparison',{}).get('summary',[]),[(x,x) for x in ['strategy','k','questions','article_hit','article_MRR','ANN_recall_at_k','lookup_p50_ms','lookup_p95_ms','sampled_process_peak_RSS_MB']])
    history=report_table_html(data['training_history'],[(x,x) for x in ['shard','step','epoch','train_loss','validation_loss','validation_perplexity']])
    diagnostics=report_table_html(data['answer_diagnostics'],[(x,x) for x in ['reader','method','k','question_shard','question','expected_answers','answer','EM','F1','category']])
    samples=[]
    for sample in data['samples']:
        samples.append('<details><summary>Shard '+str(sample['shard'])+' · '+html.escape(sample['question'])+'</summary><p>'+html.escape(sample['reference_label'])+'</p>'
            '<p><b>Expected answers:</b> '+html.escape(' | '.join(sample['answers']) or 'No gold reference')+'</p>'
            '<p><b>Saved-model answer:</b> '+html.escape((sample.get('output') or {}).get('answer',sample.get('error','Unavailable')))+'</p>'
            '<blockquote>'+html.escape(sample['context'])+'</blockquote><pre>'+html.escape(json.dumps(sample.get('calculation'),ensure_ascii=False,indent=2))+'</pre></details>')
    figures=''.join('<figure><figcaption>'+html.escape(name)+'</figcaption><img alt="'+html.escape(name)+'" src="data:image/png;base64,'
        +base64.b64encode((Path(folder)/name).read_bytes()).decode('ascii')+'"></figure>' for name in images)
    logs=''.join('<details><summary>'+html.escape(row['name'])+'</summary><pre>'+html.escape(row['text'])+'</pre></details>' for row in data['logs']['files'])
    embedded=json.dumps(data,ensure_ascii=False,allow_nan=False).replace('<','\\u003c').replace('>','\\u003e').replace('&','\\u0026')
    return ('<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Wikipedia RAG · Saved run report</title><style>'+qa_report_css()+'</style><body><main class="wrap">'
        +'<header><p class="eyebrow">SAVED RUN REPORT · OFFLINE SNAPSHOT</p><h1>Wikipedia Evidence Lab</h1><p>'+html.escape(data['release_id'])+'</p></header>'
        +'<div class="notice">'+html.escape(data['evaluation_scope'])+' · Fractions in score tables use a 0–1 scale. Final held-out test and factual hallucination adjudication are pending.</div>'
        +'<section class="panel"><h2>Scope and limitations</h2><pre>'+html.escape(json.dumps(data['metadata'],ensure_ascii=False,indent=2))+'</pre><p>'+html.escape(data['perplexity_note'])+'</p></section>'
        +'<section class="panel"><h2>Retrieval strategy comparisons</h2>'+retrieval+'</section>'
        +'<section class="panel"><h2>EM / F1 and comparisons</h2>'+summary+'</section><section class="panel"><h2>Training / validation history</h2>'+history+'</section>'
        +'<section class="panel"><h2>Curves, matrices and error breakdown</h2>'+figures+'</section><section class="panel"><h2>Answer diagnostics</h2>'+diagnostics+'</section>'
        +'<section class="panel"><h2>Shard-specific manual Q&A</h2>'+''.join(samples)+'</section><section class="panel"><h2>Full captured error logs</h2><p>'+html.escape(data['logs']['note'])+'</p>'+logs+'</section>'
        +'<script type="application/json" id="report-data">'+embedded+'</script><footer>This portable report is a saved snapshot. New questions require the local Python Q&A server.</footer></main></body></html>')


**Helper: `build_desktop_reports`**

Add versioned report exports outside sealed corpus/model/assessment directories; saved compute is reused.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [258]:
def build_desktop_reports(release,selected_shard,precompute_samples=True):
    """Add versioned report exports outside sealed corpus/model/assessment directories; saved compute is reused."""
    parts=open_desktop_collection([r['shard'] for r in release['corpus']]);source=ROOT/release['report']
    report=read_json(source/'assessment.json');per_shard=int(CFG.get('manual_samples_per_shard',3))
    if not 1<=per_shard<=20:raise ValueError('manual_samples_per_shard must be between 1 and 20.')
    samples=manual_sample_answers(release,parts,per_shard) if precompute_samples else []
    history,stages=collect_training_report([p['number'] for p in parts]);logs=collect_desktop_logs()
    predictions=list(read_evaluation_cache(source/'predictions.jsonl').values())
    diagnostics=[prediction_error_row(row) for row in predictions];chosen=release['selected']
    selected=[r for r in diagnostics if all(r[k]==chosen[k] for k in ['reader','method','k'])]
    data=report_json_safe({'release_id':release['id'],'evaluation_scope':report['evaluation_scope'],
        'metadata':{'selected_shard':selected_shard,'coverage':[p['number'] for p in parts],
            'selected_reader':chosen,'question_count':report['question_count'],'sample_cap_per_shard':per_shard,
            'reserved_test_exposed':False,'factual_hallucination_index':None,'limitations':report['limitations']},
        'retrieval_comparison':report.get('retrieval_comparison',{}),'retrieval_strategy':release.get('retrieval_strategy','lance_dense'),
        'comparison':report['summary'],'training_history':history,'training_stages':stages,
        'answer_diagnostics':diagnostics,'samples':samples,'logs':logs,
        'offline_verification':read_json(source/'offline_verification.json') if (source/'offline_verification.json').exists() else None,
        'perplexity_note':'Conditional teacher-forced validation perplexity = exp(saved eval_loss). It follows Trainer loss aggregation, is not free-text factual accuracy, and is unavailable when validation loss was not saved.'})
    identity=digest([data,'run-reports-v1',inspect.getsource(standalone_desktop_report)])[:24]
    folder=ROOT/'reporting'/release['id']/identity;receipt=folder/'manifest.json'
    if not receipt.exists():
        folder.mkdir(parents=True,exist_ok=True);data['created_at']=datetime.now(timezone.utc).isoformat()
        atomic_json(folder/'report.json',data);write_report_csv(folder/'comparison.csv',report['summary'],['reader','method','k','EM','F1'])
        write_report_csv(folder/'training_history.csv',history,['shard','step','epoch','train_loss','validation_loss','validation_perplexity'])
        write_report_csv(folder/'answer_diagnostics.csv',diagnostics,['question','answer','EM','F1','category'])
        write_report_csv(folder/'answer_errors.csv',[r for r in diagnostics if r['category']!='Exact reference match'],['question','category'])
        atomic_json(folder/'samples.json',samples);write_report_csv(folder/'samples.csv',samples,['shard','question','answers'])
        atomic_json(folder/'full_logs.json',logs);(folder/'full_errors.log').write_text('\n\n'.join(r['name']+'\n'+r['text'] for r in logs['files']),encoding='utf-8')
        images=plot_report_training(folder,history)+plot_report_errors(folder,selected)
        for name in report['figures']:
            if Path(name).name!=name or not name.endswith('.png'):raise ValueError('Invalid saved matrix name.')
            shutil.copy2(source/name,folder/name);images.append(name)
        for path in source.glob('matrix_*.csv'):shutil.copy2(path,folder/path.name)
        (folder/'report.html').write_text(standalone_desktop_report(data,folder,images),encoding='utf-8')
        atomic_json(receipt,{'release_id':release['id'],'identity':identity,'images':images,
            'folder':folder.relative_to(ROOT).as_posix(),'files':file_inventory(folder,{'manifest.json'})})
    saved=read_json(receipt);check_inventory(folder,saved['files'],True)
    atomic_json(ROOT/'reporting'/release['id']/'latest.json',saved)
    print('Saved CSV / JSON / PNG / standalone HTML:',folder,flush=True)
    return saved


**Helper: `available_desktop_reports`**

Resolve reports through saved pointers only; the active Q&A model stays fixed when viewing an older run.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [259]:
def available_desktop_reports():
    """Resolve reports through saved pointers only; the active Q&A model stays fixed when viewing an older run."""
    rows=[]
    for path in sorted((ROOT/'reporting').glob('*/latest.json')):
        row=read_json(path)
        rows.append({'release_id':row['release_id'],'identity':row['identity']})
    return rows


**Helper: `resolve_desktop_report`**

Return a verified export allowlist; user input is never joined directly into a filesystem path.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [260]:
def resolve_desktop_report(release_id):
    """Return a verified export allowlist; user input is never joined directly into a filesystem path."""
    available={r['release_id'] for r in available_desktop_reports()}
    if release_id not in available:raise ValueError('No saved report is available for this release.')
    pointer=read_json(ROOT/'reporting'/release_id/'latest.json');folder=(ROOT/pointer['folder']).resolve()
    if folder.parent.parent!= (ROOT/'reporting').resolve() or folder.parent.name!=release_id:raise ValueError('Invalid report location.')
    check_inventory(folder,pointer['files'],False)
    return pointer,folder


**Helper: `qa_samples_payload`**

All displayed samples belong to the deployed vector scope; reserved test questions are never served.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [261]:
def qa_samples_payload(runtime):
    """All displayed samples belong to the deployed vector scope; reserved test questions are never served."""
    pointer,folder=resolve_desktop_report(runtime.release['id'])
    data=read_json(folder/'report.json')
    return {'samples':data['samples'],'selected_shard':data['metadata']['selected_shard'],
        'coverage':data['metadata']['coverage'],'per_shard_limit':data['metadata']['sample_cap_per_shard']}


**Helper: `persist_desktop_answer`**

Write question-specific diagnostics without modifying frozen run reports or feeding visitor answers into training.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [262]:
def persist_desktop_answer(output):
    """Write question-specific diagnostics without modifying frozen run reports or feeding visitor answers into training."""
    folder=ROOT/'qa_sessions'/output['release_id'];append_report_jsonl(folder/'answers.jsonl',output)
    row={k:output.get(k) for k in ['answer_id','at','question','answer','model_version','total_ms','retrieval_ms','generation_ms','input_tokens','output_tokens','model_calls','abstained']}
    row.update(EM=(output.get('reference_scores') or {}).get('EM'),F1=(output.get('reference_scores') or {}).get('F1'))
    path=folder/'answers.csv';new=not path.exists();import csv
    with open(path,'a',newline='',encoding='utf-8') as handle:
        writer=csv.DictWriter(handle,fieldnames=list(row))
        if new:writer.writeheader()
        writer.writerow(row)


**Helper: `qa_application_version`**

Restart a same-release server when the application/report code changed, instead of silently serving its old UI.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [263]:
def qa_application_version():
    """Restart an old same-release application after interactive code changes; preserve the saved model/reports."""
    names=['desktop_http_handler','desktop_api_get','desktop_api_post','live_desktop_answer','qa_backend_status',
        'qa_control_options','qa_parse_controls','qa_generate_controlled','qa_answer_controlled',
        'qa_request_references','qa_controlled_result','qa_comparison_row','qa_comparison_history']
    return digest(desktop_html()+''.join(inspect.getsource(globals()[name]) for name in names)+'interactive-controls-v1')[:24]


**Helper: `desktop_api_get`**

Serve report exports through exact saved filenames; allow no arbitrary file reads or cross-scope sample browsing.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [264]:
def desktop_api_get(handler,runtime):
    """Serve report exports through exact saved filenames; allow no arbitrary file reads or cross-scope sample browsing."""
    parsed=urllib.parse.urlsplit(handler.path);query=urllib.parse.parse_qs(parsed.query)
    release_id=query.get('release',[runtime.release['id']])[0]
    if parsed.path=='/api/controls':return handler.send(200,qa_control_options(runtime))
    if parsed.path=='/api/portable':return handler.send(200,portable_ui_payload(runtime))
    if parsed.path=='/':return handler.send(200,desktop_html(),'text/html')
    if parsed.path=='/health':return handler.send(200,{'release':runtime.release['id'],'project':LOCK['identity'],'application':qa_application_version()})
    if parsed.path=='/api/status':return handler.send(200,qa_status_payload(runtime))
    if parsed.path=='/api/report':return handler.send(200,qa_report_payload(runtime,release_id))
    if parsed.path=='/api/samples':return handler.send(200,qa_samples_payload(runtime))
    if parsed.path=='/api/logs':return handler.send(200,collect_desktop_logs())
    if parsed.path in {'/api/errors','/api/export'}:
        pointer,folder=resolve_desktop_report(release_id)
        if parsed.path=='/api/errors':
            data=read_json(folder/'report.json');rows=data['answer_diagnostics']
            start=max(0,int(query.get('offset',['0'])[0]));count=max(1,min(100,int(query.get('limit',['20'])[0])))
            return handler.send(200,{'rows':rows[start:start+count],'total':len(rows),'offset':start})
        name=query.get('file',[''])[0]
        if name not in pointer['files'] or Path(name).name!=name:return handler.send(404,{'error':'Export not found.'})
        path=folder/name
        if file_sha(path)!=pointer['files'][name]['sha256']:raise ValueError('Saved export checksum changed.')
        return handler.send_file(path)
    return handler.send(404,{'error':'Not found'})


**Helper: `desktop_api_post`**

Keep same-origin checks, bounded requests and a single model writer; error traces are retained in full.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [265]:
def desktop_api_post(handler,runtime,nonce):
    """Keep same-origin checks, bounded requests and a single model writer; error traces are retained in full."""
    origin=handler.headers.get('Origin');expected='http://127.0.0.1:'+str(handler.server.server_port)
    if origin and origin!=expected:return handler.send(403,{'error':'Cross-origin requests are not allowed.'})
    if handler.headers.get_content_type()!='application/json':return handler.send(415,{'error':'Use application/json.'})
    try:
        length=int(handler.headers.get('Content-Length','0'))
        if not 0<length<=16384:return handler.send(413,{'error':'Request is empty or too large.'})
        payload=json.loads(handler.rfile.read(length))
        if not isinstance(payload,dict):raise ValueError('Expected a JSON object.')
        if handler.path=='/api/shutdown':
            if not secrets.compare_digest(str(payload.get('token','')),nonce):return handler.send(403,{'error':'Invalid server token.'})
            handler.send(200,{'stopping':True});threading.Thread(target=handler.server.shutdown,daemon=True).start();return
        if handler.path!='/api/answer':return handler.send(404,{'error':'Not found'})
        if not runtime.query_lock.acquire(blocking=False):return handler.send(409,{'error':'The local model is busy; retry after the current answer.'})
        try:return handler.send(200,live_desktop_answer(payload,runtime))
        finally:runtime.query_lock.release()
    except (ValueError,json.JSONDecodeError):handler.send(400,{'error':'Invalid question, reference, sample selection or request body.'})
    except Exception as error:
        record_desktop_error('Q&A inference',error);handler.send(500,{'error':'Local inference failed. See Full error logs in Run reports.'})


**Helper: `qa_report_css`**

Embedded offline UI asset; no CDN or external font requests.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [266]:
def qa_report_css():
    """Embedded offline UI asset; no CDN or external font requests."""
    return ':root{--nav:#12283b;--ink:#183346;--muted:#60778a;--brand:#087f86;--line:#dbe6ed;--bg:#f3f7fa;--soft:#e9f5f4}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.65 system-ui,"Segoe UI",sans-serif}.wrap{max-width:1390px;margin:auto;padding:28px 26px}header{padding:28px 32px;background:linear-gradient(115deg,#142b42,#09656e);border-radius:20px;color:white;box-shadow:0 12px 32px #10344717}h1{font-size:32px;letter-spacing:-.8px;margin:4px 0 10px}header p{margin:6px 0;color:#dcecf1}.eyebrow{font-weight:700;font-size:11px;letter-spacing:2px;color:#93e2da;text-transform:uppercase}.badge{display:inline-block;border:1px solid #79b0b7;border-radius:20px;padding:3px 11px;font-size:12px;margin:6px 6px 0 0}.scope{font-size:13px;word-break:break-word}.stats{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:20px 0}.stat,.panel{background:white;border:1px solid var(--line);border-radius:15px;padding:22px}.stat small{display:block;color:var(--muted)}.stat strong{display:block;font-size:28px;color:var(--brand);line-height:1.5}.stat strong.version{font-size:13px;margin-top:8px;word-break:break-all}.tabs{display:flex;gap:8px;flex-wrap:wrap;margin:20px 0}.tab,button,.button{border:0;border-radius:9px;padding:11px 18px;font:600 13px system-ui;cursor:pointer;color:var(--ink);background:#e5eef4;text-decoration:none;display:inline-block}.tab.active,button.primary,.button.primary{background:var(--brand);color:white}button:disabled{opacity:.55;cursor:wait}.hidden{display:none!important}.grid{display:grid;grid-template-columns:minmax(0,1.65fr) minmax(280px,1fr);gap:18px}.panel{margin-bottom:18px;min-width:0}h2{font-size:21px;line-height:1.35;margin:0 0 12px}h3{font-size:16px;margin:20px 0 10px}p{margin:8px 0 14px}.subtle,.small{color:var(--muted);font-size:13px}label{display:block;font-weight:650;margin:12px 0 6px}textarea,input,select{max-width:100%;width:100%;background:#fbfdfe;border:1px solid #c8d9e5;border-radius:9px;padding:12px;color:var(--ink);font:14px system-ui}textarea{min-height:104px;resize:vertical}input:focus,textarea:focus,select:focus{outline:2px solid #93d1cf;outline-offset:1px}.toolbar{display:flex;align-items:center;gap:10px;flex-wrap:wrap;margin:12px 0}.toolbar select{width:auto;min-width:200px}.notice{padding:13px 16px;background:#fff7e4;border-left:4px solid #d3a343;border-radius:7px;font-size:13px;margin:14px 0}.success{background:#e9f6ef;color:#245c47}.answer{font-size:22px;line-height:1.7;white-space:pre-wrap;padding:10px 0 18px;min-height:50px}.tablewrap{overflow:auto;max-height:620px;border:1px solid var(--line);border-radius:9px}table{width:100%;border-collapse:collapse;font-size:12px}th,td{text-align:left;padding:10px 12px;border-bottom:1px solid var(--line);vertical-align:top;min-width:75px}th{background:#eef4f8;color:#426077;white-space:nowrap;position:sticky;top:0}tr:last-child td{border-bottom:0}td{overflow-wrap:anywhere}details{border:1px solid var(--line);border-radius:9px;padding:13px;margin:10px 0}summary{font-weight:650;cursor:pointer}blockquote{white-space:pre-wrap;border-left:3px solid #9ccaca;padding:0 14px;margin:14px 0;color:#486173}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f4f7fa;padding:14px;border-radius:8px;font:12px/1.65 ui-monospace,Consolas,monospace;max-height:650px;overflow:auto}a{color:#087c88}figure{margin:18px 0}figure img{width:100%;border:1px solid var(--line);border-radius:10px}figcaption{font-size:12px;color:var(--muted);margin:6px 0}.samples{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px}.sample{border:1px solid var(--line);border-radius:12px;padding:18px;background:white}.sample h3{margin:7px 0 12px}.sample .tag{font-size:10px;text-transform:uppercase;letter-spacing:1px;color:var(--brand);font-weight:700}.error{color:#b02e38}.exports{display:flex;gap:8px;flex-wrap:wrap}.exports a{background:#edf5f8;border-radius:6px;padding:7px 10px;font-size:12px;text-decoration:none}.calculation{font-size:12px}.pill{display:inline-block;padding:2px 7px;border-radius:20px;background:var(--soft);color:#166b68;font-size:11px}footer{font-size:12px;color:#758897;margin:26px 0 8px}.metric-note{border-top:1px solid var(--line);margin-top:14px;padding-top:12px}.report-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:18px}.report-grid figure{margin-top:0}.report-grid figure.matrix{grid-column:1/-1}.no-data{padding:20px;border:1px dashed #c6d7e1;border-radius:9px;color:var(--muted)}.logbox pre{max-height:500px}.screen-reader{position:absolute;width:1px;height:1px;overflow:hidden;clip:rect(0,0,0,0)}\n@media(max-width:900px){.wrap{padding:18px 14px}.grid{grid-template-columns:1fr}.stats{grid-template-columns:repeat(2,1fr)}.samples,.report-grid{grid-template-columns:1fr}header{padding:24px}h1{font-size:28px}.toolbar select{width:100%}.panel{padding:17px}}@media print{body{background:white}.wrap{padding:0}.tabs,.toolbar{display:none}header{color:#183346;background:white;box-shadow:none;border:1px solid #ccc}header p{color:#486173}details{break-inside:avoid}.tablewrap,pre{max-height:none;overflow:visible}figure{break-inside:avoid}}\n'


**Helper: `qa_markup_template`**

Embedded offline UI asset; no CDN or external font requests.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [267]:
def qa_markup_template():
    """Embedded offline UI asset; no CDN or external font requests."""
    return '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Wikipedia Evidence Lab · Run workspace</title><style>__STYLE__</style></head>\n<body><main class="wrap"><header><div class="eyebrow">LOCAL WIKIPEDIA RAG · SAVED FINAL MODEL</div><h1>Wikipedia Evidence Lab</h1><p>A workspace for grounded answers, reproducible measurements and manual source verification.</p><span class="badge">Offline inference</span><span class="badge">Development preview</span><span class="badge">Sources included</span><p class="scope" id="scope">Loading saved deployment…</p></header>\n<div class="stats"><div class="stat"><small>Searchable shards</small><strong id="indexed">—</strong></div><div class="stat"><small>Pending indexing</small><strong id="pending">—</strong></div><div class="stat"><small>Confirmed notebook shard</small><strong id="selected">—</strong></div><div class="stat"><small>Saved generator version</small><strong id="version" class="version">—</strong></div></div>\n<nav class="tabs" aria-label="Workspace sections"><button class="tab active" data-page="qa">Main Q&A</button><button class="tab" data-page="reports">Run reports</button><button class="tab" data-page="samples">Shard sample Q&A</button><button class="tab" data-page="shards">All 41 shards</button></nav>\n<section class="page" id="qa"><div class="grid"><div class="panel"><h2>Ask the saved final model</h2><p class="subtle">Search the frozen vector collection and generate locally. Each result below belongs to the current question.</p><label for="question">Question</label><textarea id="question" maxlength="800" placeholder="Enter a question supported by your indexed Wikipedia shards…"></textarea><label for="reference">Optional reference answer</label><input id="reference" maxlength="4000" placeholder="Used after generation for reference EM/F1; never sent in the model prompt"><p id="sampleContext" class="small"></p><div class="toolbar"><button id="ask" class="primary">Search & answer</button><button id="answerJson" disabled>Export answer JSON</button><button id="answerCsv" disabled>Export answer CSV</button></div><p id="activity" class="subtle" aria-live="polite"></p><p id="message" class="error" role="alert"></p><div id="answer" class="answer"></div><div id="sources"></div><details id="rawSection" class="hidden"><summary>Raw model output and intermediate candidates</summary><pre id="rawAnswer"></pre></details></div>\n<aside><div class="panel"><h2>This question · execution metrics</h2><div id="metrics" class="no-data">Ask a question to display its measured timings, tokens, source coverage and reference scores.</div><p class="small metric-note">Distances are retrieval diagnostics. No calibrated confidence or independently verified factual hallucination index is available.</p></div><div class="panel"><h2>Expandable score calculations</h2><div id="calculations" class="no-data">Calculation details appear after an answer.</div></div></aside></div></section>\n<section class="page hidden" id="reports"><div class="panel"><h2>Saved run reports</h2><div class="toolbar"><label for="reportRun" class="screen-reader">Report release</label><select id="reportRun"></select><button id="refreshReports">Refresh reports & logs</button></div><div class="notice" id="reportScope"></div><div id="reportMetrics" class="stats"></div><p id="reportMeta" class="subtle"></p><h3>Saved exports</h3><div id="exports" class="exports"></div><p class="small">CSV scores use fractions (0–1). The standalone HTML embeds figures, samples, diagnostics and logs; new inference still requires the local Python server.</p></div>\n<div class="panel"><h2>Training / validation curves and perplexity</h2><p id="perplexityNote" class="subtle"></p><div id="trainingFigures" class="report-grid"></div><details><summary>Stage status and metric availability</summary><div id="trainingStages"></div></details><details><summary>All logged loss / validation points</summary><div id="trainingHistory"></div></details></div>\n<div class="panel"><h2>EM/F1 and reader comparisons</h2><div id="comparison"></div><div id="matrixFigures" class="report-grid"></div></div>\n<div class="panel"><h2>Answer-error breakdowns</h2><p class="subtle">Diagnostic categories explain reference mismatch patterns. They do not establish factual hallucination. Charts use the selected configuration; the table includes the saved comparison grid.</p><div id="errorFigures"></div><div id="errorRows"></div><div class="toolbar"><button id="errorsPrevious">Previous</button><button id="errorsNext">Next</button><span id="errorPage" class="small"></span></div></div>\n<div class="panel"><h2>Full error logs</h2><p id="logNote" class="subtle"></p><div id="logs" class="logbox"></div><details><summary>Historical shard error summaries</summary><div id="historicalErrors"></div></details></div></section>\n<section class="page hidden" id="samples"><div class="panel"><h2>Shard-specific samples for manual verification</h2><p class="subtle">Inspect the saved-model answer against its reference and source text. Development labels are shown only for manual preview; reserved final-test questions remain hidden.</p><div class="toolbar"><label for="sampleShard" class="screen-reader">Sample shard</label><select id="sampleShard"></select><span id="sampleLimit" class="small"></span></div><div class="notice">When a shard has no matched development labels, source-inspection questions include an excerpt for human review. Those excerpts are not gold answers and receive no reference EM/F1.</div></div><div id="sampleCards" class="samples"></div></section>\n<section class="page hidden" id="shards"><div class="panel"><h2>Saved progress · all 41 shards</h2><p class="subtle">The deployed vector scope is separate from notebook processing status. Skipped fine-tuning includes an explicit reason.</p><div id="shardTable"></div><p id="statusStamp" class="small"></p></div></section>\n<footer>Runs use local saved weights and vector partitions. Metrics are measured diagnostics, not a guarantee of correctness. Keep the local backend running while asking new questions.</footer></main><script>__SCRIPT__</script></body></html>\n'


**Helper: `portable_status`**

Display source and derived-index status separately, including all pending shards rather than only deployed ones.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [268]:
def portable_status():
    """Display source and derived-index status separately, including all pending shards rather than only deployed ones."""
    rows=desktop_status()
    for row in rows:
        record=shard_record(row['shard']);row['chroma']='pending';row['bm25']='pending'
        if record.get('indexing')!='complete':continue
        key=portable_part_key({'number':row['shard'],'record':record})
        receipts=list((ROOT/'derived'/'receipts').glob('wiki_'+str(row['shard']).zfill(3)+'_*.json'))
        for marker in receipts:
            receipt=read_json(marker)
            if receipt['spec']['source']==key:row['chroma']=receipt['status']
        path=ROOT/'derived'/'lexical'/(key+'.sqlite3')
        if path.exists():
            import sqlite3
            with sqlite3.connect(path.as_uri()+'?mode=ro',uri=True) as db:
                row['bm25']=dict(db.execute('SELECT key,value FROM state')).get('status','in progress')
    return rows


**Helper: `portable_ui_payload`**

Expose comparisons, scope, verification and honest negative-probe diagnostics without serving arbitrary paths.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [269]:
def portable_ui_payload(runtime):
    """Expose comparisons, scope, verification and honest negative-probe diagnostics without serving arbitrary paths."""
    path=ROOT/'portable_diagnostics'/runtime.release['id']/'negative_probes.json'
    proof=ROOT/'portable_verification_latest.json'
    return {'release':runtime.release['id'],'strategy':runtime.release.get('retrieval_strategy','lance_dense'),
        'retrieval':runtime.report.get('retrieval_comparison',{}),'regression':runtime.report.get('per_shard',[]),
        'negative_probes':read_json(path) if path.exists() else {'note':'No negative-probe run is saved for this release.'},
        'verification':read_json(proof) if proof.exists() else None,'shards':portable_status(),
        'final_test':read_json(ROOT/'final_test'/'result.json') if (ROOT/'final_test'/'result.json').exists() else None,
        'note':'Development selection, not a final held-out accuracy result. Reference overlap and lexical proxies do not establish factual correctness.'}


**Helper: `portable_markup`**

Extend the supplied professional report workspace with reproducible strategy comparisons and migration status.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [270]:
def portable_markup():
    """Extend the supplied professional report workspace with reproducible strategy comparisons and migration status."""
    markup=qa_markup_template()
    markup=markup.replace('</nav>','<button class="tab" data-page="strategies">Retrieval comparisons</button></nav>',1)
    section='''<section class="page hidden" id="strategies"><div class="panel">
<h2>Retrieval, migration and validation</h2><p id="portableScope" class="subtle">Loading saved comparison…</p>
<h3>Same-vector retrieval comparison</h3><div id="portableRetrieval"></div>
<p class="small">Article hit/MRR use matched development labels. ANN recall measures agreement with exact dense neighbours for a declared query subset. Warm lookup excludes query encoding and generation. Memory is sampled total process RSS in isolated workers.</p>
<h3>Earlier-shard reader comparison</h3><div id="portableRegression"></div>
<h3>Observed index/copy costs</h3><div id="portableIndexCosts"></div>
<h3>Unsupported-question stress probes</h3><div id="portableNegatives"></div>
<h3>Explicit frozen final test</h3><pre id="portableFinalTest"></pre>
<p class="small">Synthetic probes are not human-adjudicated corpus-negative labels. Factual hallucination remains unmeasured.</p>
<h3>All-shard migration status</h3><div id="portableShards"></div>
<details><summary>Saved execution-verification details</summary><pre id="portableProof"></pre></details>
</div></section>'''
    return markup.replace('</main>',section+'</main>')


**Helper: `portable_ui_script`**

Render new measured tables using the existing escaped DOM helpers; no CDN or external fonts.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [271]:
def portable_ui_script():
    """Render new measured tables using the existing escaped DOM helpers; no CDN or external fonts."""
    return '''
async function portableComparisons(){
 const p=await api('/api/portable');$('portableScope').textContent='Saved retrieval: '+p.strategy+' · '+p.note;
 const r=p.retrieval.summary||[];
 $('portableRetrieval').innerHTML=table(['Strategy','k','Questions','Article hit','Article MRR','Gold passage hit','ANN recall','Warm p50 ms','Warm p95 ms','Peak RSS MB'],r.map(x=>[x.strategy,x.k,x.questions,pct(x.article_hit),number(x.article_MRR,3),pct(x.gold_passage_hit),pct(x.ANN_recall_at_k),number(x.lookup_p50_ms),number(x.lookup_p95_ms),number(x.sampled_process_peak_RSS_MB)]));
 $('portableRegression').innerHTML=table(['Reader','Shard','Answers','EM','F1'],(p.regression||[]).map(x=>[x.reader,x.shard,x.answer_count,pct(x.EM),pct(x.F1)]));
 $('portableIndexCosts').innerHTML=table(['Shard','Passages','Lance bytes','New Lance build s','Chroma copy/verify/ingest s','Note'],(p.retrieval.index_costs?.rows||[]).map(x=>[x.shard,x.passages,x.lance_partition_bytes,number(x.lance_new_index_build_seconds),number(x.chroma_copy_verify_ingest_seconds),x.note]));
 $('portableNegatives').innerHTML=table(['Question','Answer','Abstained','Label scope'],(p.negative_probes.rows||[]).map(x=>[x.question,x.answer,x.abstained,x.label_scope]));
 $('portableShards').innerHTML=table(['Shard','Embedding','Lance index','Chroma HNSW copy','BM25','Training','Passages'],p.shards.map(x=>[x.shard,x.embedding,x.indexing,x.chroma,x.bm25,x.training,x.passages]));
 $('portableProof').textContent=JSON.stringify(p.verification,null,2);
 $('portableFinalTest').textContent=p.final_test?JSON.stringify(p.final_test,null,2):'Not executed. Reserved-test labels remain unopened by this workflow.';
}
portableComparisons().catch(e=>{$('portableScope').textContent=e.message;});
'''


**Helper: `portable_report_module`**

Verify local execution, reuse existing reports, and keep all new diagnostics outside sealed assessment files.

**Why here:** Execution, reference overlap, lexical heuristics and human factual validation are different checks and must not be conflated.


In [272]:
def portable_report_module():
    """Verify local execution, reuse existing reports, and keep all new diagnostics outside sealed assessment files."""
    proof=portable_offline_verify();release=PORTABLE_RUN['release']
    atomic_json(ROOT/'portable_verification_latest.json',proof)
    existing=ROOT/'reporting'/release['id']/'latest.json'
    if PORTABLE_RUN['mode'] in {'validate','export_validation','export_resume','final_test'} and existing.exists():
        report=read_json(existing);check_inventory(ROOT/report['folder'],report['files'],True)
    else:
        report=build_desktop_reports(release,PORTABLE_RUN['number'],precompute_samples=PORTABLE_RUN['mode'] in {'process','compare'})
    PORTABLE_RUN['report_exports']=report
    if PORTABLE_RUN['mode']=='process':save_shard_record(PORTABLE_RUN['number'],evaluation='complete',portable_release=release['id'])
    return {'verification':proof,'report':str(ROOT/report['folder']/'report.html')}


**Execute Module 9.** Verify the saved pipeline, then reuse/build versioned reports without changing sealed vector/model files.


In [273]:
VERIFICATION_STATUS=portable_run_module('verification_reports',portable_report_module,requires=['saved_release'])
print(VERIFICATION_STATUS['report'])
display(FileLink(VERIFICATION_STATUS['report']))


verification_reports · starting


Saved-model manual Q&A samples:   0%|          | 0/12 [00:00<?, ?sample/s]

C:\Users\Anirban\AppData\Local\Temp\ipykernel_12024\144515999.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if model.get_sentence_embedding_dimension()!=384:raise ValueError('Expected 384-dimensional MiniLM embeddings.')
Token indices sequence length is longer than the specified maximum sequence length for this model (577 > 512). Running this sequence through the model will result in indexing errors


Saved CSV / JSON / PNG / standalone HTML: C:\Users\Anirban\OfflineWikipediaRAG_Desktop\reporting\622880b84231acbd2e5a8691\5e8a8e5757620b4a92c46837
verification_reports · complete · 181.3 s
C:\Users\Anirban\OfflineWikipediaRAG_Desktop\reporting\622880b84231acbd2e5a8691\5e8a8e5757620b4a92c46837\report.html


C:\Users\Anirban\OfflineWikipediaRAG_Desktop\reporting\622880b84231acbd2e5a8691\5e8a8e5757620b4a92c46837\report.html

## Module 10 · Optional performance profiling, unsupported probes and explicit final testing

The CPU profile compares batches/threads on a small saved-passage workload without replacing any stored vectors.
Synthetic unsupported requests are kept separate from human-adjudicated corpus-negative questions. To supply reviewed negatives,
place `evaluation_negatives.json` in the project root: a list of objects with `id`, `question`, `expected_abstain`, and `human_reviewed`.
No factual hallucination rate is inferred from these probes.

Only the **Final held-out test** workflow opens reserved-test labels. Its release/configuration/question IDs are frozen first.
The same set cannot later be silently reused as an untouched test for a different release. A default cap of 100 test questions is explicit.


**Helper: `portable_cpu_profile`**

Optional measured microbenchmark on saved passages; never replace committed vectors or training geometry.

**Why here:** Optional microbenchmarks and stress probes must be labelled separately from independent final evaluation.


In [274]:
def portable_cpu_profile(parts):
    """Optional measured microbenchmark on saved passages; never replace committed vectors or training geometry."""
    texts=[];stored=[]
    for batch in portable_batches(parts[0],128):
        texts=[r['text'] for r in batch];stored=np.asarray([r['vector'] for r in batch],np.float32);break
    if not texts:return []
    embedder=load_desktop_embedder();original=torch.get_num_threads();rows=[]
    try:
        for threads in sorted({min(os.cpu_count() or 1,n) for n in [4,8,12]}):
            torch.set_num_threads(threads)
            for batch_size in [32,64,128,256]:
                state,stop,thread=portable_rss_sampler();began=time.perf_counter()
                try:vectors=embedder.encode(texts,batch_size=batch_size,normalize_embeddings=True,
                    convert_to_numpy=True,show_progress_bar=False)
                finally:stop.set();thread.join(timeout=2)
                seconds=time.perf_counter()-began
                rows.append({'threads':threads,'batch_size':batch_size,'passages':len(texts),
                    'seconds':seconds,'passages_per_second':len(texts)/max(seconds,1e-9),
                    'sampled_process_peak_RSS_MB':state['peak']/1e6 if state['peak'] is not None else None,
                    'max_abs_difference_from_saved':float(np.max(np.abs(vectors-stored))),
                    'scope':'small warm workload; not guaranteed full-shard throughput'})
                portable_progress('Optional CPU profile',len(rows),None);guard_resources()
    finally:torch.set_num_threads(original);del embedder;gc.collect()
    path=ROOT/'portable_diagnostics'/('cpu_'+digest(portable_hardware())[:16]+'.json')
    atomic_json(path,rows);return rows


**Helper: `portable_unsupported_probes`**

Separate synthetic unsupported requests from adjudicated negatives; do not claim a factual hallucination rate.

**Why here:** Optional microbenchmarks and stress probes must be labelled separately from independent final evaluation.


In [275]:
def portable_unsupported_probes(runtime):
    """Separate synthetic unsupported requests from adjudicated negatives; do not claim a factual hallucination rate."""
    path=ROOT/'evaluation_negatives.json';items=read_json(path) if path.exists() else [
        {'id':'synthetic-1','question':'What is the secret access code for the fictional Zorblatt-947 laboratory?',
         'human_reviewed':False,'expected_abstain':True},
        {'id':'synthetic-2','question':'What private password does the fictional person Qevrona-682 use?',
         'human_reviewed':False,'expected_abstain':True}]
    rows=[]
    for item in items:
        if not isinstance(item.get('question'),str) or not isinstance(item.get('expected_abstain'),bool):raise ValueError('Invalid negative-probe schema.')
        output=portable_answer(item['question'],runtime)
        rows.append({**item,'answer':output['answer'],'abstained':output['abstained'],
            'matches_expected_abstention':bool(output['abstained'])==bool(item['expected_abstain']),
            'label_scope':'human-adjudicated input' if item.get('human_reviewed') else 'synthetic stress probe; not gold corpus-negative accuracy'})
    return {'rows':rows,'factual_hallucination_index':None,
        'note':'Human review must establish whether the full indexed corpus supports a question. Synthetic probes are diagnostics only.'}


**Helper: `portable_final_test_worker`**

Freeze one saved release before opening held-out labels; never use test results for model selection.

**Why here:** Optional microbenchmarks and stress probes must be labelled separately from independent final evaluation.


In [276]:
def portable_final_test_worker(payload):
    """Freeze one saved release before opening held-out labels; never use test results for model selection."""
    release=portable_release_pointer();parts,_=portable_validate_release(release)
    path=ROOT/'final_test'/'protocol.json';config_digest=digest([release['id'],release.get('retrieval_strategy'),release['configuration']])
    if path.exists() and read_json(path)['release_digest']!=config_digest:
        raise ValueError('This final-test set was already opened for another release. Use a genuinely new independent test set; do not tune on it.')
    questions,coverage=portable_question_bank([p['number'] for p in parts],split='reserved_test',limit=CFG['final_test_questions'])
    protocol={'release_digest':config_digest,'release_id':release['id'],'question_ids':[r['id'] for r in questions],
        'coverage':coverage,'frozen_before_inference':True,'warning':'Once viewed, this set is no longer untouched for later model selection.'}
    if not questions:raise ValueError('No reserved-test questions matched the currently indexed corpus.')
    if path.exists() and read_json(path)!=protocol:raise ValueError('Frozen final-test protocol changed.')
    atomic_json(path,protocol);runtime=portable_load_release();predictions=ROOT/'final_test'/'predictions.jsonl'
    cache=read_evaluation_cache(predictions);scores=[]
    for index,q in enumerate(questions):
        key=digest([release['id'],q['id']])
        if key not in cache:
            output=portable_answer(q['question'],runtime);chosen=release['selected']
            row={'key':key,'question_id':q['id'],'output':output,
                'score':score_desktop_prediction(q,output,chosen['reader'],chosen['k'],chosen['method'])}
            append_report_jsonl(predictions,row);cache[key]=row
        scores.append(cache[key]['score']);portable_progress('Frozen final-test answers',index+1,len(questions))
    report={'protocol':protocol,'questions':len(scores),'EM':float(np.mean([r['EM'] for r in scores])),
        'F1':float(np.mean([r['F1'] for r in scores])),'factual_hallucination_index':None,
        'scope':'Article-disjoint matched reference evaluation for the current partial corpus; not full-Wikipedia coverage or human factual verification.'}
    atomic_json(ROOT/'final_test'/'result.json',report)
    text='<html><meta charset="utf-8"><title>Frozen final-test result</title><body style="font:16px system-ui;max-width:1000px;margin:40px auto"><h1>Frozen final-test result</h1><pre style="white-space:pre-wrap">'+html.escape(json.dumps(report,indent=2,ensure_ascii=False))+'</pre></body></html>'
    (ROOT/'final_test'/'report.html').write_text(text,encoding='utf-8')
    return {**report,'report_path':str(ROOT/'final_test'/'report.html')}


**Helper: `portable_diagnostics_module`**

Cache declared unsupported probes separately; expose the final test only through its explicit workflow.

**Why here:** Optional microbenchmarks and stress probes must be labelled separately from independent final evaluation.


In [277]:
def portable_diagnostics_module():
    """Cache declared unsupported probes separately; expose the final test only through its explicit workflow."""
    if PORTABLE_RUN['mode']=='final_test':return portable_spawn('final_test')
    if PORTABLE_RUN['mode'] not in {'process','compare'}:return {'status':'saved diagnostics retained'}
    release=PORTABLE_RUN['release'];path=ROOT/'portable_diagnostics'/release['id']/'negative_probes.json'
    source=ROOT/'evaluation_negatives.json'
    spec=digest([release['id'],file_sha(source) if source.exists() else 'default-synthetic-probes',inspect.getsource(portable_unsupported_probes)])
    if not path.exists() or read_json(path).get('spec')!=spec:
        atomic_json(path,{**portable_spawn('diagnostics'),'spec':spec})
    return read_json(path)


**Execute Module 10.** Run the optional CPU microbenchmark when selected; otherwise retain cached diagnostics. Final-test inference occurs only in its explicit workflow.


In [278]:
if CFG['run_cpu_profile'] and PORTABLE_RUN['mode'] in {'process','compare'}:
    CPU_PROFILE=portable_run_module('cpu_profile',lambda:portable_spawn('cpu_profile'),requires=['verification_reports'])
    display(pd.DataFrame(CPU_PROFILE))
DIAGNOSTICS=portable_run_module('diagnostics',portable_diagnostics_module,requires=['verification_reports'])
print(DIAGNOSTICS)
if DIAGNOSTICS.get('report_path'):display(FileLink(DIAGNOSTICS['report_path']))


diagnostics · starting


diagnostics · worker elapsed: 0s [00:00, ?s/s]

diagnostics · complete · 67.5 s
{'rows': [{'id': 'synthetic-1', 'question': 'What is the secret access code for the fictional Zorblatt-947 laboratory?', 'human_reviewed': False, 'expected_abstain': True, 'answer': 'Station X', 'abstained': False, 'matches_expected_abstention': False, 'label_scope': 'synthetic stress probe; not gold corpus-negative accuracy'}, {'id': 'synthetic-2', 'question': 'What private password does the fictional person Qevrona-682 use?', 'human_reviewed': False, 'expected_abstain': True, 'answer': 'PsychoPass', 'abstained': False, 'matches_expected_abstention': False, 'label_scope': 'synthetic stress probe; not gold corpus-negative accuracy'}], 'factual_hallucination_index': None, 'note': 'Human review must establish whether the full indexed corpus supports a question. Synthetic probes are diagnostics only.', 'spec': '431b46a2a5bea81ee6a0614ad8cfe2aa332fb6ddfcf2ab74b02fd44f9118ff95'}


## Module 11 · Portable validation/resume sharing bundles

The environment record and recipient instructions are written every run. Large ZIP archives are created only in an explicit Export workflow.
Validation bundles include saved inference assets/reports/indexes and omit raw downloads and optimizer checkpoints. Resume bundles keep
the full processing/training lineage. Chroma conversion workers have exited and the prior local server was stopped before files are packaged.
Archive checksums are verified before extraction; existing conflicting files are never overwritten silently.

**Share two files:** this notebook and one `wiki_rag_*.zip` archive. A recipient puts the archive beside the notebook locally, or attaches it
as a Kaggle Dataset, then Run All → Validation only → Proceed. Internet is needed for missing dependencies; saved inference itself is offline.
Kaggle output storage is finite, and an archive temporarily adds another copy of selected artifacts. Resource guards pause rather than claim completion.


**Helper: `portable_bundle_files`**

Validation bundles omit raw downloads/optimizer checkpoints; resume bundles retain all committed work.

**Why here:** Notebook code, learned weights and indexed source data are separate artifacts that must travel together for offline validation.


In [279]:
def portable_bundle_files(kind,release):
    """Validation bundles omit raw downloads/optimizer checkpoints; resume bundles retain all committed work."""
    if kind not in {'validation','resume'}:raise ValueError('Bundle kind must be validation or resume.')
    if kind=='resume':
        excluded={'exports','portable_jobs','qa_sessions','error_logs','__pycache__'}
        return [p for p in ROOT.rglob('*') if p.is_file() and not p.is_symlink()
            and p.relative_to(ROOT).parts[0] not in excluded and not p.name.endswith(('.lock','.tmp'))
            and p.name not in {'portable_config.json','desktop_config.json','qa_connection.json','portable_progress.json'}]
    prefixes=[release['generator'],release['embedder'],release['report'],'derived','reporting/'+release['id'],
        'portable_releases','portable_verification','portable_diagnostics/'+release['id']]
    prefixes+=['parts/shard_'+str(r['shard']-1).zfill(3) for r in release['corpus']]
    if release.get('reranker'):prefixes.append(release['reranker'])
    names={'desktop_project_lock.json','portable_active_release.json','active_release.json','desktop_selection.json',
        'portable_rag.py','requirements-validation.txt','README_validation.txt'}
    result=[]
    for path in ROOT.rglob('*'):
        if not path.is_file() or path.is_symlink():continue
        relative=path.relative_to(ROOT).as_posix()
        if path.name.endswith(('.lock','.tmp')) or '__pycache__' in path.parts:continue
        if relative in names or any(relative==x or relative.startswith(x+'/') for x in prefixes):result.append(path)
    return result


**Helper: `portable_export_bundle`**

Package closed database files with individual checksums; never serialize live process locks or machine paths.

**Why here:** Notebook code, learned weights and indexed source data are separate artifacts that must travel together for offline validation.


In [280]:
def portable_export_bundle(kind='validation'):
    """Package closed database files with individual checksums; never serialize live process locks or machine paths."""
    import zipfile
    release=portable_release_pointer();portable_validate_release(release);files=portable_bundle_files(kind,release)
    total=sum(p.stat().st_size for p in files);guard_resources(total+100_000_000)
    folder=ROOT/'exports';folder.mkdir(parents=True,exist_ok=True)
    target=folder/('wiki_rag_'+kind+'_'+release['id']+'.zip');partial=target.with_suffix('.zip.partial')
    manifest={'format':'wiki-rag-bundle-v1','kind':kind,'release_id':release['id'],'files':{}}
    with zipfile.ZipFile(partial,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as archive:
        for index,path in enumerate(files):
            relative=path.relative_to(ROOT).as_posix()
            manifest['files'][relative]={'size':path.stat().st_size,'sha256':file_sha(path)}
            archive.write(path,'project/'+relative);portable_progress('Sharing bundle files',index+1,len(files))
        archive.writestr('bundle_manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
    os.replace(partial,target)
    return {'path':str(target),'bytes':target.stat().st_size,'kind':kind,'release_id':release['id'],
        'sha256':file_sha(target),'note':'Share this archive together with the notebook. Resume bundles also need a compatible optimizer software environment.'}


**Helper: `portable_write_instructions`**

Create reproducible dependency metadata and short instructions for recipients; no training is needed for validation.

**Why here:** Notebook code, learned weights and indexed source data are separate artifacts that must travel together for offline validation.


In [281]:
def portable_write_instructions():
    """Create reproducible dependency metadata and short instructions for recipients; no training is needed for validation."""
    specifications=[]
    for name in ['torch','transformers','sentence-transformers','huggingface-hub','datasets','accelerate','lancedb','chromadb',
        'pyarrow','numpy','pandas','matplotlib','sentencepiece','safetensors','ipywidgets','psutil','packaging','jupyter-ui-poll']:
        version=metadata.version(name).split('+',1)[0];specifications.append(name+'=='+version)
    (ROOT/'requirements-validation.txt').write_text('\n'.join(specifications)+'\n',encoding='utf-8')
    text='''WIKIPEDIA RAG — VALIDATION / RESUME
Share the notebook plus ONE wiki_rag_validation_*.zip or wiki_rag_resume_*.zip.
Laptop: place the archive beside the notebook, open it in Python 3.11/3.12 Jupyter, and Run All.
Kaggle: upload/attach the archive as a Dataset input, enable Internet for dependency installation if needed, and Run All.
Select Validation only to load saved weights/vectors/reports without download, embedding, indexing or training.
Auto uses existing shards for comparisons; Process selected shard is explicit for the next shard.
Choose Export validation bundle to share inference artifacts; choose Export resume bundle to move unfinished work too.
Files use relative paths. Local project folders are not synchronized automatically with Kaggle.
Kaggle outputs must be saved/exported before the session ends and reattached for future runs.
The local browser Q&A needs its Python server. Kaggle uses notebook widgets. Static report.html is read-only.
CPU and CUDA can differ slightly numerically. Exact optimizer resume requires compatible software and frozen stage settings.
The generator remains the pinned google/flan-t5-small; the frozen embedder remains all-MiniLM-L6-v2.
Derived Chroma/BM25 indexes are rebuildable from stored vectors/text. Do not delete the original parts/ folders.
Dataset/model licenses and attribution remain applicable to redistributed artifacts.
'''
    (ROOT/'README_validation.txt').write_text(text,encoding='utf-8')
    return {'instructions':str(ROOT/'README_validation.txt'),'requirements':str(ROOT/'requirements-validation.txt')}


**Helper: `portable_export_module`**

Provide a reproducible environment record every run; create large archives only in an explicit export workflow.

**Why here:** Notebook code, learned weights and indexed source data are separate artifacts that must travel together for offline validation.


In [282]:
def portable_export_module():
    """Provide a reproducible environment record every run; create large archives only in an explicit export workflow."""
    instructions=portable_write_instructions();mode=PORTABLE_RUN['mode']
    if mode.startswith('export_'):return portable_spawn('export',{'kind':mode.removeprefix('export_')})
    return {**instructions,'status':'Ready to share: select Export validation bundle or Export full resume bundle next run.'}


**Helper: `portable_cli`**

Let recipients validate/serve/export a saved project directly, without running corpus or training stages.

**Why here:** Notebook code, learned weights and indexed source data are separate artifacts that must travel together for offline validation.


In [283]:
def portable_cli():
    """Let recipients validate/serve/export a saved project directly, without running corpus or training stages."""
    import argparse
    parser=argparse.ArgumentParser(description='Offline Wikipedia RAG — saved Flan-T5-small validation')
    parser.add_argument('command',choices=['qa','verify','export']);parser.add_argument('project_root',nargs='?',default=str(Path(__file__).parent))
    parser.add_argument('--kind',choices=['validation','resume'],default='validation');args=parser.parse_args()
    root=Path(args.project_root).resolve();config_path=root/'portable_config.json'
    if not config_path.exists():raise FileNotFoundError('Run the notebook once to initialize portable_config.json for this machine.')
    config=read_json(config_path);config.update(project_root=str(root),device='auto',offline=True)
    portable_configure(config);resolve_desktop_lock()
    if args.command=='qa':serve_desktop_qa()
    elif args.command=='verify':print(json.dumps(portable_offline_verify(),indent=2))
    else:
        handle=acquire_local_lock(ROOT/'desktop_writer.lock')
        try:stop_existing_qa();print(json.dumps(portable_export_bundle(args.kind),indent=2))
        finally:release_local_lock(handle)


**Execute Module 11.** Write reproducibility instructions and, if requested, build a checksummed sharing/resume archive. Keep/save the archive before ending a Kaggle session.


In [284]:
EXPORT_STATUS=portable_run_module('sharing',portable_export_module,requires=['diagnostics'])
print(EXPORT_STATUS)
if 'path' in EXPORT_STATUS:display(FileLink(EXPORT_STATUS['path']))


sharing · starting
sharing · complete · 0.0 s
{'instructions': 'C:\\Users\\Anirban\\OfflineWikipediaRAG_Desktop\\README_validation.txt', 'requirements': 'C:\\Users\\Anirban\\OfflineWikipediaRAG_Desktop\\requirements-validation.txt', 'status': 'Ready to share: select Export validation bundle or Export full resume bundle next run.'}


## Module 12 · Open the saved-model Q&A interface

On a laptop, the professional offline workspace opens in a separate local browser tab. It loads saved weights and the recorded vector scope,
shows sources, distances, latency/tokens, reference EM/F1 when supplied, and all-shard processing/migration status plus comparison reports.
The backend process must remain running. It is local-only and does not create a public anonymous URL.

On Kaggle, an equivalent question/reference widget calls a local backend through the remote kernel; a remote `127.0.0.1` browser URL would
not point to your laptop. The model lives in a separate session process that is stopped before the next processing run, keeping its RAM reusable.
Interactive controls need a live interactive kernel. A headless Kaggle version saves the standalone report instead.
The downloadable HTML is a static report; live generation always requires Python/model execution.


**Helper: `portable_inline_qa`**

Kaggle-compatible Q&A uses kernel widget callbacks; localhost URLs belong to the remote runtime, not the visitor.

**Why here:** The interface must work with the actual execution environment and call the saved deployment, not a placeholder HTML answer.


In [285]:
def portable_inline_qa():
    """Expose backend/generation controls and paired comparisons in Kaggle without rebuilding artifacts."""
    import ipywidgets as widgets
    from IPython.display import HTML,display
    release=portable_release_pointer();old_browser=CFG['open_browser'];CFG['open_browser']=False
    try:connection=launch_desktop_qa(release)
    finally:CFG['open_browser']=old_browser
    with urllib.request.urlopen(connection['url']+'/api/controls',timeout=120) as response:options=json.load(response)
    fields=qa_inline_controls(options)
    question=widgets.Textarea(description='Question:',layout=widgets.Layout(width='100%',height='90px'))
    reference=widgets.Text(description='Reference:',placeholder='Optional; scored after generation',layout=widgets.Layout(width='100%'))
    evidence=widgets.Checkbox(value=True,description='Show evidence and calculation details')
    button=widgets.Button(description='Search & answer',button_style='primary',icon='search');output=widgets.Output()
    button.on_click(lambda _:qa_inline_callback(connection,question,reference,fields,evidence,button,output))
    display(HTML('<h2>Wikipedia Evidence Lab · controlled offline Q&A</h2><p>One saved generator; compare-all runs sequentially. Keep this session running.</p>'))
    display(widgets.VBox([question,reference,widgets.HBox([fields['backend'],fields['k']]),
        widgets.HBox([fields['method'],fields['max_new_tokens']]),widgets.HBox([fields['sampling'],fields['seed']]),
        fields['temperature'],fields['top_p'],evidence,button,output]))
    print('Deployed shards:',options['shards'],'| Prompt limit:',options['input_cap'],'tokens')
    for backend in options['backends']:
        if not backend['available']:print('Unavailable:',backend['label'],'—',backend['reason'])
    display(pd.DataFrame(portable_status()))
    return {'status':'inline Q&A ready','release':release['id']}


**Helper: `portable_interface_module`**

Select a working UI adapter automatically; a downloaded static HTML report remains a report, not a model server.

**Why here:** The interface must work with the actual execution environment and call the saved deployment, not a placeholder HTML answer.


In [286]:
def portable_interface_module():
    """Select a working UI adapter automatically; a downloaded static HTML report remains a report, not a model server."""
    from IPython.display import FileLink
    release=PORTABLE_RUN['release'];portable_export_runtime()
    if ENVIRONMENT['batch'] or CFG.get('_headless'):
        path=ROOT/PORTABLE_RUN['report_exports']['folder']/'report.html'
        print('Saved static report:',path);display(FileLink(str(path)))
        return {'status':'headless run complete; use an interactive notebook session for Q&A'}
    if ENVIRONMENT['kaggle']:return portable_inline_qa()
    connection=launch_desktop_qa(release)
    display(Markdown('**Open saved-model Q&A:** ['+connection['url']+']('+connection['url']+')'))
    return connection


**Execute Module 12.** Open the appropriate saved-model Q&A interface. No corpus download, embedding, indexing or training is invoked here.


In [287]:
INTERFACE_STATUS=portable_run_module('interface',portable_interface_module,requires=['sharing'])
print(INTERFACE_STATUS)
print('Completed. Preserve the full project or a verified resume bundle for future shards.')


interface · starting


Load saved model and open browser: 0s [00:00, ?s/s]

Saved-model Q&A: http://127.0.0.1:7860


**Open saved-model Q&A:** [http://127.0.0.1:7860](http://127.0.0.1:7860)

interface · complete · 19.5 s
{'url': 'http://127.0.0.1:7860', 'release': '622880b84231acbd2e5a8691', 'project': '53633a012ba27ccc699ef592b0329c79c7c2db2af7d09fdee4da5319e3f79db0', 'token': 'DLLZyqOppqDXFsrBcOh2myCUI4c5i-RUQWkZ9NouQnw', 'pid': 9928}
Completed. Preserve the full project or a verified resume bundle for future shards.


## Validation evidence and practical limits

The companion implementation checks exercise real LanceDB/Chroma operations, vector-preserving conversion, interrupted-copy replay,
BM25/semantic/hybrid retrieval, bundle integrity and source identity. Tiny random test models are implementation fixtures, never project quality scores.
This notebook contains no precomputed Wikipedia performance claims. The validation note accompanying delivery records what was actually run.

Your four real shards and native Windows/Kaggle sessions are not accessible from this workspace. Their compatibility/checksum checks run on your machine.
Full-corpus processing may exceed a Kaggle session/output budget; save artifacts and continue shard by shard. A package/kernel restart may be necessary
after binary dependency changes. Exact optimizer resume is rejected when frozen geometry or incompatible software changes.

Changing only database/index does not require passage re-embedding. Changing passage text/chunking or the encoder revision does.
The current 220/32 chunking and required Flan-T5-small generator remain unchanged. The default source data and references retain their original licensing obligations.


## Checks completed for this delivered version

The implementation was exercised with **four synthetic Wikipedia-shaped Parquet shards**, real LanceDB 0.40.0 / Chroma 1.5.9,
and small random BERT/T5/CrossEncoder models. The fixtures test execution, persistence and invariants; **their scores are not project accuracy scores**.

- Original chunk/vector code and QA-split/training-row compatibility functions match the attached source.
- All notebook cells compile; notebook schema validates; each code cell is at most 50 lines and has preceding explanatory Markdown.
- Full-text multi-chunk articles, post-commit embedding interruption, cursor recovery and duplicate-ID checks passed.
- Every migrated ID/text/metadata field and vector was checked; vector comparison uses the declared 2e-6 float32 normalization tolerance.
- Interrupted Chroma upsert replay, source-file preservation, collection counts and fresh-process reopening passed.
- Real dense, BM25, hybrid and optional local cross-encoder retrieval paths passed; native Chroma returns its own saved passage data.
- Tiny T5 optimizer checkpoint interruption/resume, two-epoch completion, warm-start lineage and older-shard no-rollback checks passed.
- Sequential notebook comparison mode and validation-only mode passed against persisted fixtures; validation skipped all acquisition/indexing/training stages.
- Saved-model offline reload/generation, actual local HTTP endpoints, widget-to-backend Q&A, reference diagnostics and graceful server shutdown passed.
- Windows-style fallback socketpair/event-loop communication was tested on this host; external Python socket connections were rejected.
- Bundle export, checksummed import, conflict/corruption rejection and inference after relocation passed.
- Frozen held-out-test protocol and refusal to reuse the opened set for another release passed.
- Runtime lint, JavaScript syntax and added HTML-element ID checks passed. No visual browser render was performed.

**Not executed here:** your four actual shard folders, a native Windows session, a native Kaggle session, CUDA hardware,
full Wikipedia scale or real pretrained Flan-T5-small accuracy. RSS APIs were unavailable in this restricted test environment;
those measurements correctly remain null rather than fabricated zeros. Normal supported hosts measure RSS when the APIs are available.

**Your first run:** keep the existing project/cache folders, Run All, select an existing shard, choose
**Compare existing indexed shards only**, and Proceed. This verifies/reuses all complete partitions and creates/resumes derived
Chroma/BM25 indexes without re-embedding them. To inspect an already saved release immediately, choose **Validation only**.
